<a href="https://colab.research.google.com/github/Geauga/Proteina-Complexa-coLab/blob/dev/coLab/Proteina_Complexa_coLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section0

<details>
<summary><font size="5"><b>⚙️ Expand It: User Quick Start README & Global Configuration</b></font></summary>

<br>

### 💡 ARCHITECTURE OVERVIEW
This notebook employs a decoupled "Front-end UI / Back-end Engine" architecture. Please read this quick start guide and configure all parameters exclusively within the **Section 0** dashboard. This centralized panel persists your configuration data to Google Drive, ensuring seamless execution even across kernel restarts.

### 🛑 CRITICAL NOTICE: MANDATORY KERNEL RESTART
After completing **Section 1** (Environment Configuration), a Colab runtime restart is **strictly required** to apply core library upgrades (JAX/Flax/CUDA).
**ACTION:** Navigate to `Runtime -> Restart session` from the menu, and then **run Section 1 again**. Failure to do so will cause the pipeline to terminate with an error.

### 📦 ZERO-CONFIGURATION ASSETS
Demonstration protein structures (`GFP_1_10.pdb` and `GFP_11.pdb`) are pre-integrated.
* **Custom Targets:** Upload your PDB files to the `Proteina-Complexa/targets/` directory on Google Drive, or use the integrated upload toggle in Section 0.
* **Configuration:** After uploading, ensure you update the `task_name` and `pdb_file_name` in Section 0 accordingly.

---

### 🎛️ PARAMETER GLOSSARY (Section 0 Dashboard)
Before running the pipeline, configure the following variables in the UI panel:

**Global & Session Management**
* `task_name`: A unique identifier prefix for your current project (e.g., `POR_Attention`). All output folders will be grouped under this name.
* `target_chains`: The specific chain ID(s) (e.g., `A`) on the target protein you want the AI to analyze.
* `RESUME_LAST_SESSION`: Toggle **ON** to inherit the previous `run_id`. This prevents overwriting and safely appends new generated structures to existing checkpoint directories if your session was disconnected.

**Auto-Pilot Generation Config (Section 4)**
* `hotspots_input`: A comma-separated list of critical residue indices on the target protein. The AI will strictly force the generated binder to interact with these specific positions.
* `binder_length_min` / `max`: The acceptable length range (number of amino acids) for the generated binder peptide.
* `target_threshold`: The minimum binding score (`max_ipSAE`) required to classify a generated structure as a "success".
* `target_success_count`: The Auto-Pilot loop will automatically halt once it accumulates this number of successful designs.
* `max_iterations`: The absolute maximum number of generation cycles the Auto-Pilot will execute, acting as a failsafe to prevent infinite loops.

**Geometric Surface Scanning Config (GeoScan - Section 6)**
* `anchor_residues`: A set of starting residue indices used as the geometric center to map the target's surface.
* `global_search_boundary` (Å): The **Macro Search Zone**. Defines the maximum outward expansion limit from the anchors. A larger value forces the algorithm to explore a wider, more distant surface area for candidate patches.
* `local_interaction_radius` (Å): The **Micro Design Canvas**. Determines the physical size of the binding interface fed to the AI for a single generation task. A larger radius provides the neural network with a broader structural context, resulting in bulkier binders.
* `designs_per_patch`: The number of stochastic designs the AI will generate for each isolated local patch coordinate.

---
### 🔄 EXECUTION WORKFLOWS

**Phase I: Universal Initialization**
1.  **Hardware Requirements & Pre-check:**
    * **Storage:** At least 20GB of free space on Google Drive is recommended; 30GB+ is required if localizing program initialization data to accelerate startup.
    * **GPU:** A minimum of L4 GPU runtime is required. For longer protein chains with high VRAM usage, A100 runtime is needed. *Note: The GFP tutorial can be completed using L4 runtime with default settings.*
2.  **Read & Configure:** Review these guidelines and adjust all parameters (Auto-Pilot, Standard, Geometric Scanning) in the **Section 0** UI dashboard. Execute the cell to register the variables persistently.
3.  **Environment Initialization:** Execute **Section 1**.
    * *Note:* Initial compilation and asset synchronization take approximately 10–20 minutes. Enabling Google Drive persistent backup (~10GB) is recommended, allowing subsequent runs to skip this stage even if the session is reset.
4.  **Restart Session & Verification:** Perform `Runtime -> Restart session`. **You must re-run Section 1 after the restart** (re-running Section 0 is optional).
5.  **Target Pre-processing:** Expand and execute **Section 2** to parse and prepare the target structure.

**Phase II: Divergent Execution Pathways**
* **Section 3: GFP Tutorial.** The system automatically scans the surface of Split-GFP subunit 1 to identify the subunit 2 binding groove. It then calls the `Proteina_Complexa` evaluation module to generate the complex, providing baseline data for the natural peptide and 3D structural models of the complex.
* **Section 4: High-Throughput Auto-Pilot.** Continuously generates, evaluates, and logs candidates until a predefined **success threshold** is met. Default is set to screen for subunit 1 binding peptides; users can upload or specify their target protein in the Section 0 panel.
* **Section 5: Targeted Surface Screening.** Screens for the hotspots with the strongest binding affinity on the target protein's surface and their corresponding peptides. Default is set to screen for subunit 1 binding peptides; users can upload or specify their target protein in the Section 0 panel.

</details>

In [ ]:
# Section_0_Global_Configuration_Dashboard.py
# Requirement: Save global parameters to the configuration file immediately before invoking the interactive file upload widget, preventing the parameter saving process from being blocked or bypassed if no file is uploaded.
# UPDATE: Removed global force_restart. Added section-specific resume/restart toggles (Section 5, 6, 7) directly within their respective configuration blocks for granular session management.

# @title 🎯 Section_0_Global_Configuration_Dashboard { display-mode: "form" }
# @markdown Complete this dashboard **ONCE**. All settings are persistently saved to your Google Drive.

import os, json, shutil, sys
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
from pathlib import Path
from IPython.display import display, HTML

# ==========================================
# 0. Infrastructure & Google Drive Mount
# ==========================================
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    print("🔄 Mounting Google Drive for persistent storage...")
    drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")

# --- Source Repository Detection and Cloning ---
if not os.path.exists('/content/drive/MyDrive/Proteina-Complexa/pyproject.toml'):
    print(f">>> Source repository not detected. Executing initial clone to {ROOT_DIR}...")
    !git clone https://github.com/Geauga/Proteina-Complexa-coLab {ROOT_DIR}
else:
    print(f">>> Source repository already exists at {ROOT_DIR}. Skipping clone step.")

TARGETS_DIR = BASE_DIR / 'targets'
TARGETS_DIR.mkdir(parents=True, exist_ok=True)
SETTING_FILE = BASE_DIR / 'internal_settings.json'
# @markdown ---
# @markdown ---
# ==========================================
# --- Configuration for Environment Setup & Cache (Cell 3) ---
# ==========================================
# @markdown ### 💾 Environment & Cache Settings
# @markdown **[Accelerate Startup]** Enable Google Drive backup for the build cache?
# @markdown > *Note: The very first run will be slightly slower (1-2 mins) to package the backup; subsequent startups will be lightning fast.*
enable_cache_backup = True # @param {type:"boolean"}

# ==========================================
# --- Hardware Memory VRAM Limits ---
# ==========================================
# @markdown ### 💻 Hardware Memory Limits (VRAM Limit)
# @markdown Set this lower if you encounter OOM (Out Of Memory). This is shared across evaluating and generating sections.
generation_batch_size = 36 #@param {type:"integer"}
# @markdown **[AF2 Evaluation Batch]** Safe limit: 1. Evaluator uses massive memory. Keep at 1 for 16GB GPUs, or max 2-3 for A100. Over 3 will OOM.
evaluation_batch_size = 1 #@param {type:"integer"}
gpu_batch_size = 1 #@param {type:"integer"}

# ==========================================
# --- Configuration for Target Setup & Processing (Cell 4) ---
# ==========================================
# @markdown ---
# @markdown ---
# ==========================================
# --- Time Machine / Global Resume ---
# ==========================================
# @markdown ---
# @markdown ---
# @markdown ### 🔮 Time Machine / Global Resume
# @markdown Fill in a Specific Run ID (e.g. `AR_LBD_20260418_043148`) to travel back and resume from it. Leave blank for normal operation.
specific_resume_run_id = "" #@param {type:"string"}

# @markdown ### 📁 Target File Selection
UPLOAD_NEW_PDB = False #@param {type:"boolean"}
pdb_file_name = "AR_LBD_ChainB_5JJM.pdb" #@param {type:"string"}

# @markdown ### 🧬 Target Definition
task_name = "AR_LBD_wolf_evo" #@param {type:"string"}
target_chains = "B" #@param {type:"string"}

# @markdown ---
# @markdown ---
run_filter = True #@param {type:"boolean"}
run_evaluate = True #@param {type:"boolean"}
run_analyze = True #@param {type:"boolean"}

# @markdown ---
# @markdown ---
# @markdown ### 🎯 Interaction Hotspots (Comma separated)
hotspots_input = "681,682,752,754, 755, 756, 757, 758, 759, 760,763,764,766, 767" #@param {type:"string"}

# @markdown ### 📏 Binder Peptide Length
binder_length_min = 20 #@param {type:"integer"}
binder_length_max = 30 #@param {type:"integer"}

# ==========================================
# --- Auto-Pilot Screening Configuration (Section 4) ---
# ==========================================
# @markdown ---
# @markdown ---
# @markdown ### 🎯 Auto-Pilot Screening Configuration (Section 4)
target_threshold = 0.85 #@param {type:"number"}
target_success_count = 10 #@param {type:"integer"}
max_iterations = 30 #@param {type:"integer"}
designs_per_loop = 36 #@param {type:"integer"}

# ==========================================
# --- Auto-Evolution Target Screening Setting (Section 7) ---
# ==========================================
# @markdown ---
# @markdown ---
# @markdown ### 🚀 Auto-Evolution Target Screening Setting (Section 7)
# @markdown Force restart Section 7 completely? (Ignore existing history)
force_restart_section_7 = False #@param {type:"boolean"}
# 👉 [新增] 仅控制 Cell 22 演化逻辑的重置 (保留 Cell 19/20 的 Patch 扫描结果)
force_restart_evo = False # @param {type:"boolean"}
evo_strict_rmsd_filter = False #@param {type:"boolean"}
evo_designs_per_loop = 8 #@param {type:"integer"}
evo_max_hotspots = 8 #@param {type:"integer"}
evo_update_count = 2 #@param {type:"integer"}
total_designs_per_branch = 8  #@param {type:"integer"}
generation_batches_per_core_hot_spot = 4 #@param {type:"integer"}
generation_batches_per_island = 6 #@param {type:"integer"}
evo_target_threshold = 0.85 #@param {type:"number"}
generation_batches_per_iteration = 3 #@param {type:"integer"}

# ==========================================
# Execution Logic: Validation & Persistence
# ==========================================
import ipywidgets as widgets
from IPython.display import clear_output

final_pdb_name = pdb_file_name

def write_settings_to_disk(pdb_name):
    settings = {
        "specific_resume_run_id": specific_resume_run_id,
        "enable_cache_backup": enable_cache_backup,
        "pdb_file": pdb_name,
        "task_name": task_name,
        "target_chains": target_chains,


        # Section-specific restart toggles

        "force_restart_section_7": force_restart_section_7,
        "force_restart_evo": force_restart_evo, # 👉 确保保存到 JSON
        "hotspots_input": hotspots_input,
        "binder_length_min": binder_length_min,
        "binder_length_max": binder_length_max,
        "run_filter": run_filter,
        "run_evaluate": run_evaluate,
        "run_analyze": run_analyze,
        "target_threshold": target_threshold,
        "target_success_count": target_success_count,
        "designs_per_loop": designs_per_loop,
        "max_iterations": max_iterations,
        "generation_batch_size": generation_batch_size,
        "evaluation_batch_size": evaluation_batch_size,
        "gpu_batch_size": gpu_batch_size,
        "total_designs_per_branch": total_designs_per_branch,
        # MAPPED UI VARIABLES TO BACKWARD COMPATIBLE JSON KEYS

        "evo_designs_per_loop": evo_designs_per_loop,
        "evo_max_hotspots": evo_max_hotspots,
        "evo_update_count": evo_update_count,
        "evo_target_threshold": evo_target_threshold,
        "generation_batches_per_core_hot_spot" :generation_batches_per_core_hot_spot,
        "generation_batches_per_island" : generation_batches_per_island,
        "evo_strict_rmsd_filter": evo_strict_rmsd_filter,
        "generation_batches_per_ev": generation_batches_per_iteration,
    }

    # Preserve existing run_id if it exists to allow proper resumption
    if SETTING_FILE.exists():
        try:
            with open(SETTING_FILE, 'r') as f:
                old_cfg = json.load(f)
                if 'run_id' in old_cfg:
                    settings['run_id'] = old_cfg['run_id']
        except Exception:
            pass

    with open(SETTING_FILE, 'w') as f:
        json.dump(settings, f, indent=4)
    print(f"\n✅ SETTINGS REGISTERED")
    print(f"Task Name  : {task_name}")
    print(f"Target PDB : {pdb_name} (Chain: {target_chains})")
    print(f"Restart Sec 7 (Evolution): {'YES' if force_restart_section_7 else 'NO (Resume)'}")
    print("💾 Data saved. The backend engines will seamlessly inherit these unified configurations.")

# Immediately save parameters regardless of upload status to prevent blocking.
write_settings_to_disk(final_pdb_name)

if UPLOAD_NEW_PDB:
    display(HTML('<p style="font-size:16px; font-weight:bold; color:#E32636;">👉 PLEASE UPLOAD YOUR PDB FILE (Settings have been saved):</p>'))

    upload_widget = widgets.FileUpload(accept='.pdb', multiple=False, description='Upload PDB')
    cancel_button = widgets.Button(description='Cancel & Clear', button_style='danger', icon='trash')
    out_log = widgets.Output()

    upload_state = {"current_file": None}

    def process_upload(change):
        with out_log:
            if upload_widget.value:
                # Handle data structures for both ipywidgets v7 and v8
                val = upload_widget.value
                if isinstance(val, dict):
                    filename = list(val.keys())[0]
                    content = val[filename]['content']
                else:
                    filename = val[0]['name']
                    content = val[0]['content']

                file_path = TARGETS_DIR / filename
                with open(file_path, "wb") as f:
                    f.write(content)

                upload_state["current_file"] = file_path
                print(f"File '{filename}' uploaded successfully to target directory.")

                # Re-save settings with the newly uploaded filename
                write_settings_to_disk(filename)

    def clear_upload(b):
        with out_log:
            clear_output()
            if upload_state["current_file"] and os.path.exists(upload_state["current_file"]):
                os.remove(upload_state["current_file"])
                print(f"Deleted physical file: {upload_state['current_file'].name}")
                upload_state["current_file"] = None

            # Reset widget value safely based on version type
            upload_widget.value = {} if isinstance(upload_widget.value, dict) else ()
            if hasattr(upload_widget, '_counter'):
                upload_widget._counter = 0

            print("Upload canceled and cleared. Ready for new file.")

    upload_widget.observe(process_upload, names='value')
    cancel_button.on_click(clear_upload)

    display(widgets.VBox([widgets.HBox([upload_widget, cancel_button]), out_log]))

🔄 Mounting Google Drive for persistent storage...
Mounted at /content/drive
>>> Source repository already exists at /content/drive/MyDrive/Proteina-Complexa. Skipping clone step.

✅ SETTINGS REGISTERED
Task Name  : AR_LBD_wolf_evo
Target PDB : AR_LBD_ChainB_5JJM.pdb (Chain: B)
Restart Sec 7 (Evolution): NO (Resume)
💾 Data saved. The backend engines will seamlessly inherit these unified configurations.


In [ ]:

# Cell_1_Environment_Initialization.py
# Requirement: Upgrade essential logging and serialization dependencies (wandb, protobuf) for the pipeline, and automatically patch the colabdesign source code to fix JAX compatibility.

# ==========================================
# --- Environment Base Dependencies ---
# ==========================================
!pip install --upgrade wandb protobuf

# ==========================================
# --- Automated Source-Code Patching ---
# ==========================================
import os

print("Scanning and patching colabdesign JAX compatibility...")
loss_file_path = '/content/drive/MyDrive/Proteina-Complexa/community_models/colabdesign/af/loss.py'

if os.path.exists(loss_file_path):
    with open(loss_file_path, 'r') as f:
        content = f.read()

    if 'a_min=' in content or 'a_max=' in content:
        modified_content = content.replace('a_min=', 'min=').replace('a_max=', 'max=')
        with open(loss_file_path, 'w') as f:
            f.write(modified_content)
        print(f"Successfully patched JAX 'clip' argument names in {loss_file_path}")
    else:
        print("Patch already applied. System is ready.")
else:
    print(f"Warning: Target file not found: {loss_file_path}")

# Purpose: Ensure the base environment has the latest versions of wandb and protobuf. Automatically scan and patch the third-party 'colabdesign' source code to prevent JAX TypeError crashes during downstream evaluation.
# Upstream Code: Fresh Google Colab instance.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-17 18:05 EDT.
# Changed Lines:
# 2-3: Updated requirement description.
# 10-29: Added Python block to physically patch the loss.py file if the outdated 'a_min' syntax is detected, ensuring idempotency across environment restarts.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 25.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.34.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.34.1 which is incompatible.
grpcio-status 1.71.2 requires proto

Scanning and patching colabdesign JAX compatibility...
Patch already applied. System is ready.


# Section 1: Environment Configuration & Dependency Management, Run All 3 Cells, Restart Session and Run Next Cell.

In [ ]:
#@title Cell_2_Mount_Google_Drive.py
# Requirement: Mount Google Drive to the Colab instance to enable access to persistent storage for the Proteina-Complexa project files. Added force remount to handle potential non-empty mountpoint errors.

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Purpose: Connect the Google Colab virtual machine to the user's Google Drive, allowing the pipeline to read configurations, load models, and save generated protein structures persistently.
# Upstream Code: The original Google Drive mounting cell (formerly labeled as Cell 1) from the uploaded notebook.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-11 09:17 EDT.
# Changed Lines:
# * Line 5: Added `force_remount=True` parameter to the `drive.mount` function to ensure stable mounting.

Mounted at /content/drive


In [ ]:
#@title Cell_2b_environment_setup_engine.py
import os,json
from pathlib import Path
from IPython.display import display, HTML

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')

SETTING_FILE = BASE_DIR / 'internal_settings.json'
# --- Load Persistent Variables ---
enable_cache_backup = True
if os.path.exists(SETTING_FILE):
    with open(SETTING_FILE, 'r') as f:
        settings = json.load(f)
        enable_cache_backup = settings.get("enable_cache_backup", True)
else:
    print(f">>> Warning: {SETTING_FILE} not found. Defaulting enable_cache_backup to True.")

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")

%cd {ROOT_DIR}

def setup_environment():
    print(">>> Initializing installation and environment configuration...")

    # 1. Base Installation
    !pip install uv
    !uv pip install --system --index-strategy unsafe-best-match -e .

    # 2. Consolidated Dependency Installation
    !uv pip install --system git+https://github.com/RosettaCommons/atomworks.git
    !uv pip install --system torch-scatter --no-build-isolation
    !uv pip install --system dm-haiku openbabel-wheel graphein e3nn

    # 3. Deploy Foldseek Binary
    if not os.path.exists(".venv/bin/foldseek"):
        !mkdir -p .venv/bin
        !wget -q -nc https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
        !tar xzf foldseek-linux-avx2.tar.gz
        !mv foldseek/bin/foldseek .venv/bin/ && chmod +x .venv/bin/foldseek
        !rm -rf foldseek-linux-avx2.tar.gz foldseek/

    # 4. Create sc Script Mock
    !mkdir -p env/docker/internal/
    !echo -e '#!/bin/bash\necho "0.000"' > env/docker/internal/sc && chmod +x env/docker/internal/sc

    # 5. Initialization and Model Download
    !complexa init uv
    !source env.sh && complexa download --complexa-all
    !source env.sh && complexa download --all

    # 6. Path Correction (.env)
    code_path = ROOT_DIR + "/"
    data_path = os.path.join(ROOT_DIR, "assets/")
    if os.path.exists(ENV_FILE):
        with open(ENV_FILE, "r") as f: lines = f.readlines()
        with open(ENV_FILE, "w") as f:
            for line in lines:
                if line.startswith("LOCAL_CODE_PATH="): f.write(f"LOCAL_CODE_PATH={code_path}\n")
                elif line.startswith("LOCAL_DATA_PATH="): f.write(f"LOCAL_DATA_PATH={data_path}\n")
                else: f.write(line)

    # 7. Task Completion: Conditional Backup (Reads enable_cache_backup from Cell 2a)
    if enable_cache_backup:
        if not os.path.exists(UV_CACHE_TAR):
            print(">>> User opted in for cache backup. Extracting and backing up underlying C++ build cache to Drive...")
            !mkdir -p {BACKUP_DIR}
            !mkdir -p ~/.cache/uv
            !cd ~ && tar -cf {UV_CACHE_TAR} .cache/uv
            print(">>> Cache backup complete. Future reboots will achieve rapid recovery.")
        else:
            print(">>> Existing cache backup detected in Drive. Skipping redundant packaging.")
    else:
        print(">>> Cache backup bypassed per user configuration.")

# ====================================================================
# --- Main Logic: Inject cache first, then execute configuration ---
# ====================================================================
if os.path.exists(UV_CACHE_TAR):
    print(">>> 🛡️ Cloud uv build cache detected. Performing extraction and overwrite...")
    !mkdir -p ~/.cache
    !tar -xf {UV_CACHE_TAR} -C ~
    print(">>> Cache injection complete. Preparing environment reconstruction...")
    setup_environment()
else:
    print(">>> No cache detected. Commencing initial heavy compilation (please wait)...")
    setup_environment()

print(">>> Environment ready. Initiating final validation...")
!source env.sh && complexa validate design configs/search_binder_local_pipeline.yaml

/content/drive/MyDrive/Proteina-Complexa
>>> 🛡️ Cloud uv build cache detected. Performing extraction and overwrite...
>>> Cache injection complete. Preparing environment reconstruction...
>>> Initializing installation and environment configuration...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.9 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 138 packages in 6.13s
Prepared 4 packages in 30.77s
Uninstalled 13 packages in 549ms
Installed 41 packages in 107ms
 + biopandas==0.5.1
 + biopython==1.87
 + biotite==0.41.2
 + cftime==1.6.5
 + contextlib2==21.6.0
 + cpdb-protein==0.2.0
 + deepdiff==9.0.0
 - dm-tree==0.1.10
 + dm-tree==0.1.8
 - einops==0.8.2
 + einops==0.6.0
 + hydra-core==1.3.1
 + jaxtyping==0.3.9
 - joblib==1.5.3
 + joblib==1.4.2
 + lightning==2.5.6
 + lightning-utilities==0.15.3
 + loguru==0.7.2
 + looseversion==1.1.2
 + loralib==0.1.2
 + mdtraj==1.10.2
 + ml-collections==0.1.1
 + mmtf-python==1.1.3
 + modin==0.37.1
 + netcdf4==1.7.4
 - numpy

In [ ]:
#@title Cell_3_Repository_Clone_and_JAX_Patch.py
# Requirement: Detect and clone the Proteina-Complexa repository if absent. Upgrade core machine learning libraries (JAX, Flax, Haiku), and dynamically patch deprecated syntax in JAX and ColabDesign to ensure compatibility with Python 3.12+ and modern JAX versions.

# ==========================================
# --- Repository Initialization & Environment Patching ---
# ==========================================
import os
import sys
import subprocess
from pathlib import Path

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")


# Align Paths
BASE_DIR = Path(ROOT_DIR)
COLABDESIGN_DIR = BASE_DIR / 'community_models/colabdesign'

def run_command(command_list):
    print(f"🚀 Executing: {' '.join(command_list)[:80]}...")
    process = subprocess.Popen(command_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout: print(line, end='', flush=True)
    process.wait()

# 1. Upgrade core dependencies (tailored for GPU environment)
print("\n[Stage 1/3] Upgrading JAX and core ecosystem libraries...")
run_command([sys.executable, "-m", "pip", "install", "--upgrade", "jax[cuda12]", "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"])
run_command([sys.executable, "-m", "pip", "install", "--upgrade", "flax", "dm-haiku", "chex", "optax"])

# 2. Patch JAX underlying NumPy compatibility (for Python 3.12+)
print("\n[Stage 2/3] Patching JAX and NumPy conflicts...")
literals_path = Path('/usr/local/lib/python3.12/dist-packages/jax/_src/literals.py')
if literals_path.exists():
    code = literals_path.read_text()
    if ", copy=copy" in code:
        literals_path.write_text(code.replace(", copy=copy", ""))
        print("✅ JAX copy=copy patch applied successfully!")

# 3. Deep patching for ColabDesign source code
print("\n[Stage 3/3] Patching ColabDesign source code compatibility...")
if COLABDESIGN_DIR.exists():
    for filepath in COLABDESIGN_DIR.rglob('*.py'):
        code = filepath.read_text()
        new_code = code.replace('jax.lib.xla_bridge.get_backend()', 'jax.extend.backend.get_backend()') \
                       .replace('.live_buffers()', '.live_arrays()') \
                       .replace('jax.tree_map', 'jax.tree_util.tree_map') \
                       .replace('jax.tree_flatten', 'jax.tree_util.tree_flatten') \
                       .replace('jax.tree_unflatten', 'jax.tree_util.tree_unflatten') \
                       .replace('jax.tree_leaves', 'jax.tree_util.tree_leaves')
        if '@jax.util.wraps' in new_code:
            new_code = new_code.replace('@jax.util.wraps(fun, docstr=docstr)', '@functools.wraps(fun)') \
                               .replace('@jax.util.wraps(fun)', '@functools.wraps(fun)')
            if 'import functools' not in new_code: new_code = 'import functools\n' + new_code
        if new_code != code: filepath.write_text(new_code)
    print("✅ ColabDesign patching completed.")
else:
    print("⚠️ ColabDesign directory not found. Skipping Stage 3 patching.")

print("\n✨ Environment patching is complete. You may now run the evaluation cells below.")

# Purpose: Integrate repository cloning with machine learning dependency upgrades. Patches deprecated syntax in JAX and ColabDesign to ensure compatibility with Python 3.12+ and modern JAX versions.
# Upstream Code: User provided combined logic for Git cloning and JAX/ColabDesign patching.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 09:42 EDT.
# Changed Lines:
# * Consolidated duplicate imports and harmonized base directory variables natively.
# * Added an `else` safeguard block for Stage 3 in case the ColabDesign directory is absent at runtime.


[Stage 1/3] Upgrading JAX and core ecosystem libraries...
🚀 Executing: /usr/bin/python3 -m pip install --upgrade jax[cuda12] -f https://storage.googlea...
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 132.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 120.4 MB/s eta 0:00:00
  Attempting uninstall: jax-cuda12-pjrt
    Found existing installation: jax-cuda12-pjrt 0.7.2
    Uninstalling jax-cuda12-pjrt-0.7.2:
      Successfully uninstalled jax-cuda12-pjrt-0.7.2
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: 

In [ ]:
# # @title 👁️‍🗨️ Web Monitor for VS Code Process
# # Cell_24_Live_Monitor.py
# # ==============================================================================
# # 📡 DUAL-CHANNEL LOGGER MONITOR (Web Keep-Alive)
# # ==============================================================================
# # Description:
# # 终极自适应探针。自动轮询最新的 run_id，无缝切换日志，彻底规避 FileNotFoundError。
# # ==============================================================================

# import time, os, json, re
# from pathlib import Path

# print("📡 正在启动自适应 Live Monitor 探针 (防断连模式)...")

# BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
# ROOT_SETTING = BASE_DIR / 'internal_settings.json'
# last_run_file = BASE_DIR / 'screening_results' / 'last_runs.txt'
# inference_dir = BASE_DIR / 'inference'

# def get_latest_run_id():
#     """动态解析最新的 Run ID"""
#     if not ROOT_SETTING.exists(): return None
#     try:
#         with open(ROOT_SETTING, 'r') as f:
#             task_name = json.load(f).get('task_name', '')
#         if not last_run_file.exists(): return None

#         with open(last_run_file, 'r', encoding='utf-8') as f:
#             lines = f.readlines()
#             pattern = rf"Section2_TargetPreprocess(?:_TimeMachine)? \| Run_ID: ({re.escape(task_name)}_\d+_\d+)"
#             for line in reversed(lines):
#                 match = re.search(pattern, line)
#                 if match:
#                     return match.group(1)
#     except Exception as e:
#         return None
#     return None

# current_run_id = None
# current_log_path = None
# f_handle = None

# try:
#     while True:
#         # 1. 动态获取当前最新的 run_id
#         latest_run_id = get_latest_run_id()

#         # 2. 如果发生了 Run_ID 的切换（例如开启了新的一轮任务）
#         if latest_run_id and latest_run_id != current_run_id:
#             if f_handle:
#                 f_handle.close()
#                 f_handle = None
#             current_run_id = latest_run_id
#             current_log_path = inference_dir / current_run_id / f"Live_Sync_{current_run_id}.log"
#             print(f"\\n🔄 [探针检测到环境更新] 发现最新的执行任务: {current_run_id}")
#             print(f"🔗 正在对齐新日志流: {current_log_path}\\n")

#         # 3. 稳健等待：如果还没找到 run_id，或者 log_path 还没在硬盘上物理生成，安全挂起
#         if not current_log_path or not current_log_path.exists():
#             time.sleep(2)
#             continue

#         # 4. 首次打开文件，跳到文件末尾开始追更
#         if f_handle is None:
#             f_handle = open(current_log_path, "r", encoding="utf-8")
#             f_handle.seek(0, os.SEEK_END)
#             print("✅ 成功接入数据流！开始实时广播 (同时维持心跳)：\\n")

#         # 5. 读取新的一行
#         line = f_handle.readline()
#         if not line:
#             # 维持 Colab 虚拟机的心跳保活
#             time.sleep(1)
#             continue

#         print(line, end="", flush=True)

# except KeyboardInterrupt:
#     if f_handle: f_handle.close()
#     print("\\n🛑 监控已由用户手动停止。")


📡 正在启动自适应 Live Monitor 探针 (防断连模式)...
\n🔄 [探针检测到环境更新] 发现最新的执行任务: AR_LBD_wolf_evo_20260501_193250
🔗 正在对齐新日志流: /content/drive/MyDrive/Proteina-Complexa/inference/AR_LBD_wolf_evo_20260501_193250/Live_Sync_AR_LBD_wolf_evo_20260501_193250.log\n
\n🛑 监控已由用户手动停止。


# Section 2: Target PDB Parsing & Structural Pre-processing

In [ ]:
#@title Cell_4b_PDB_Processing_and_YAML_Configuration_Generation.py
# Requirement: Implement Smart In-Place Resume. Automatically inherited run_id if physical parameters match.
# If core parameters (like PDB name) are changed, smoothly transition to generating a new session instead of crashing.
# UPDATE: Added history_run.csv tracker to maintain global session states, parameters, and prepare for downstream metrics injection.

import os, json, yaml, shutil, time, sys, datetime, re, csv
from pathlib import Path

print("🔍 Initializing Target Processing Engine (Smart In-Place Resume Architecture)...")

# ---------------------------------------------------------
# 1. Base Environment Setup
# ---------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found in root.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']

CORE_PHYSICAL_KEYS = [
    'pdb_file', 'target_chains', 'hotspots_input',
    'binder_length_min', 'binder_length_max',
    'anchor_residues', 'global_search_boundary', 'peptide_interaction_radius'
]

# ---------------------------------------------------------
# 2. Run ID Resolution & Smart Directory Targeting
# ---------------------------------------------------------
last_run_file = BASE_DIR / 'screening_results' / 'last_runs.txt'
active_run_id = None
active_final_dir = None
is_resuming = False
last_recorded_time = "N/A"

specific_resume_run_id = cfg.get("specific_resume_run_id", "").strip()
force_restart_task = cfg.get("force_restart_task", False)

if specific_resume_run_id:
    # --- Time Machine Mode ---
    try:
        hist_timestamp = specific_resume_run_id.split('_')[-2] + "_" + specific_resume_run_id.split('_')[-1]
    except:
        hist_timestamp = specific_resume_run_id.replace(task_name + "_", "")

    candidate_dir = BASE_DIR / 'screening_results' / task_name / f"final_{hist_timestamp}"
    hist_setting_path = candidate_dir / 'internal_settings.json'

    if hist_setting_path.exists():
        with open(hist_setting_path, 'r') as hf:
            hist_cfg = json.load(hf)

        # OVERWRITE CURRENT CONFIG with history config!
        cfg.update(hist_cfg)
        task_name = cfg.get('task_name', task_name)  # update if needed

        active_run_id = specific_resume_run_id
        active_final_dir = candidate_dir
        is_resuming = True
        print(f"\n🚀 Time Machine Active: Resuming from specific historical session [{specific_resume_run_id}]")
        print(f"✅ All physical parameters reverted to historical state.")

    else:
        print(f"❌ Time Machine Error: Historical folder not found at {candidate_dir.relative_to(BASE_DIR)}. Aborting.")
        sys.exit(1)

elif force_restart_task:
    print("⚡ Force Restart is ON. Bypassing history checks and generating new session.")
    is_resuming = False
else:
    # --- Normal Resume Mode ---
    if last_run_file.exists():
        with open(last_run_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            pattern = rf"Section2_TargetPreprocess \| Run_ID: ({re.escape(task_name)}_\d+_\d+)"
            for line in reversed(lines):
                match = re.search(pattern, line)
                if match:
                    candidate_run_id = match.group(1)
                    hist_timestamp = candidate_run_id.split('_')[-2] + "_" + candidate_run_id.split('_')[-1] if len(candidate_run_id.split('_')) >= 3 else candidate_run_id.split('_')[-1]
                    candidate_dir = BASE_DIR / 'screening_results' / task_name / f"final_{hist_timestamp}"
                    hist_setting_path = candidate_dir / 'internal_settings.json'

                    time_match = re.search(r"Date:\s*(.*?)(?:$|\|)", line)
                    if time_match: last_recorded_time = time_match.group(1).strip()

                    if hist_setting_path.exists():
                        with open(hist_setting_path, 'r') as hf:
                            hist_cfg = json.load(hf)
                        print(f"🔄 ARCHIVE FOUND: Validating historical session [{candidate_run_id}]...")

                        mismatches = []
                        for key in CORE_PHYSICAL_KEYS:
                            val_curr = str(cfg.get(key, '')).strip()
                            val_hist = str(hist_cfg.get(key, '')).strip()
                            if val_curr != val_hist:
                                mismatches.append(f"{key}: Current='{val_curr}' | Archive='{val_hist}'")

                        if mismatches:
                            print(f"⚠️ Notice: Core physical parameters changed. Generating a NEW session instead of resuming.")
                            for m in mismatches: print(f"   -> {m}")
                            break

                        active_run_id = candidate_run_id
                        active_final_dir = candidate_dir
                        is_resuming = True
                        print(f"✅ VALIDATION PASSED: Re-activating existing workspace for in-place continuation.")
                    break

# If not resuming, generate new
if not is_resuming:
    SESSION_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    active_run_id = f"{task_name}_{SESSION_TIMESTAMP}"
    active_final_dir = BASE_DIR / 'screening_results' / task_name / f"final_{SESSION_TIMESTAMP}"
    active_final_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 【新增】last_runs.txt 倒序模式：新记录永远在最前面 + 自动清理当前 task_name 的旧记录
# ---------------------------------------------------------
last_run_file.parent.mkdir(parents=True, exist_ok=True)

# 1. 读取现有所有记录
if last_run_file.exists():
    with open(last_run_file, 'r', encoding='utf-8') as f:
        existing_lines = f.readlines()
else:
    existing_lines = []

# 2. 过滤掉当前 task_name 的旧记录（保证同一个任务只保留最新一条）
new_lines = []
for line in existing_lines:
    if not line.startswith("Section2_TargetPreprocess") or \
       re.search(rf"Run_ID: {re.escape(task_name)}_", line) is None:
        new_lines.append(line)

# 3. 构造新记录
current_line = (
    f"Section2_TargetPreprocess | Run_ID: {active_run_id} | "
    f"Dir: {active_final_dir.name} | "
    f"Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
)

# 4. 新记录插入最顶部 → 实现倒序
new_lines.insert(0, current_line)

# 5. 覆盖写入
with open(last_run_file, 'w', encoding='utf-8') as f:
    f.writelines(new_lines)

print(f"✅ last_runs.txt 已更新为倒序模式（最新记录在最前面）")
print(f"   当前最新 Run ID: {active_run_id}")

# ---------------------------------------------------------
# 3. History Tracker (history_run.csv) Initialization
# ---------------------------------------------------------
# 构建/更新全局断点续传历史文件
HISTORY_CSV = BASE_DIR / 'screening_results' / 'history_run.csv'

# 🌟 表头明确包含 Top_Max_ipSAE 和 Overall_Data_Link
history_headers = [
    "Timestamp", "Last_Run_Time", "Task_Name", "Run_ID",
    "Program_Section", "Core_Parameters",
    "Screening_Islands", "Screening_Cycles", "Top_Max_ipSAE", "Overall_Data_Link", "Status"
]

# 如果不存在则初始化表头
if not HISTORY_CSV.exists():
    with open(HISTORY_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(history_headers)

# 整理当前写入的数据
current_time_str = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
param_summary = f"Len:{cfg['binder_length_min']}-{cfg['binder_length_max']}|HS:{cfg['hotspots_input']}"
run_status = "Resumed" if is_resuming else "New_Session"

# 将当前运行信息追加到 history_run.csv，注意：筛选出的信息留空
with open(HISTORY_CSV, 'a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([
        current_time_str,      # Timestamp
        last_recorded_time,    # Last_Run_Time
        task_name,             # Task_Name
        active_run_id,         # Run_ID
        "Section_4",           # Program_Section
        param_summary,         # Core_Parameters
        "",                    # Screening_Islands (留空给下游更新)
        "",                    # Screening_Cycles (留空给下游更新)
        "",                    # Top_Max_ipSAE (留空给下游更新)
        "",                    # Overall_Data_Link (留空给下游更新)
        run_status             # Status
    ])
print(f"📊 HISTORY TRACKER: Session logged to history_run.csv. Awaiting downstream metric injection.")


# ---------------------------------------------------------
# 4. Configuration Sync
# ---------------------------------------------------------
cfg['run_id'] = active_run_id
with open(SETTING_FILE, 'w') as f:
    json.dump(cfg, f, indent=4)

print(f"📂 Active Workspace: {active_final_dir.relative_to(BASE_DIR)}")

# ---------------------------------------------------------
# 5. PDB Parsing & Base YAML Generation
# ---------------------------------------------------------
pdb_file = cfg['pdb_file']
target_chains = cfg['target_chains']
hotspots_input = str(cfg['hotspots_input'])
binder_length_min = int(cfg['binder_length_min'])
binder_length_max = int(cfg['binder_length_max'])

TARGETS_INPUT_DIR = BASE_DIR / 'targets'
RUN_DATA_DIR = BASE_DIR / 'assets' / 'target_data' / task_name
YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'

RUN_DATA_DIR.mkdir(parents=True, exist_ok=True)
raw_pdb_path = TARGETS_INPUT_DIR / pdb_file
fixed_pdb_path = RUN_DATA_DIR / f"{task_name}_fixed.pdb"

print(f"⚙️ Processing PDB: {pdb_file}...")
valid_res_nums = set()
with open(raw_pdb_path, 'r') as f:
    for line in f:
        if line.startswith('ATOM') and line[12:16].strip() == 'CA' and line[21] == target_chains:
            try: valid_res_nums.add(int(line[22:26].strip()))
            except: continue

with open(raw_pdb_path, 'r') as f_in, open(fixed_pdb_path, 'w') as f_out:
    for line in f_in:
        if line.startswith('ATOM') and line[21] == target_chains:
            try:
                if int(line[22:26].strip()) in valid_res_nums: f_out.write(line)
            except: pass
        elif line.startswith('TER'): f_out.write(line)

res_nums = sorted(list(valid_res_nums))
ranges = []
if res_nums:
    start = prev = res_nums[0]
    for n in res_nums[1:]:
        if n == prev + 1: prev = n
        else:
            ranges.append(f"{target_chains}{start}-{prev}" if start!=prev else f"{target_chains}{start}")
            start = prev = n
    ranges.append(f"{target_chains}{start}-{prev}" if start!=prev else f"{target_chains}{start}")

hotspots_list = [int(x.strip()) for x in hotspots_input.split(',') if x.strip().isdigit()]
yaml_data = {'target_dict_cfg': {
    task_name: {
        'target_input': ','.join(ranges),
        'binder_length': [binder_length_min, binder_length_max],
        'target_chains': [target_chains],
        'pdb_id': task_name,
        'source': task_name,
        'target_filename': task_name + "_fixed",
        'hotspot_residues': [f"{target_chains}{i}" for i in hotspots_list]
    }
}}

with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

# ---------------------------------------------------------
# 6. Final Workspace Synchronization
# ---------------------------------------------------------
shutil.copy2(SETTING_FILE, active_final_dir / 'internal_settings.json')
shutil.copy2(YAML_PATH, active_final_dir / f'targets_dict_{active_run_id}.yaml')

print(f"💾 ARCHIVE SYNCED: Current dynamic settings securely written to {active_final_dir.name}")
print(f"🎯 SUCCESS: Base YAML generated. Ready for downstream sections.")

🔍 Initializing Target Processing Engine (Smart In-Place Resume Architecture)...
🔄 ARCHIVE FOUND: Validating historical session [AR_LBD_wolf_evo_20260501_193250]...
✅ VALIDATION PASSED: Re-activating existing workspace for in-place continuation.
✅ last_runs.txt 已更新为倒序模式（最新记录在最前面）
   当前最新 Run ID: AR_LBD_wolf_evo_20260501_193250
📊 HISTORY TRACKER: Session logged to history_run.csv. Awaiting downstream metric injection.
📂 Active Workspace: screening_results/AR_LBD_wolf_evo/final_20260501_193250
⚙️ Processing PDB: AR_LBD_ChainB_5JJM.pdb...
💾 ARCHIVE SYNCED: Current dynamic settings securely written to final_20260501_193250
🎯 SUCCESS: Base YAML generated. Ready for downstream sections.


# Section 3: Split-GFP Tutorial — Natural Baseline Evaluation & Complex Generation

In [ ]:
#@title Cell_5_Automated_Groove_Scanning_Engine.py
# Requirement: Implement Morphological Bottleneck Severing via EDT to mathematically sever narrow topological channels.
# Integrate in-notebook py3Dmol visualization to render the host protein as a gray mesh and highlight identified grooves as an orange surface.

import os, json, sys, yaml
import numpy as np
import pandas as pd
import py3Dmol
from pathlib import Path
from scipy.spatial import KDTree
from scipy.ndimage import binary_closing, distance_transform_edt, label

from IPython.display import display, HTML

print("🧬 Initializing Morphological Bottleneck Severing Engine (Optimized for Split-Barrels)...")

# ---------------------------------------------------------
# 1. Load Configurations & YAML Interface
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'
YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'

if not SETTING_FILE.exists() or not YAML_PATH.exists():
    print("❌ Error: Required configuration files not found.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
target_chains = cfg['target_chains']

scan_shallow_groove = bool(cfg.get('scan_shallow_groove', True))
scan_deep_pocket = bool(cfg.get('scan_deep_pocket', False))
scan_hydrophobic_patch = bool(cfg.get('scan_hydrophobic_patch', True))
scan_charged_patch = bool(cfg.get('scan_charged_patch', False))

MIN_POCKET_SIZE = int(cfg.get('min_pocket_size', 6))
HYDROPHOBIC_THRESHOLD = float(cfg.get('hydrophobic_threshold', 0.35))
CHARGED_THRESHOLD = float(cfg.get('charged_threshold', 0.30))

with open(YAML_PATH, 'r') as f:
    yaml_data = yaml.safe_load(f)
try:
    target_info = yaml_data['target_dict_cfg'][task_name]
    pdb_filename = target_info['target_filename'] + ".pdb"
    SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / target_info['source'] / pdb_filename
except KeyError:
    print(f"❌ Error: Target '{task_name}' not properly defined in YAML.")
    sys.exit(1)

RESULTS_CSV = BASE_DIR / 'screening_results' / task_name / f"{task_name}_automated_scans.csv"
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

KD_SCALE = {'ILE': 4.5, 'VAL': 4.2, 'LEU': 3.8, 'PHE': 2.8, 'CYS': 2.5, 'MET': 1.9, 'ALA': 1.8, 'GLY': -0.4,
            'THR': -0.7, 'SER': -0.8, 'TRP': -0.9, 'TYR': -1.3, 'PRO': -1.6, 'HIS': -3.2, 'GLU': -3.5,
            'GLN': -3.5, 'ASP': -3.5, 'ASN': -3.5, 'LYS': -3.9, 'ARG': -4.5}
CHARGE_SCALE = {'ARG': '+', 'LYS': '+', 'HIS': '+', 'ASP': '-', 'GLU': '-'}

# ---------------------------------------------------------
# 2. PDB Parsing & Grid Initialization
# ---------------------------------------------------------
ca_coords, res_ids, res_names = [], [], []
all_coords = []

with open(SOURCE_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[21] == target_chains:
            coord = [float(line[30:38]), float(line[38:46]), float(line[46:54])]
            all_coords.append(coord)
            if line[12:16].strip() == "CA":
                res_names.append(line[17:20].strip())
                res_ids.append(int(line[22:26].strip()))
                ca_coords.append(coord)

ca_coords = np.array(ca_coords)
res_ids = np.array(res_ids)
res_names = np.array(res_names)
all_coords = np.array(all_coords)
protein_kdtree = KDTree(ca_coords)

print("   -> Constructing 3D Voxel Cartography...")
GRID_RES = 1.0
PADDING = 15.0
coords_min = all_coords.min(axis=0) - PADDING
coords_max = all_coords.max(axis=0) + PADDING
grid_shape = np.ceil((coords_max - coords_min) / GRID_RES).astype(int)

protein_grid = np.zeros(grid_shape, dtype=bool)
for pt in all_coords:
    idx = np.round((pt - coords_min) / GRID_RES).astype(int)
    x, y, z = idx
    protein_grid[max(0, x-1):x+2, max(0, y-1):y+2, max(0, z-1):z+2] = True

# ---------------------------------------------------------
# 3. Virtual Shrink-Wrap & Bottleneck Severing
# ---------------------------------------------------------
print("   -> Deploying sturdy shrink-wrap boundary to define cavities...")
PROBE_RADIUS = 15.0
r = int(np.ceil(PROBE_RADIUS / GRID_RES))
zz, yy, xx = np.ogrid[-r:r+1, -r:r+1, -r:r+1]
sphere_struct = xx**2 + yy**2 + zz**2 <= r**2

wrapped_protein = binary_closing(protein_grid, structure=sphere_struct)
cavity_mask = wrapped_protein & ~protein_grid

print("   -> Severing narrow topological channels (EDT Bottleneck Erosion)...")
empty_space = ~protein_grid
edt = distance_transform_edt(empty_space) * GRID_RES

SEVERING_RADIUS = 2.5
isolated_cores = cavity_mask & (edt >= SEVERING_RADIUS)

if not np.any(isolated_cores):
    print("🛑 Error: No core cavities survived the bottleneck severing process.")
    sys.exit(0)

# ---------------------------------------------------------
# 4. Spatial Clustering of Isolated Cores
# ---------------------------------------------------------
print("   -> Clustering cleanly severed topological features...")
labeled_array, num_features = label(isolated_cores)
vertex_clusters = []

for i in range(1, num_features + 1):
    core_indices = np.argwhere(labeled_array == i)
    if len(core_indices) >= 40:
        voxel_coords = core_indices * GRID_RES + coords_min
        vertex_clusters.append(voxel_coords)

if not vertex_clusters:
    print("🛑 Scanning Complete: No disconnected core trenches found meeting volume limits.")
    sys.exit(0)

# ---------------------------------------------------------
# 5. Restoring Contact Surfaces (Dilation to CA atoms)
# ---------------------------------------------------------
pocket_results = []
categorized_findings = {'Grooves': [], 'Pockets': [], 'Hydrophobic_Patches': [], 'Charged_Patches': []}
count_groove = 0; count_pocket = 0; count_hydro = 0; count_charge = 0

for cluster_coords in vertex_clusters:
    neighbors = protein_kdtree.query_ball_point(cluster_coords, r=6.5)

    res_indices = set()
    for nl in neighbors: res_indices.update(nl)

    if len(res_indices) < MIN_POCKET_SIZE:
        continue

    cluster_res_indices = list(res_indices)
    c_res_names = res_names[cluster_res_indices]
    total = len(cluster_res_indices)

    topo_type = "Deep Pocket" if total > 25 else "Shallow Groove"

    hydro_count = sum(1 for name in c_res_names if KD_SCALE.get(name, 0) > 0)
    pos_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '+')
    neg_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '-')
    charge_count = pos_count + neg_count

    hydro_ratio = hydro_count / total
    charge_ratio = charge_count / total

    is_hydro = hydro_ratio >= HYDROPHOBIC_THRESHOLD
    is_charge = charge_ratio >= CHARGED_THRESHOLD

    c_res_ids = res_ids[cluster_res_indices]
    patch_str = ",".join(map(str, sorted(c_res_ids)))
    pymol_sel = "+".join(map(str, sorted(c_res_ids)))

    site_id = f"Site_{len(pocket_results)+1}"
    pocket_results.append({
        'Site_ID': site_id, 'Topology': topo_type, 'Residues': total,
        'Hydrophobic_Ratio': round(hydro_ratio, 3), 'Charged_Ratio': round(charge_ratio, 3),
        'Hotspot_Sequence': patch_str
    })

    # Added raw c_res_ids list to tuple for py3Dmol rendering
    if topo_type == "Shallow Groove" and scan_shallow_groove:
        count_groove += 1
        spec_id = f"Groove_{count_groove}"
        cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color orange, {spec_id}; show surface, {spec_id}"
        categorized_findings['Grooves'].append((spec_id, cmd, list(c_res_ids)))

    if topo_type == "Deep Pocket" and scan_deep_pocket:
        count_pocket += 1
        spec_id = f"Pocket_{count_pocket}"
        cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color magenta, {spec_id}; show surface, {spec_id}"
        categorized_findings['Pockets'].append((spec_id, cmd, list(c_res_ids)))

    if is_hydro and scan_hydrophobic_patch:
        count_hydro += 1
        spec_id = f"Hydro_Patch_{count_hydro}"
        cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color yellow, {spec_id}; show surface, {spec_id}"
        categorized_findings['Hydrophobic_Patches'].append((spec_id, cmd, list(c_res_ids)))

    if is_charge and scan_charged_patch:
        count_charge += 1
        spec_id = f"Charge_Patch_{count_charge}"
        cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color cyan, {spec_id}; show surface, {spec_id}"
        categorized_findings['Charged_Patches'].append((spec_id, cmd, list(c_res_ids)))

# ---------------------------------------------------------
# 6. Printed Output
# ---------------------------------------------------------
print("\n" + "="*70)
print(f"Summary: {count_groove} Grooves, {count_pocket} Pockets, {count_hydro} Hydro Patches, {count_charge} Charged Patches.")
print("="*70 + "\n")

if categorized_findings['Grooves']:
    print("Grooves:")
    for t_id, cmd, _ in categorized_findings['Grooves']:
        print(f"  {t_id}:\n  {cmd}\n")

if categorized_findings['Pockets']:
    print("Pockets:")
    for t_id, cmd, _ in categorized_findings['Pockets']:
        print(f"  {t_id}:\n  {cmd}\n")

if categorized_findings['Hydrophobic_Patches']:
    print("Patches (Hydrophobic):")
    for t_id, cmd, _ in categorized_findings['Hydrophobic_Patches']:
        print(f"  {t_id}:\n  {cmd}\n")

if categorized_findings['Charged_Patches']:
    print("Patches (Charged):")
    for t_id, cmd, _ in categorized_findings['Charged_Patches']:
        print(f"  {t_id}:\n  {cmd}\n")

if pocket_results:
    pd.DataFrame(pocket_results).to_csv(RESULTS_CSV, index=False)

# ---------------------------------------------------------
# 7. In-Notebook 3D Morphological Rendering
# ---------------------------------------------------------
if SOURCE_PDB_PATH.exists() and count_groove > 0:
    print("\n   -> Rendering 3D Topographical Map (Gray Mesh for Protein, Orange Surface for Grooves)...")

    with open(SOURCE_PDB_PATH, 'r') as f:
        pdb_data = f.read()

    viewer = py3Dmol.view(width=800, height=500)
    viewer.addModel(pdb_data, 'pdb')

    # Render main protein as gray cartoon and gray mesh (wireframe surface)
    viewer.setStyle({'chain': target_chains}, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.7}})
    viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'wireframe': True}, {'chain': target_chains})

    # Iterate through found grooves and render them as orange solid surfaces
    for spec_id, cmd, resi_list in categorized_findings['Grooves']:
        str_resis = [str(r) for r in resi_list]
        groove_sel = {'chain': target_chains, 'resi': str_resis}

        # Color the backbone slightly for better internal contrast
        viewer.setStyle(groove_sel, {'cartoon': {'color': 'orange', 'opacity': 1.0}})
        # Overlay the solid orange surface
        viewer.addSurface(py3Dmol.SES, {'color': 'orange', 'opacity': 1.0}, groove_sel)

    viewer.zoomTo()
    viewer.show()
elif count_groove == 0:
    print("\n   -> Visualization skipped: No grooves identified meeting the designated criteria.")

# Purpose: Analyze spatial topologies to locate protein interaction sites, generating Pymol commands and in-notebook 3D mesh/surface renderings.
# Upstream Code: Global configuration settings.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-04 13:03 EDT.
# Changed Lines:
# - Inserted Lines 11-12: Imported py3Dmol and IPython display modules.
# - Modified Lines 160-179: Appended the raw list of residue integers `list(c_res_ids)` to the `categorized_findings` dictionary values.
# - Inserted Lines 212-237: Added Section 7 logic. Constructed interactive spatial viewer to render target chains as gray wireframe (mesh) and highlighted grooves as orange solid surfaces utilizing boolean residue mapping.

In [ ]:
#@title # Cell_5b_Automated_Groove_Scanning_Engine.py
# # Requirement: Paradigm shift to Opposing Ridges Void Midpoint (ORVM) algorithm. Specifically targets the user's request to exclusively highlight the two parallel ridges forming a trench. It identifies pairs of sequentially distant atoms separated by an 8.5-13.5A continuous void, meticulously isolating the "left bank" and "right bank" of missing beta-strands.

# import os, json, sys, yaml
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from scipy.spatial import KDTree

# print("🔍 Initializing Opposing Ridges Void Midpoint (ORVM) Engine...")

# # ---------------------------------------------------------
# # 1. Load Configurations & YAML Interface
# # ---------------------------------------------------------
# BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
# SETTING_FILE = BASE_DIR / 'internal_settings.json'
# YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'

# if not SETTING_FILE.exists() or not YAML_PATH.exists():
#     print("❌ Error: Required configuration files not found.")
#     sys.exit(1)

# with open(SETTING_FILE, 'r') as f:
#     cfg = json.load(f)

# task_name = cfg['task_name']
# target_chains = cfg['target_chains']

# scan_shallow_groove = bool(cfg.get('scan_shallow_groove', True))
# scan_deep_pocket = bool(cfg.get('scan_deep_pocket', False))
# scan_hydrophobic_patch = bool(cfg.get('scan_hydrophobic_patch', True))
# scan_charged_patch = bool(cfg.get('scan_charged_patch', False))

# MIN_POCKET_SIZE = int(cfg.get('min_pocket_size', 6))
# HYDROPHOBIC_THRESHOLD = float(cfg.get('hydrophobic_threshold', 0.35))
# CHARGED_THRESHOLD = float(cfg.get('charged_threshold', 0.30))

# with open(YAML_PATH, 'r') as f:
#     yaml_data = yaml.safe_load(f)
# try:
#     target_info = yaml_data['target_dict_cfg'][task_name]
#     pdb_filename = target_info['target_filename'] + ".pdb"
#     SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / target_info['source'] / pdb_filename
# except KeyError:
#     print(f"❌ Error: Target '{task_name}' not properly defined in YAML.")
#     sys.exit(1)

# RESULTS_CSV = BASE_DIR / 'screening_results' / task_name / f"{task_name}_automated_scans.csv"
# RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

# KD_SCALE = {'ILE': 4.5, 'VAL': 4.2, 'LEU': 3.8, 'PHE': 2.8, 'CYS': 2.5, 'MET': 1.9, 'ALA': 1.8, 'GLY': -0.4,
#             'THR': -0.7, 'SER': -0.8, 'TRP': -0.9, 'TYR': -1.3, 'PRO': -1.6, 'HIS': -3.2, 'GLU': -3.5,
#             'GLN': -3.5, 'ASP': -3.5, 'ASN': -3.5, 'LYS': -3.9, 'ARG': -4.5}
# CHARGE_SCALE = {'ARG': '+', 'LYS': '+', 'HIS': '+', 'ASP': '-', 'GLU': '-'}

# # ---------------------------------------------------------
# # 2. PDB Parsing & Coordinate Extraction
# # ---------------------------------------------------------
# ca_coords, res_ids, res_names = [], [], []

# with open(SOURCE_PDB_PATH, 'r') as f:
#     for line in f:
#         if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
#             res_names.append(line[17:20].strip())
#             res_ids.append(int(line[22:26].strip()))
#             ca_coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])

# ca_coords = np.array(ca_coords)
# res_ids = np.array(res_ids)
# res_names = np.array(res_names)
# protein_kdtree = KDTree(ca_coords)

# # ---------------------------------------------------------
# # 3. Opposing Ridges Void Midpoint (ORVM) Algorithm
# # ---------------------------------------------------------
# print("   -> Scanning for opposing structural ridges separated by a continuous void...")

# # Extract all CA atom pairs within a 14.0A distance
# pairs = protein_kdtree.query_pairs(r=14.0)
# valid_pairs = []
# midpoints = []

# for i, j in pairs:
#     # a. Physical cross-trench distance: The gap of a missing Beta strand typically maintains a bank-to-bank distance between 8.5A and 13.5A
#     dist = np.linalg.norm(ca_coords[i] - ca_coords[j])
#     if dist < 8.5 or dist > 13.5:
#         continue

#     # b. Sequence disconnection filter: Ensure the pairs belong to spatially adjacent but sequentially distant strands, avoiding continuous backbone
#     if np.abs(res_ids[i] - res_ids[j]) < 15:
#         continue

#     # c. Vacuum midpoint generation: Calculate the spatial geometric center between the pair
#     M = (ca_coords[i] + ca_coords[j]) / 2.0

#     # d. Midpoint clearance validation: The midpoint must exist in a void (at least 3.8A away from any CA backbone atom)
#     nearest_dist, _ = protein_kdtree.query(M, k=1)
#     if nearest_dist < 3.8:
#         continue

#     # e. Surface vs Core Filter:
#     # A true surface groove's midpoint is surrounded only by the floor and sides (10-32 atoms).
#     # If trapped inside a chromophore cavity, it would be surrounded by the entire barrel from all directions (> 35 atoms).
#     neighbors_count = len(protein_kdtree.query_ball_point(M, r=12.0))
#     if neighbors_count < 10 or neighbors_count > 32:
#         continue

#     valid_pairs.append((i, j))
#     midpoints.append(M)

# midpoints = np.array(midpoints)

# if len(midpoints) == 0:
#     print("🛑 Scanning Complete: No opposing ridges across a valid topological trench were found.")
#     sys.exit(0)

# # ---------------------------------------------------------
# # 4. Clustering Validated Midpoints into Continuous Trenches
# # ---------------------------------------------------------
# print("   -> Stitching cross-trench voids to isolate distinct binding grooves...")
# midpoint_kdtree = KDTree(midpoints)

# # If multiple vacuum midpoints are exceedingly close (4.5A), they belong to the same continuous groove
# mp_pairs = midpoint_kdtree.query_pairs(r=4.5)

# adj_list = {i: set() for i in range(len(midpoints))}
# for i, j in mp_pairs:
#     adj_list[i].add(j); adj_list[j].add(i)

# visited = set()
# final_ridge_clusters = []

# for i in range(len(midpoints)):
#     if i not in visited:
#         queue = [i]
#         component = []
#         while queue:
#             node = queue.pop(0)
#             if node not in visited:
#                 visited.add(node)
#                 component.append(node)
#                 queue.extend(list(adj_list[node] - visited))

#         # A valid biological cleft will generate at least 8 valid vacuum midpoint pairs
#         if len(component) >= 8:
#             # Extract all atom indices from the left and right banks contributing to this groove
#             ridge_indices = set()
#             for idx in component:
#                 u, v = valid_pairs[idx]
#                 ridge_indices.add(u)
#                 ridge_indices.add(v)
#             final_ridge_clusters.append(list(ridge_indices))

# if not final_ridge_clusters:
#     print("🛑 Scanning Complete: Detected opposing ridges were too fragmented to form a continuous groove.")
#     sys.exit(0)

# # ---------------------------------------------------------
# # 5. Categorization & Annotation
# # ---------------------------------------------------------
# pocket_results = []
# categorized_findings = {'Grooves': [], 'Pockets': [], 'Hydrophobic_Patches': [], 'Charged_Patches': []}

# count_groove = 0; count_pocket = 0; count_hydro = 0; count_charge = 0

# for cluster in final_ridge_clusters:
#     c_res_names = res_names[cluster]
#     total = len(cluster)

#     # This pairing algorithm purely extracts the elevated opposing ridges on both sides
#     topo_type = "Deep Pocket" if total > 35 else "Shallow Groove"

#     hydro_count = sum(1 for name in c_res_names if KD_SCALE.get(name, 0) > 0)
#     pos_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '+')
#     neg_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '-')
#     charge_count = pos_count + neg_count

#     hydro_ratio = hydro_count / total
#     charge_ratio = charge_count / total

#     is_hydro = hydro_ratio >= HYDROPHOBIC_THRESHOLD
#     is_charge = charge_ratio >= CHARGED_THRESHOLD

#     c_res_ids = res_ids[cluster]
#     patch_str = ",".join(map(str, sorted(c_res_ids)))
#     pymol_sel = "+".join(map(str, sorted(c_res_ids)))

#     site_id = f"Site_{len(pocket_results)+1}"
#     pocket_results.append({
#         'Site_ID': site_id, 'Topology': topo_type, 'Residues': total,
#         'Hydrophobic_Ratio': round(hydro_ratio, 3), 'Charged_Ratio': round(charge_ratio, 3),
#         'Hotspot_Sequence': patch_str
#     })

#     if topo_type == "Shallow Groove" and scan_shallow_groove:
#         count_groove += 1
#         spec_id = f"Groove_{count_groove}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color orange, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Grooves'].append((spec_id, cmd))

#     if topo_type == "Deep Pocket" and scan_deep_pocket:
#         count_pocket += 1
#         spec_id = f"Pocket_{count_pocket}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color magenta, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Pockets'].append((spec_id, cmd))

#     if is_hydro and scan_hydrophobic_patch:
#         count_hydro += 1
#         spec_id = f"Hydro_Patch_{count_hydro}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color yellow, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Hydrophobic_Patches'].append((spec_id, cmd))

#     if is_charge and scan_charged_patch:
#         count_charge += 1
#         spec_id = f"Charge_Patch_{count_charge}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color cyan, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Charged_Patches'].append((spec_id, cmd))

# # ---------------------------------------------------------
# # 6. Printed Output
# # ---------------------------------------------------------
# print("\n" + "="*70)
# print(f"Summary: {count_groove} Grooves, {count_pocket} Pockets, {count_hydro} Hydro Patches, {count_charge} Charged Patches.")
# print("="*70 + "\n")

# if categorized_findings['Grooves']:
#     print("Grooves:")
#     for t_id, cmd in categorized_findings['Grooves']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Pockets']:
#     print("Pockets:")
#     for t_id, cmd in categorized_findings['Pockets']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Hydrophobic_Patches']:
#     print("Patches (Hydrophobic):")
#     for t_id, cmd in categorized_findings['Hydrophobic_Patches']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Charged_Patches']:
#     print("Patches (Charged):")
#     for t_id, cmd in categorized_findings['Charged_Patches']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if pocket_results:
#     pd.DataFrame(pocket_results).to_csv(RESULTS_CSV, index=False)

# # ==============================================================================
# # Update Log:
# # Purpose: Deployed the Opposing Ridges Void Midpoint (ORVM) algorithm specifically designed to fulfill the user's intent to highlight *only* the two parallel ridges flanking a trench. It calculates geometric midpoints between sequentially distant atoms (gap 8.5-13.5A). It strictly guarantees void clearance (no CA within 3.8A) and rejects central barrel voids by limiting neighborhood counts (10-32 atoms within 12A). By extracting only the CA pairs generating these clustered void midpoints, it provides an exquisite isolation of trench lips with zero bleed to the floor or surrounding convex topology.
# # Upstream Code: Complete algorithmic refactor of Section 3-4.
# # Runtime Environment: Google Colab.
# # Generation Time: 2026-04-03 22:15 EDT.
# # Changed Lines:
# # - Lines 85-115: Implemented Void Midpoint generation and verification logic.
# # - Lines 103: `nearest_dist < 3.8` enforces perfect physical emptiness between the two ridges.
# # - Lines 110: `neighbors_count < 10 or neighbors_count > 32` prevents central GFP barrel hollows from registering as surface grooves.
# # - Lines 142-146: Extracted only the original `(u, v)` pair atom indices that formed the valid trench void, directly producing the two distinct mountain ridges.
# # ==============================================================================

In [ ]:

# =================================================================
#@title Cell_6_Automated Pipeline for GFP Natural Substrate Benchmarking, Metrics Dashboard, and 3D Visualization
# =================================================================

import os, sys, subprocess, shutil, gc, torch, time, json
import pandas as pd
from pathlib import Path
import py3Dmol

#--- [Configuration Section] ---
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
TARGET_PDB_NAME = "GFP_1_10.pdb"
BINDER_PDB_NAME = "GFP_11.pdb"
TARGET_TASK = "GFP"
RUN_NAME = "Run_Native_Baseline"
JOB_ID = 0

# --- [Global Settings Import] ---
ROOT_SETTING = BASE_DIR / 'internal_settings.json'
if ROOT_SETTING.exists():
    with open(ROOT_SETTING, 'r') as f: root_cfg = json.load(f)
else:
    root_cfg = {}
evaluation_batch_size = int(root_cfg.get('evaluation_batch_size', 1))

# --- [Path Pre-check] ---
target_pdb_path = BASE_DIR / "targets" / TARGET_PDB_NAME
binder_pdb_path = BASE_DIR / "targets" / BINDER_PDB_NAME

if not target_pdb_path.exists() or not binder_pdb_path.exists():
    print(f"Error: Specified PDB file could not be located.")
    sys.exit(1)

inference_dir = BASE_DIR / 'inference' / f'search_binder_local_pipeline_{TARGET_TASK}_{RUN_NAME}'
if inference_dir.exists(): shutil.rmtree(inference_dir)
inference_dir.mkdir(parents=True, exist_ok=True)

mock_design_id = f"job_{JOB_ID}_native_baseline"
mock_job_dir = inference_dir / mock_design_id
mock_job_dir.mkdir(parents=True, exist_ok=True)

# --- [Core Patch 1: Generic Parameter] ---
utils_file = BASE_DIR / 'src/proteinfoundation/metrics/metric_utils.py'
if utils_file.exists():
    with open(utils_file, 'r') as f: content = f.read()
    if "def replace_seq_in_generated_pdb(" in content and "def _disabled_replace" not in content:
        shutil.copy(utils_file, str(utils_file) + ".original")
        safe_override = (
            "def replace_seq_in_generated_pdb(*args, **kwargs):\n"
            "    import shutil, os\n"
            "    for val in list(args) + list(kwargs.values()):\n"
            "        if isinstance(val, str) and val.endswith('.pdb'):\n"
            "            out_path = val.replace('.pdb', '_updated.pdb')\n"
            "            if not os.path.exists(out_path): shutil.copy(val, out_path)\n"
            "    return\n\n"
            "def _disabled_replace("
        )
        with open(utils_file, 'w') as f: f.write(content.replace("def replace_seq_in_generated_pdb(", safe_override))

# --- [Core Patch 2: Resolving Misaligned Index and Out-of-Bounds Bug] ---
binder_metrics_file = BASE_DIR / 'src/proteinfoundation/metrics/binder_metrics.py'
if binder_metrics_file.exists():
    with open(binder_metrics_file, 'r') as f:
        bm_content = f.read()
    buggy_line = 'interface_seq = "".join([sequence[i] for i in interface_residues])'
    safe_line = 'interface_seq = "".join([sequence[i] for i in interface_residues if i < len(sequence)])'
    if buggy_line in bm_content:
        shutil.copy(binder_metrics_file, str(binder_metrics_file) + ".bak")
        with open(binder_metrics_file, 'w') as f:
            f.write(bm_content.replace(buggy_line, safe_line))
        print("Patch applied: Fixed index out-of-bounds crash in interface_seq.")

#--- [PDB Assembly (Chain A+B)] ---
target_pdb_eval_path = mock_job_dir / f"{mock_design_id}.pdb"
atom_idx = 1
with open(target_pdb_eval_path, 'w') as f_out:
    for chain, p in [('A', target_pdb_path), ('B', binder_pdb_path)]:
        with open(p, 'r') as f_in:
            for line in f_in:
                if line.startswith(("ATOM", "HETATM")):
                    f_out.write(line[:6] + f"{atom_idx:>5}" + line[11:21] + chain + line[22:])
                    atom_idx += 1
        f_out.write("TER\n")
    f_out.write("END\n")

# --- [4. VRAM Cleanup & Function Execution] ---
def run_PF_command(command_str):
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    env = os.environ.copy()
    env.update({'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'TF_FORCE_GPU_ALLOW_GROWTH': 'true'})

    full_command = f"source env.sh && {command_str}"
    process = subprocess.Popen(full_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, shell=True, executable='/bin/bash', cwd=str(BASE_DIR), env=env)
    for line in process.stdout: print(line, end='', flush=True)
    return process.wait()

base_args = (f"--config-path {BASE_DIR}/configs --config-name search_binder_local_pipeline "
             f"++job_id={JOB_ID} ++generation.task_name={TARGET_TASK} ++run_name={RUN_NAME} "
             f"++generation.dataloader.dataset.conditional_features.0.pdb_path={target_pdb_path} "
             f"++generation.dataloader.batch_size={evaluation_batch_size} ++hydra.job.chdir=False ++eval_nmodels=1")

print(f"\n AlphaFold2 Evaluation Started...")
ret = run_PF_command(f"python3 -m proteinfoundation.evaluate {base_args} ++run_evaluate=True ++evaluate.num_recycles=1 ++evaluate.pad_to_max_length=True ++evaluate.use_msa=False ++evaluate.msa_mode=single_sequence ++eval_njobs=1 ++evaluate.save_pae=True ++evaluate.dump_json=True ++evaluate.save_outputs=True")

if ret == 0:
    run_PF_command(f"python3 -m proteinfoundation.analyze {base_args} ++run_analyze=True")
    eval_out_dir = BASE_DIR / 'evaluation_results' / f'search_binder_local_pipeline_{TARGET_TASK}_{RUN_NAME}'
    csv_files = list(eval_out_dir.glob("binder_results_*.csv"))

    if csv_files:

        df = pd.read_csv(max(csv_files, key=lambda x: x.stat().st_size))
        d = df.iloc[0]


        print("\n🏆 Gold Standard Baseline (10AA Core Complex Reference):")
        header_format = "{:<8} | {:<15} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8}"
        print(header_format.format("Rank", "Design_ID", "ipSAE", "pTM", "iPTM", "pLDDT", "RMSD"))
        print("-" * 85)
        print(header_format.format(
            "CORE", "GFP_11_CORE",
            f"{d.get('self_complex_max_ipSAE', 0):.4f}",
            f"{d.get('self_complex_pTM', 0):.4f}",
            f"{d.get('self_complex_i_pTM', 0):.4f}",
            f"{d.get('self_complex_pLDDT', 0):.2f}",
            f"{d.get('self_binder_scRMSD_ca', 0):.2f}Å"
        ))
        print("-" * 85)


        print("\n🎯 --- GFP_1_10.pdb + GFP_11.pdb ---")
        for index, row in df.iterrows():

            print(f"  > i_pSAE: {row.get('self_complex_max_ipSAE', row.get('complex_i_pAE', 'N/A'))}")
            print(f"  > pTM            : {row.get('self_complex_pTM', row.get('complex_pTM', 'N/A'))}")
            print(f"  > ipTM           : {row.get('self_complex_i_pTM', row.get('complex_ipTM', 'N/A'))}")
            print(f"  > pLDDT   : {row.get('self_complex_pLDDT', row.get('complex_pLDDT', 'N/A'))}")
            print(f"  > Binder scRMSD  : {row.get('self_binder_scRMSD_ca', row.get('binder_scRMSD_ca', 'N/A'))}")
            print("\n Complete Data Sheet：")
            print(df.T)

# --- Save CSV outside the loop, alongside the PDB dir ---
        target_csv_path = mock_job_dir / f"{mock_design_id}_metrics.csv"
        df.to_csv(target_csv_path, header=True)
        print(f"\n✅ Data Sheet saved alongside PDB at: {target_csv_path.relative_to(BASE_DIR)}")
   # --- [6. Complex Visualization] ---


        COMPLEX_PDB_PATH = eval_out_dir / mock_design_id / "AF2" / f"{mock_design_id}_self_seq_0_model1.pdb"
        SCAN_CSV_PATH = BASE_DIR / 'screening_results' / TARGET_TASK / f"{TARGET_TASK}_automated_scans.csv"

        if COMPLEX_PDB_PATH.exists():
            print(f"\n   -> Rendering 3D Complex Map: Target (Gray Mesh), Grooves (Orange Surface), Binder (Bright Green)...")
            with open(COMPLEX_PDB_PATH, 'r') as f:
                pdb_data = f.read()

            viewer = py3Dmol.view(width=800, height=500)
            viewer.addModel(pdb_data, 'pdb')

            # 1. Target Base: Gray cartoon and gray mesh (wireframe surface)
            target_sel = {'chain': 'A'}
            viewer.setStyle(target_sel, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.7}})
            viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'wireframe': True}, target_sel)

            # 2. Target Grooves: Parse CSV and apply Orange surface
            if SCAN_CSV_PATH.exists():
                df_scan = pd.read_csv(SCAN_CSV_PATH)
                groove_df = df_scan[df_scan['Topology'] == 'Shallow Groove']
                groove_resis = []
                for _, row in groove_df.iterrows():
                    # Handle potential float/nan issues and split comma-separated strings
                    seq_val = str(row.get('Hotspot_Sequence', ''))
                    if seq_val and seq_val != 'nan':
                        groove_resis.extend(seq_val.split(','))

                if groove_resis:
                    groove_sel = {'chain': 'A', 'resi': groove_resis}
                    viewer.setStyle(groove_sel, {'cartoon': {'color': 'orange', 'opacity': 1.0}})
                    viewer.addSurface(py3Dmol.SES, {'color': 'orange', 'opacity': 1.0}, groove_sel)

            # 3. Binder: Bright green cartoon and solid surface
            binder_sel = {'chain': 'B'}
            viewer.setStyle(binder_sel, {'cartoon': {'color': 'lime', 'opacity': 1.0}})
            viewer.addSurface(py3Dmol.SES, {'color': 'lime', 'opacity': 1.0}, binder_sel)

            viewer.zoomTo()
            viewer.show()

# Purpose: Merge upstream topological data with the structural evaluation viewer. Reads the pre-calculated CSV to render specific target residues as orange grooves, maintaining gray wireframes for the rest of the target and bright green for the binder.
# Upstream Code: Relies on Cell 11 for the generation of the automated_scans.csv file.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-04 19:49 EDT.
# Changed Lines:
# * Added pandas import and SCAN_CSV_PATH resolution.
# * Inserted logic block #2 to read CSV, filter for 'Shallow Groove', extract 'Hotspot_Sequence', and apply py3Dmol styling dynamically to those specific residues on chain A.
        else:
            print(f"\n   -> ⚠️ Visualization skipped: Could not find complex PDB files in : {COMPLEX_PDB_PATH}")

else:
    print(f"\n❌ Evaluation failed with code: {ret}。")


 AlphaFold2 Evaluation Started...
Complexa environment initialized for uv runtime.



KeyboardInterrupt



# Section 6: Screening Functions

In [ ]:

# @title Cell_19_Topological_Seeding_and_Visualization.py
# Requirements:
# - Read target_fixed.pdb & extract CA and CB coordinates.
# - Dynamic Representative Atom: Use CB for side-chain representation; fallback to CA for GLY or missing atoms.
# - Calculate surface exposure using KDTree filtering (10A radius, <20 CB/CA neighbors) to preserve Alpha-helices.
# - Extract dilation ring (5-8A) around the user-provided initial hotspots.
# - Apply Greedy Minimum Set Cover (5A sphere) to establish non-overlapping satellite islands.
# - Section 7 Smart Restart: Read force_restart_section_7 to wipe old geometric manifests if triggered.
# - Export the purely geometric satellite islands to Geometric_Islands_Manifest.json.
# - Visualizations: py3Dmol showing gray mesh (SES) for target, solid orange (VDW) for core hotspots, and solid white (VDW) for new islands. Integer casting implemented for resi selectors.
# - Strictly English comments and variables.

import os
import sys
import json
import string
import re
from pathlib import Path
import numpy as np
from scipy.spatial import KDTree
import py3Dmol

# --- 1. Objective and Setup ---
print("🌍 Initializing Cell 19: Topological Seeding (CB-Anchored) and Visualization...")

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
ROOT_SETTING = BASE_DIR / 'internal_settings.json'

if not ROOT_SETTING.exists():
    print("❌ Error: Root configuration missing.")
    sys.exit(1)

with open(ROOT_SETTING, 'r') as f:
    root_cfg = json.load(f)

task_name = root_cfg['task_name']
last_run_file = BASE_DIR / 'screening_results' / 'last_runs.txt'

run_id, target_dir_name = None, None
if last_run_file.exists():
    with open(last_run_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        pattern = rf"Section2_TargetPreprocess \| Run_ID: ({re.escape(task_name)}_\d+_\d+) \| Dir: (final_\d+_\d+)"
        for line in reversed(lines):
            match = re.search(pattern, line)
            if match:
                run_id = match.group(1)
                target_dir_name = match.group(2)
                break

if not run_id:
    print(f"❌ Error: Run history not found for {task_name}.")
    sys.exit(1)

final_dir = BASE_DIR / 'screening_results' / task_name / target_dir_name
# ==================== 【新增】Run ID 启动打印（便于续传）====================
print("\n" + "="*80)
print(f"🔑 【Cell 21 Visualizations 启动】 当前全局 Run ID: {run_id}")
print(f"📁 当前 Workspace 目录: {final_dir.relative_to(BASE_DIR) if final_dir else 'N/A'}")
print(f"💡 续传时请直接复制以下内容填入 Section 0 的 specific_resume_run_id 字段：")
print(f"   specific_resume_run_id = \"{run_id}\"")
print("="*80 + "\n")
# ============================================================================
ISOLATED_SETTING = final_dir / 'internal_settings.json'

with open(ISOLATED_SETTING, 'r') as f:
    cfg = json.load(f)

target_chains = str(cfg.get('target_chains', 'A')).split(',')[0].strip()
base_hotspots = [str(x).strip() for x in cfg.get('hotspots_input', '').split(',') if str(x).strip()]
SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

geometric_manifest_path = final_dir / "Geometric_Islands_Manifest.json"

# --- SMART RESTART LOGIC FOR SECTION 6 ---
force_restart = bool(root_cfg.get('force_restart_section_7', False))
if force_restart:
    print("\n🗑️ [COMMAND] Force Restart Section 6 is ENABLED.")
    if geometric_manifest_path.exists():
        geometric_manifest_path.unlink()
        print("🧹 Wiped old Geometric_Islands_Manifest.json to generate fresh geometric seeds.")
else:
    print(f"\n⏭️ [RESUME] Smart Resume active. Regenerating topological surface arrays in memory...")

# --- 2. Architecture: PDB Parsing & Dynamic Representative Atom Extraction (CB/CA) ---
print("🔬 Parsing structures: Extracting CB atoms (fallback to CA for GLY)...")
ca_atoms = {}
cb_atoms = {}
res_names = {}

if not SOURCE_PDB_PATH.exists():
    print(f"❌ Error: Source PDB not found at {SOURCE_PDB_PATH}")
    sys.exit(1)

with open(SOURCE_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[21] == target_chains:
            atom_name = line[12:16].strip()
            res_id = line[22:26].strip()
            res_name = line[17:20].strip()

            if atom_name == "CA":
                ca_atoms[res_id] = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                res_names[res_id] = res_name
            elif atom_name == "CB":
                cb_atoms[res_id] = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])

# Build Representative Atoms mapping
rep_atoms = {}
for rid in ca_atoms.keys():
    if res_names.get(rid) == 'GLY':
        rep_atoms[rid] = ca_atoms[rid]  # GLY has no CB, use CA
    else:
        rep_atoms[rid] = cb_atoms.get(rid, ca_atoms[rid]) # Use CB, fallback to CA if missing

all_res_ids = list(rep_atoms.keys())
all_coords = np.array([rep_atoms[rid] for rid in all_res_ids])
hotspot_coords = np.array([rep_atoms[rid] for rid in base_hotspots if rid in rep_atoms])

# --- 3. Dilation & Filtering (Half-Sphere Exposure / HSE Vector Check) ---
print("🧮 Applying Half-Sphere Exposure (HSE) vector analysis to precisely preserve surface helices...")

ca_coords_array = np.array([ca_atoms[rid] for rid in all_res_ids])
ca_kdtree = KDTree(ca_coords_array)
surface_residues = []

for i, rid in enumerate(all_res_ids):
    ca = ca_atoms[rid]

    # 1. Establish side-chain extension vector (CB - CA)
    if res_names.get(rid) != 'GLY' and rid in cb_atoms:
        v_out = cb_atoms[rid] - ca
    else:
        # GLY fallback: calculate approximate outward vector using center of mass of 8A neighbors
        neighbors_idx = ca_kdtree.query_ball_point(ca, r=8.0)
        if len(neighbors_idx) > 1:
            local_com = np.mean(ca_coords_array[neighbors_idx], axis=0)
            v_out = ca - local_com
        else:
            v_out = np.array([0.0, 0.0, 1.0])

    # Vector normalization
    norm = np.linalg.norm(v_out)
    if norm > 1e-4:
        v_out = v_out / norm
    else:
        v_out = np.array([0.0, 0.0, 1.0])

    # 2. Count neighbors within 10A in the "forward hemisphere" of the side chain
    neighbors_idx = ca_kdtree.query_ball_point(ca, r=10.0)
    forward_neighbors = 0

    for n_idx in neighbors_idx:
        if all_res_ids[n_idx] == rid:
            continue
        vec_to_neighbor = ca_coords_array[n_idx] - ca

        # Dot product > 0 indicates the neighbor is in the forward hemisphere
        if np.dot(vec_to_neighbor, v_out) > 0:
            forward_neighbors += 1

    # 3. Core decision logic:
    # Internal residues usually have 15-25 obstructing neighbors in the forward hemisphere.
    # Surface residues (including solvent-exposed helices) have an almost empty front, typically < 12.
    if forward_neighbors < 12:
        surface_residues.append(rid)

print(f"🌊 Identified {len(surface_residues)} surface-exposed residues (Strict HSE filtration passed).")

if len(hotspot_coords) == 0:
    print("❌ Error: No valid starting hotspots found matching the PDB numbering.")
    sys.exit(1)

# Build KDTree with representative atoms (CB primary) for distance dilation
rep_coords_array = np.array([rep_atoms[rid] for rid in surface_residues])
surface_kdtree = KDTree(rep_coords_array)

hotspot_kdtree = KDTree(hotspot_coords)
dilation_candidates = []

for rid in surface_residues:
    if rid in base_hotspots:
        continue
    coord = rep_atoms[rid]
    dists, _ = hotspot_kdtree.query(coord, k=1)

    # Maintain 5A - 8A geometric dilation ring
    if 5.0 <= dists <= 8.0:
        dilation_candidates.append(rid)

print(f"🪐 Extracted {len(dilation_candidates)} candidates in the 5Å - 8Å dilation ring.")

# --- 4. Island Generation (Greedy Minimum Set Cover, 5A radius) ---
uncovered_candidates = set(dilation_candidates)
satellite_islands = []
alphabet = string.ascii_uppercase

while uncovered_candidates:
    best_center = None
    best_coverage = set()

    for cand in uncovered_candidates:
        cand_coord = rep_atoms[cand]
        coverage = set()
        for other in uncovered_candidates:
            if np.linalg.norm(cand_coord - rep_atoms[other]) <= 5.0:
                coverage.add(other)
        if len(coverage) > len(best_coverage):
            best_coverage = coverage
            best_center = cand

    if best_center:
        idx = len(satellite_islands)
        suffix = alphabet[idx] if idx < 26 else f"X{idx}"
        isl_name = f"Isl_{suffix}"
        satellite_islands.append({
            "name": isl_name,
            "center": best_center,
            "residues": list(best_coverage)
        })
        uncovered_candidates -= best_coverage
    else:
        break

print(f"🏝️ Generated {len(satellite_islands)} dynamically distributed Satellite Islands.")

# --- 5. Export Geometric Manifest for GPU stages ---
geometric_manifest = {
    "base_hotspots": base_hotspots,
    "satellite_islands": satellite_islands
}
with open(geometric_manifest_path, 'w') as f:
    json.dump(geometric_manifest, f, indent=4)
print(f"💾 Exported Geometric Islands Manifest to {geometric_manifest_path.name}")

# --- 6. Visualizations: Mesh using py3Dmol ---
print("🎨 Rendering Topological Surface Analysis (Orange: Core Hotspots, White: Satellite Islands)")

# ==========================================
# 🛠️ PATCH 2: 防弹版智能翻译坐标彻底杜绝 ValueError
# ==========================================
def safe_extract_resi(resi_list):
    res = []
    for x in resi_list:
        if str(x).strip():  # 防御空字符串
            num_str = re.sub(r'\D', '', str(x))
            if num_str:     # 防御正则提取后为空 (如纯字母输入)
                res.append(int(num_str))
    return res

if SOURCE_PDB_PATH.exists():
    with open(SOURCE_PDB_PATH, 'r') as f:
        pdb_data = f.read()

    viewer = py3Dmol.view(width=800, height=600)
    viewer.addModel(pdb_data, 'pdb')

    # Type cast selectors to integers to bypass py3Dmol string failure using safe extractor
    hotspot_resi_int = safe_extract_resi(base_hotspots)

    all_satellite_res = []
    for isl in satellite_islands:
        all_satellite_res.extend(isl["residues"])

    sat_resi_int = safe_extract_resi(list(set(all_satellite_res)))

    # Target protein: Gray cartoon + Gray wireframe mesh (SES is valid for large contiguous structures)
    target_sel = {'chain': target_chains}
    viewer.setStyle(target_sel, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.7}})
    viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'wireframe': True, 'opacity': 0.5}, target_sel)

    # Core Hotspots: Orange cartoon + Solid Orange surface (Use VDW for fragmented sets)
    if hotspot_resi_int:
        hotspot_sel = {'chain': target_chains, 'resi': hotspot_resi_int}
        viewer.setStyle(hotspot_sel, {'cartoon': {'color': 'orange', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.VDW, {'color': 'orange', 'opacity': 1.0}, hotspot_sel)

    # Satellite Islands: White cartoon + Solid White surface (Use VDW for fragmented sets)
    if sat_resi_int:
        sat_sel = {'chain': target_chains, 'resi': sat_resi_int}
        viewer.setStyle(sat_sel, {'cartoon': {'color': 'gray', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.VDW, {'color': 'gray', 'opacity': 1.0}, sat_sel)

    viewer.zoomTo()

    try:
        from IPython.display import display
        display(viewer.show())
    except ImportError:
        viewer.show()

'''
==============================================================================
Objective: CPU-only Phase for Topological Seeding and Visualization via CB Atoms. Establishes satellite islands, meshes target, and exports geometric parameters.
           Includes Section 6 Smart Restart logic for cache invalidation.
           py3Dmol selectors strictly cast to integers and VDW algorithms applied to subsets to prevent silent render failures.
Upstream Dependencies: py3Dmol, scipy.spatial.KDTree, internal_settings.json
Runtime Environment: Google Colab (CPU compatible)
Generation Timestamp: Auto-generated via Script execution.
==============================================================================
'''

🌍 Initializing Cell 19: Topological Seeding (CB-Anchored) and Visualization...

🔑 【Cell 21 Visualizations 启动】 当前全局 Run ID: AR_LBD_wolf_evo_20260501_193250
📁 当前 Workspace 目录: screening_results/AR_LBD_wolf_evo/final_20260501_193250
💡 续传时请直接复制以下内容填入 Section 0 的 specific_resume_run_id 字段：
   specific_resume_run_id = "AR_LBD_wolf_evo_20260501_193250"


⏭️ [RESUME] Smart Resume active. Regenerating topological surface arrays in memory...
🔬 Parsing structures: Extracting CB atoms (fallback to CA for GLY)...
🧮 Applying Half-Sphere Exposure (HSE) vector analysis to precisely preserve surface helices...
🌊 Identified 219 surface-exposed residues (Strict HSE filtration passed).
🪐 Extracted 21 candidates in the 5Å - 8Å dilation ring.
🏝️ Generated 18 dynamically distributed Satellite Islands.
💾 Exported Geometric Islands Manifest to Geometric_Islands_Manifest.json
🎨 Rendering Topological Surface Analysis (Orange: Core Hotspots, White: Satellite Islands)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

None

'\n==============================================================================\nObjective: CPU-only Phase for Topological Seeding and Visualization via CB Atoms. Establishes satellite islands, meshes target, and exports geometric parameters.\n           Includes Section 6 Smart Restart logic for cache invalidation.\n           py3Dmol selectors strictly cast to integers and VDW algorithms applied to subsets to prevent silent render failures.\nUpstream Dependencies: py3Dmol, scipy.spatial.KDTree, internal_settings.json\nRuntime Environment: Google Colab (CPU compatible)\nGeneration Timestamp: Auto-generated via Script execution.\n==============================================================================\n'

In [ ]:
# @title Cell_19b_Generation_Engine.py
# 需求：底层生成引擎。新增激进的 OOM 防御环境变量，纯粹负责批量结构的物理生成与基本评估。

import os, sys, json, subprocess, time, re, yaml, gc
from pathlib import Path
import pandas as pd
import torch

def resolve_workspace(base_dir, task_name, specific_id=""):
    last_run_file = base_dir / 'screening_results' / 'last_runs.txt'
    run_id, target_dir_name = None, None
    if last_run_file.exists():
        with open(last_run_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            pattern = rf"Section2_TargetPreprocess(?:_TimeMachine)? \| Run_ID: ({re.escape(task_name)}_\d+_\d+) \| Dir: (final_\d+_\d+)"
            for line in reversed(lines):
                match = re.search(pattern, line)
                if match:
                    found_id, found_dir = match.group(1), match.group(2)
                    if specific_id:
                        if found_id == specific_id.strip():
                            run_id, target_dir_name = found_id, found_dir
                            break
                    else:
                        run_id, target_dir_name = found_id, found_dir
                        break
    if not run_id:
        print(f"❌ Error: Could not resolve Run_ID ({specific_id or 'latest'}) from last_runs.txt")
        sys.exit(1)

    final_dir = base_dir / 'screening_results' / task_name / target_dir_name
    return run_id, final_dir

def fs(val, is_rmsd=False, is_plddt=False):
    if pd.isna(val) or val == -999.0: return "N/A"
    try:
        v = float(val)
        if is_rmsd: return f"{v:<8.2f}Å"
        if is_plddt: return f"{v:<8.2f}"
        return f"{v:<8.4f}"
    except: return "N/A"

def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0: return f"{h}h {m}m {s}s"
    return f"{m}m {s}s"

def clean_vram():
    """激进清理显存"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def run_complexa_batch(
    base_dir, task_name, run_name, target_chains, hotspots_list,
    num_designs, batch_size, eval_batch_size, dynamic_seed, env
):
    # 强制清理主进程显存，为子进程腾出空间
    clean_vram()

    yaml_path = base_dir / 'configs/targets/targets_dict.yaml'
    if yaml_path.exists() and hotspots_list:
        with open(yaml_path, 'r') as f: yaml_data = yaml.safe_load(f)
        yaml_data['target_dict_cfg'][task_name]['hotspot_residues'] = [f"{target_chains}{r}" for r in hotspots_list]
        with open(yaml_path, 'w') as f: yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

    # 注入抵抗碎片化的环境变量
    safe_env = env.copy()
    safe_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

    cmd_str = (
        f"source env.sh && complexa design configs/search_binder_local_pipeline.yaml "
        f"++run_name={run_name} ++generation.task_name={task_name} "
        f"++generation.num_designs={num_designs} "
        f"++generation.dataloader.batch_size={batch_size} ++generation.search.max_batch_size={batch_size} "
        f"++generation.temperature=1.5 ++generation.noise_scale=1.2 "
        f"++evaluation.dataloader.batch_size={eval_batch_size} ++generation.seed={dynamic_seed} "
        f"++run_filter=True ++run_evaluate=True ++evaluate.num_recycles=1 ++evaluate.pad_to_max_length=True "
        f"++evaluate.use_msa=False ++evaluate.msa_mode=single_sequence ++run_analyze=True "
        f"++evaluate.save_outputs=True"
    )

    batch_start = time.time()
    process = subprocess.Popen(
        cmd_str, env=safe_env, shell=True, executable='/bin/bash', cwd=str(base_dir),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    full_logs = []
    for line in process.stdout:
        full_logs.append(line)
        if any(k in line for k in ["Error", "Exception", "Traceback"]):
            print(f"      {line.strip()}")

    process.wait()
    batch_duration = time.time() - batch_start

    if process.returncode != 0:
        full_output = "".join(full_logs)
        print(f"\n❌ [CRITICAL ERROR] 管线崩溃 (Exit Code {process.returncode}).")
        log_paths = re.findall(r'(?:📝 Log:|Check log for details:)\s*([^\n]+\.log)', full_output)

        if not log_paths:
            print("="*80)
            print(full_output[-2000:])
            print("="*80)
        else:
            for log_rel_path in set(log_paths):
                log_abs_path = base_dir / log_rel_path.strip()
                if log_abs_path.exists():
                    print(f"\n📄 [Deep Log Extracted]: {log_abs_path.name}")
                    print("="*80)
                    try:
                        with open(log_abs_path, 'r') as lf:
                            deep_lines = lf.readlines()
                            print("".join(deep_lines[-100:]))
                    except: pass
                    print("="*80)
        clean_vram()
        return False, [], batch_duration, None

    inference_dir = base_dir / 'inference' / f'search_binder_local_pipeline_{task_name}_{run_name}'
    valid_csvs = [f for f in inference_dir.rglob('*.csv') if 'timing' not in f.name.lower()] if inference_dir.exists() else []
    success = False
    parsed_data = []

    if valid_csvs:
        target_csv = max(valid_csvs, key=lambda x: x.stat().st_size)
        try:
            df = pd.read_csv(target_csv)
            score_col = next((col for col in ['af2folding_max_ipsae', 'max_ipSAE'] if col in df.columns), None)

            if score_col:
                df[score_col] = pd.to_numeric(df[score_col], errors='coerce')
                df = df.sort_values(by=score_col, ascending=False).reset_index(drop=True)

                for idx, row in df.iterrows():
                    design_id = row.get('design_id', f'Design_{idx}')
                    rel_path = row.get('pdb_path', '')
                    abs_pdb_path = str(inference_dir.resolve() / rel_path) if rel_path else "N/A"
                    if abs_pdb_path == "N/A" or not Path(abs_pdb_path).exists():
                        potential_pdbs = list(inference_dir.rglob(f"{design_id}*.pdb"))
                        if potential_pdbs: abs_pdb_path = str(potential_pdbs[0].resolve())

                    parsed_data.append({
                        "Design_ID": design_id,
                        "ipSAE": float(row.get(score_col, -999.0)),
                        "pTM": float(row.get('af2folding_ptm_log', -999.0)),
                        "iPTM": float(row.get('af2folding_i_ptm_log', -999.0)),
                        "pLDDT": float(row.get('af2folding_plddt', -999.0)),
                        "RMSD": float(row.get('af2folding_rmsd', -999.0)),
                        "Reward": float(row.get('total_reward', row.get('reward', -999.0))),
                        "Abs_Path": abs_pdb_path
                    })
                success = True
        except Exception as e:
            print(f"      ⚠️ CSV Parsing Error: {e}")

    clean_vram()
    return success, parsed_data, batch_duration, inference_dir

print("✅ Engine loaded: Advanced memory handling deployed.")

✅ Engine loaded: Advanced memory handling deployed.


In [ ]:
# @title Cell_19c_PAE_Evaluation_Engine.py
# 需求：将 AF2 的 PAE 计算彻底剥离为主进程之外的独立脚本。利用操作系统的进程级物理回收解决 CUDA 显存泄漏。

import os
from pathlib import Path
import stat

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')

pae_script_content = """import os
import sys
import json
import numpy as np
from pathlib import Path

# 解析命令行参数
source_pdb_path = sys.argv[1]
target_chains = sys.argv[2]
binder_seq = sys.argv[3]
output_json_path = sys.argv[4]
base_dir = Path(sys.argv[5])

# 挂载参数目录
colab_params_dir = Path("/content/params")
af2_params_found = False
common_weights_dirs = [base_dir / 'weights', base_dir / 'params', base_dir / 'assets' / 'weights']

for d in common_weights_dirs:
    if (d / "params_model_1_multimer_v3.npz").exists():
        os.system(f"ln -sf {d} /content/params")
        af2_params_found = True
        break
    elif (d / "params" / "params_model_1_multimer_v3.npz").exists():
        os.system(f"ln -sf {d}/params /content/params")
        af2_params_found = True
        break

if not af2_params_found and not (colab_params_dir / "params_model_1_multimer_v3.npz").exists():
    os.system("mkdir -p /content/params")
    os.system("curl -fsSL https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar | tar x -C /content/params")

# 引入并执行 AF2 (此环境完全独立于 Jupyter)
try:
    from colabdesign import mk_afdesign_model
except ImportError:
    os.system("pip install git+https://github.com/sokrypton/ColabDesign.git -q")
    from colabdesign import mk_afdesign_model

af_model = mk_afdesign_model(protocol="binder", use_multimer=True, data_dir="/content")
af_model.prep_inputs(pdb_filename=source_pdb_path, chain=target_chains, binder_len=len(binder_seq))
len_target = af_model._lengths[0]

af_model.restart(seq=binder_seq)
af_model.predict(num_recycles=1)
pae_matrix = af_model.aux["pae"]

cross_pae_target_to_binder = pae_matrix[:len_target, len_target:]
mean_pae_per_residue = np.mean(cross_pae_target_to_binder, axis=1)

result_payload = {
    "raw_pae": pae_matrix.tolist(),
    "mean_pae_array": mean_pae_per_residue.tolist()
}

with open(output_json_path, 'w') as f:
    json.dump(result_payload, f)
"""

script_path = BASE_DIR / 'run_pae_hook.py'
with open(script_path, 'w') as f:
    f.write(pae_script_content)

os.chmod(script_path, stat.S_IRWXU)
print(f"✅ Independent Subprocess PAE Engine compiled to: {script_path}")

✅ Independent Subprocess PAE Engine compiled to: /content/drive/MyDrive/Proteina-Complexa/run_pae_hook.py


# Section 7: Topological Seeding and Deep Greedy Convergence

In [ ]:
# @title Cell_20b_Core_Hotspot_Screening.py
# 需求：独立执行 Core Hotspot 筛选。降低 Batch Size 至 4 以彻底根治 Forward Pass OOM。

# ==========================================
# 🟢 User Configuration
# ==========================================
force_restart_core_islands = False
specific_resume_run_id_core = ""

import os, sys, json, datetime, random, math, subprocess, re, gc
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display

print("\n🔥 Initializing GPU Generation and Real-time Tracking (Core Residue Master Mode)...")

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
ROOT_SETTING = BASE_DIR / 'internal_settings.json'
with open(ROOT_SETTING, 'r') as f: root_cfg = json.load(f)
task_name = root_cfg['task_name']

run_id, final_dir = resolve_workspace(BASE_DIR, task_name, specific_resume_run_id_core)

# ----------------------------------------------------
# 📡 内部双向流同步探针 (Dual-Channel Logger)
# ----------------------------------------------------
LIVE_LOG_FILE = final_dir / f"Live_Sync_{run_id}.log"

class DualLogger:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        # buffering=1 启用行级强缓冲，确保 minimum invasive 的实时性
        self.log = open(filepath, "a", buffering=1, encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# 全局接管标准输出与标准错误
sys.stdout = DualLogger(LIVE_LOG_FILE)
sys.stderr = sys.stdout
print(f"📡 笔记本模式双向同步日志流已接管: {LIVE_LOG_FILE}")
# ----------------------------------------------------


ISOLATED_SETTING = final_dir / 'internal_settings.json'
GEOMETRIC_MANIFEST = final_dir / "Geometric_Islands_Manifest.json"
with open(ISOLATED_SETTING, 'r') as f: cfg = json.load(f)
with open(GEOMETRIC_MANIFEST, 'r') as f: geo_data = json.load(f)

name = "Isl_0"
target_chains = str(cfg.get('target_chains', 'A')).split(',')[0].strip()
patch_residues = geo_data["base_hotspots"]

evaluation_batch_size = int(root_cfg.get('evaluation_batch_size', 1))
# ❗️ 核心修改：为了对抗 Forward Pass OOM，把并行生成数量从 8 强制降到 4 ❗️

# 为了弥补每个 Batch 产量减半，把 Batch 数量翻倍以维持总生成量 (12 -> 24)
generation_batches_per_core_hot_spot = int(root_cfg.get('generation_batches_per_core_hot_spot', 8))
actual_batches = generation_batches_per_core_hot_spot

master_csv_path = final_dir / "Isl_0_DeepDive_MasterScores.csv"
path_csv_path = final_dir / "Isl_0_DeepDive_PathMapping.csv"
res_master_csv = final_dir / "Core_Residue_PAE_Master.csv"

core_res_pae_tracker = {}
if res_master_csv.exists() and not force_restart_core_islands:
    try:
        rdf = pd.read_csv(res_master_csv)
        core_res_pae_tracker = {str(r['Residue_ID']): float(r['Best_PAE']) for _, r in rdf.iterrows()}
    except: pass

completed_batches = set()
if master_csv_path.exists():
    try:
        existing_df = pd.read_csv(master_csv_path)
        if not existing_df.empty and 'Batch' in existing_df.columns:
            completed_batches = set(existing_df['Batch'].astype(int).unique())
    except: pass

SESSION_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
SESSION_NAME = f"DeepDive_{task_name}_{SESSION_TIMESTAMP}"

# ❗️ 核心修改：注入最严厉的显存限制环境变量 ❗️
env = os.environ.copy()
env.update({
    'HYDRA_FULL_ERROR': '1',
    'PYTHONUNBUFFERED': '1',
    'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
    'XLA_PYTHON_CLIENT_ALLOCATOR': 'platform',
    'TF_FORCE_GPU_ALLOW_GROWTH': 'true'
})

SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"
global_ordered_resis = []
with open(SOURCE_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
            global_ordered_resis.append(line[22:26].strip())
natural_to_af2 = {nat: i for i, nat in enumerate(global_ordered_resis)}
af2_to_natural = {i: nat for i, nat in enumerate(global_ordered_resis)}

d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K', 'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N', 'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W', 'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}
hotspot_weights = {res: 1.0 for res in patch_residues}

def adaptive_weighted_sample(population, weights_dict, k_items):
    pop = list(population)
    sampled = []
    for _ in range(k_items):
        w = [weights_dict[x] for x in pop]
        if sum(w) <= 0: break
        chosen = random.choices(pop, weights=w, k=1)[0]
        sampled.append(chosen)
        pop.remove(chosen)
    return sampled

clean_vram()

display(HTML(f"""<hr><h3 style='color: #8B008B;'>🚀 Initiating Low-VRAM Deep Dive: {name}</h3>
<p><b>Target:</b> {actual_batches} Batches (4 designs/batch)<br><b>Strategy:</b> OS-Level Subprocess PAE Eval (Zero OOM).</p>"""))

for batch_idx in range(1, actual_batches + 1):
    if batch_idx in completed_batches: continue

    sampled_patch = adaptive_weighted_sample(patch_residues, hotspot_weights, max(1, len(patch_residues) // 2))
    sampled_patch.sort(key=lambda x: int(re.sub(r'\D', '', x)))

    run_name = f"{SESSION_NAME}_{name}_Batch{batch_idx}"
    dynamic_seed = random.randint(10000, 99999)

    print(f"\n   🎲 [Batch {batch_idx}/{actual_batches}] Sampled -> {','.join(sampled_patch)}")

    success, parsed_data, duration, inf_dir = run_complexa_batch(
        BASE_DIR, task_name, run_name, target_chains, sampled_patch,
        base_batch_size, base_batch_size, evaluation_batch_size, dynamic_seed, env
    )

    if success and parsed_data:
        # 寻找既有高 ipSAE，又满足基础折叠度 (pLDDT >= 0.40) 的构象
        valid_designs = [d for d in parsed_data if float(d['pLDDT']) >= 0.40]

        if not valid_designs:
            print(f"   ⚠️ Batch {batch_idx} generated only ghost conformations (pLDDT < 0.40). Discarding PAE extraction to prevent false hotspots.")
            completed_batches.add(batch_idx)
            continue

        best_design = valid_designs[0] # 取物理合规的最高分构象
        abs_pdb_path = best_design['Abs_Path']
        pae_table_html = "N/A"

        if Path(abs_pdb_path).exists():
            with open(abs_pdb_path, 'r') as f: pdb_data = f.read()
            chains_seq = {}
            for line in pdb_data.split('\n'):
                if line.startswith("ATOM") and line[12:16].strip() == "CA":
                    res_name, chain_id = line[17:20].strip(), line[21]
                    if chain_id not in chains_seq: chains_seq[chain_id] = ""
                    chains_seq[chain_id] += d3to1.get(res_name, 'X')
            binder_seq = chains_seq.get(sorted(chains_seq.keys(), key=lambda c: len(chains_seq[c]))[0], "N/A")

            if binder_seq != "N/A":
                pae_out_path = Path(abs_pdb_path).parent / f"{Path(abs_pdb_path).stem}_pae.json"
                print(f"   🧬 [Subprocess Hook] Delegating PAE evaluation to isolated process...")

                try:
                    script_path = BASE_DIR / 'run_pae_hook.py'
                    sub_p = subprocess.run(
                        ["python3", str(script_path), str(SOURCE_PDB_PATH), target_chains, binder_seq, str(pae_out_path), str(BASE_DIR)],
                        capture_output=True, text=True, check=True
                    )

                    if pae_out_path.exists():
                        with open(pae_out_path, 'r') as f:
                            pae_result_payload = json.load(f)
                            cross_pae = np.array(pae_result_payload['mean_pae_array'])

                        ranked_indices = np.argsort(cross_pae)
                        table_html = "<table style='width:100%; text-align:center; border-collapse: collapse; font-size: 12px; margin-top: 5px;'>"
                        table_html += "<tr><th style='border: 1px solid #ddd; padding: 4px; background-color: #e9ecef;'>Rank</th>"
                        for i in range(10): table_html += f"<td style='border: 1px solid #ddd; padding: 4px; font-weight: bold; background-color: #f8f9fa;'>#{i+1}</td>"
                        table_html += "</tr><tr><th style='border: 1px solid #ddd; padding: 4px; background-color: #e9ecef;'>Residue</th>"
                        for i in range(10): table_html += f"<td style='border: 1px solid #ddd; padding: 4px;'>{af2_to_natural.get(ranked_indices[i], f'Idx_{ranked_indices[i]}')}</td>"
                        table_html += "</tr><tr><th style='border: 1px solid #ddd; padding: 4px; background-color: #e9ecef;'>PAE</th>"
                        for i in range(10):
                            pae_val = cross_pae[ranked_indices[i]]
                            color = "green" if pae_val < 10 else ("orange" if pae_val < 20 else "red")
                            table_html += f"<td style='border: 1px solid #ddd; padding: 4px; color:{color}; font-weight: bold;'>{pae_val:.2f}</td>"
                        table_html += "</tr></table>"
                        pae_table_html = table_html

                        for t_idx, p_val in enumerate(cross_pae):
                            nat_r = af2_to_natural.get(t_idx)
                            if nat_r:
                                if nat_r not in core_res_pae_tracker or p_val < core_res_pae_tracker[nat_r]:
                                    core_res_pae_tracker[nat_r] = float(p_val)

                        for r in sampled_patch:
                            if r in natural_to_af2:
                                res_pae = cross_pae[natural_to_af2[r]]
                                if res_pae < 10.0: hotspot_weights[r] = min(10.0, hotspot_weights[r] * 1.5)
                                elif res_pae > 15.0: hotspot_weights[r] = max(0.1, hotspot_weights[r] * (0.95 if hotspot_weights[r] >= 1.5 else 0.85))
                except subprocess.CalledProcessError as e:
                    print(f"      ⚠️ Subprocess PAE Engine crashed. Stderr: {e.stderr}")

        batch_df = pd.DataFrame(parsed_data)
        batch_df['Batch'], batch_df['Sampled_Hotspots'] = batch_idx, ','.join(sampled_patch)
        batch_df.to_csv(master_csv_path, mode='a', index=False, header=not master_csv_path.exists())
        pd.DataFrame([{'Residue_ID': k, 'Best_PAE': v} for k, v in core_res_pae_tracker.items()]).to_csv(res_master_csv, index=False)

        display_text = "\n".join([f"{r['Design_ID']:<16} | {fs(r['ipSAE']):<8}" for r in parsed_data])
        display(HTML(f"""<div style='background: #f8f9fa; padding: 10px; border-left: 4px solid #0056b3;'><pre>{display_text}</pre><div>{pae_table_html}</div></div>"""))

        print(f"   💾 Batch {batch_idx} saved. VRAM reclaimed safely.")
        completed_batches.add(batch_idx)

print("✅ Low-VRAM Core Screening Pipeline Finished.")


🔥 Initializing GPU Generation and Real-time Tracking (Core Residue Master Mode)...


✅ Low-VRAM Core Screening Pipeline Finished.


In [ ]:
#@title Cell_21_Physical_Generation_Visualizations.py
# Requirements:
# 1. Change the output title to "Best PAE Amino Acids".
# 2. Utilize the abstracted Cell 19c engine to extract static PAE interactions and return a 3x10 HTML table.
# 3. Track the best PAE score for each residue across all top designs and save it as Global_Best_Cross_PAE_Summary.csv.
# 4. Keep 3D rendering logic unchanged.

# ==========================================
# 🟢 User Configuration
# ==========================================
specific_run_id = ""

import os, sys, json, re
from pathlib import Path
import numpy as np
import py3Dmol
import pandas as pd
from IPython.display import HTML, display, clear_output

clear_output(wait=True)
print("🎨 Initiating Deep Dive Review Mode: Rendering Global Top Candidates with Static PAE Evaluation...")

# ==========================================
# 🛠️ 1. Directory & Configuration Setup
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
ROOT_SETTING = BASE_DIR / 'internal_settings.json'

if not ROOT_SETTING.exists():
    print("❌ Error: Root configuration missing.")
    sys.exit(1)

with open(ROOT_SETTING, 'r') as f: root_cfg = json.load(f)
task_name = root_cfg['task_name']
last_run_file = BASE_DIR / 'screening_results' / 'last_runs.txt'

run_id, target_dir_name = None, None

if last_run_file.exists():
    with open(last_run_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        pattern = rf"Section2_TargetPreprocess(?:_TimeMachine)? \| Run_ID: ({re.escape(task_name)}_\d+_\d+) \| Dir: (final_\d+_\d+)"
        for line in reversed(lines):
            match = re.search(pattern, line)
            if match:
                found_id, found_dir = match.group(1), match.group(2)
                if specific_run_id:
                    if found_id == specific_run_id.strip():
                        run_id, target_dir_name = found_id, found_dir
                        break
                else:
                    run_id, target_dir_name = found_id, found_dir
                    break

final_dir = BASE_DIR / 'screening_results' / task_name / target_dir_name if run_id else None

if not final_dir:
    print(f"❌ Error: 无法解析 Run_ID。请检查 last_runs.txt。")
    sys.exit(1)

ISOLATED_SETTING = final_dir / 'internal_settings.json'
with open(ISOLATED_SETTING, 'r') as f: cfg = json.load(f)
target_chains = str(cfg.get('target_chains', 'A')).split(',')[0].strip()

# ==========================================
# 🛠️ 2. Mapping Engine Setup
# ==========================================
SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"
global_ordered_resis = []

if SOURCE_PDB_PATH.exists():
    with open(SOURCE_PDB_PATH, 'r') as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
                global_ordered_resis.append(line[22:26].strip())

natural_to_af2 = {nat: i+1 for i, nat in enumerate(global_ordered_resis)}
af2_to_natural = {i: nat for i, nat in enumerate(global_ordered_resis)}

def translate_to_model_idx(natural_resis, mapping_dict):
    if not natural_resis: return []
    if isinstance(natural_resis, str):
        raw_list = [r.strip() for r in natural_resis.replace(',', ' ').split() if r.strip()]
    else:
        raw_list = [str(r).strip() for r in natural_resis]
    mapped_indices = []
    for r in raw_list:
        if r in mapping_dict:
            mapped_indices.append(mapping_dict[r])
    return mapped_indices

# ==========================================
# 🛠️ 3. Load Data & Abstracted Model Init
# ==========================================
master_csv_path = final_dir / "Isl_0_DeepDive_MasterScores.csv"
geo_manifest_path = final_dir / "Geometric_Islands_Manifest.json"

if not master_csv_path.exists():
    print(f"⏳ Waiting for Master Scores table...")
    sys.exit(1)

df = pd.read_csv(master_csv_path)
global_base_hotspots = []
if geo_manifest_path.exists():
    with open(geo_manifest_path, 'r') as f:
        global_base_hotspots = json.load(f).get("base_hotspots", [])

df_sorted = df.sort_values(by='ipSAE', ascending=False).reset_index(drop=True)
render_df = df_sorted[df_sorted['ipSAE'] >= 0.5].head(10)
if render_df.empty: render_df = df_sorted.head(10)

# 🚀 调用 Cell 19c 中的全局引擎进行初始化
af_model = initialize_af2_model(BASE_DIR)

global_best_pae_tracker = {}
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K', 'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N', 'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W', 'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

# ==========================================
# 🎨 4. Render Loop with Abstracted Engine
# ==========================================
for rank_idx, (idx, row) in enumerate(render_df.iterrows(), start=1):
    batch_idx = row.get('Batch', 'N/A')
    design_id, ipsae_val, abs_pdb_path = row['Design_ID'], row['ipSAE'], row['Abs_Path']
    sampled_hotspots_raw = str(row['Sampled_Hotspots'])

    if abs_pdb_path == "N/A" or not Path(abs_pdb_path).exists(): continue
    with open(abs_pdb_path, 'r') as f: pdb_data = f.read()

    chains_seq = {}
    first_resi_binder_int = None

    for line in pdb_data.split('\n'):
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            res_name, chain_id = line[17:20].strip(), line[21]
            if chain_id not in chains_seq: chains_seq[chain_id] = ""
            chains_seq[chain_id] += d3to1.get(res_name, 'X')

    sorted_chains = sorted(chains_seq.keys(), key=lambda c: len(chains_seq[c]))
    binder_chain, target_chain_ids = sorted_chains[0], sorted_chains[1:]
    binder_seq = chains_seq.get(binder_chain, "N/A")

    if binder_seq == "N/A": continue

    for line in pdb_data.split('\n'):
        if line.startswith("ATOM") and line[21] == binder_chain:
            first_resi_binder_int = int(line[22:26].strip())
            break

    core_mapped_idxs = translate_to_model_idx(global_base_hotspots, natural_to_af2)
    sampled_mapped_idxs = translate_to_model_idx([r.strip() for r in sampled_hotspots_raw.split(',')], natural_to_af2)

    # --- 🚀 调用 Cell 19c 抽象引擎进行 PAE 计算与表格生成 ---
    binder_length = len(binder_seq)
    print(f"🧬 Extracting PAE for {design_id} (Binder Length: {binder_length})...")

    pae_out_path = Path(abs_pdb_path).parent / f"{Path(abs_pdb_path).stem}_pae.json"
    pae_result = evaluate_static_pae(
        af_model=af_model,
        source_pdb_path=str(SOURCE_PDB_PATH),
        target_chains=target_chains,
        binder_seq=binder_seq,
        output_json_path=str(pae_out_path),
        af2_to_natural_map=af2_to_natural
    )

    top_10_display = pae_result['html_table']
    mean_pae_per_residue = pae_result['mean_pae_per_residue']

    # 更新全局最优 PAE 追踪器
    for t_idx, p_val in enumerate(mean_pae_per_residue):
        nat_r = af2_to_natural.get(t_idx)
        if nat_r:
            if nat_r not in global_best_pae_tracker or p_val < global_best_pae_tracker[nat_r]['Best_PAE']:
                global_best_pae_tracker[nat_r] = {
                    'Residue_ID': nat_r,
                    'Best_PAE': float(p_val),
                    'Source_Design': design_id
                }

    html_content = f"""
    <hr>
    <h3 style='color: #0056b3;'>🏆 Global Rank {rank_idx} | Batch {batch_idx} | {design_id}</h3>
    <div style='background: #f8f9fa; padding: 12px; border-radius: 6px; margin-bottom: 10px; border-left: 4px solid #28a745; box-shadow: 0 1px 3px rgba(0,0,0,0.1);'>
        <p style='margin: 5px 0;'><b>🌟 ipSAE Score:</b> <span style='color:red; font-size:18px; font-weight:bold;'>{ipsae_val:.4f}</span> | <b>Targeted Anchors:</b> {sampled_hotspots_raw}</p>
        <p style='margin: 5px 0;'><b>🧬 Peptide Sequence:</b> <code style='background:#fff; padding:3px 8px; border: 1px solid #ddd; border-radius:4px; font-size:14px; color:#d63384;'>{binder_seq}</code></p>
        <p style='margin: 5px 0; font-size: 13px;'><b>💾 PAE Saved To:</b> {pae_out_path.name}</p>
        <div style='margin-top: 8px; padding-top: 8px; border-top: 1px dashed #ccc;'>
            <p style='margin: 0 0 5px 0; font-size: 13px; font-weight: bold; color: #555;'>🔬 Best PAE Amino Acids:</p>
            {top_10_display}
        </div>
    </div>
    <div style='font-size: 13px; color: #555; margin-bottom: 5px;'>
        <b>Color Map:</b>
        <span style='color:orange; font-weight:bold;'>All Core Hotspots (Orange)</span> |
        <span style='color:red; font-weight:bold;'>Active Sampled Anchors (Red)</span> |
        <span style='color:lime; font-weight:bold;'>Peptide (Green)</span> |
        <span style='color:blue; font-weight:bold;'>Peptide N-Term (Blue)</span>
    </div>
    """
    display(HTML(html_content))

    viewer = py3Dmol.view(width=800, height=450)
    viewer.addModel(pdb_data, 'pdb')

    target_sel = {'chain': target_chain_ids}
    viewer.setStyle(target_sel, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.7}})
    viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'wireframe': True}, target_sel)

    if core_mapped_idxs:
        c_sel = {'chain': target_chain_ids, 'resi': core_mapped_idxs}
        viewer.setStyle(c_sel, {'cartoon': {'color': 'orange', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.VDW, {'color': 'orange', 'opacity': 0.8}, c_sel)

    if sampled_mapped_idxs:
        n_sel = {'chain': target_chain_ids, 'resi': sampled_mapped_idxs}
        viewer.setStyle(n_sel, {'cartoon': {'color': 'red', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.VDW, {'color': 'red', 'opacity': 1.0}, n_sel)

    b_sel = {'chain': binder_chain}
    viewer.setStyle(b_sel, {'cartoon': {'color': 'lime', 'opacity': 1.0}})
    viewer.addSurface(py3Dmol.SES, {'color': 'lime', 'opacity': 1.0}, b_sel)

    if first_resi_binder_int is not None:
        nterm_sel = {'chain': binder_chain, 'resi': [first_resi_binder_int]}
        viewer.setStyle(nterm_sel, {'cartoon': {'color': 'blue', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.VDW, {'color': 'blue', 'opacity': 1.0}, nterm_sel)

    viewer.zoomTo()
    try:
        display(viewer.show())
    except Exception:
        viewer.show()

# ==========================================
# 📊 5. Export Global Best PAE Summary
# ==========================================
if global_best_pae_tracker:
    pae_summary_path = final_dir / "Global_Best_Cross_PAE_Summary.csv"
    pae_summary_list = list(global_best_pae_tracker.values())
    df_pae_summary = pd.DataFrame(pae_summary_list)
    df_pae_summary['Residue_Num'] = df_pae_summary['Residue_ID'].apply(lambda x: int(re.sub(r'\D', '', str(x))))
    df_pae_summary = df_pae_summary.sort_values('Residue_Num').drop(columns=['Residue_Num'])
    df_pae_summary.to_csv(pae_summary_path, index=False)
    print(f"\n✅ 全局最优 PAE 总表已保存至: {pae_summary_path.absolute()}")

# # # Purpose: Render top candidates using abstracted PAE engine and output global PAE summary table.
# # # Upstream Code: Cell 19c (PAE Evaluation Engine).
# # # Runtime Environment: Google Colab.
# # # Generation Time: 2026-04-29 14:05 EDT.
# # # Changed Lines:
# # # 行 112: 通过全局函数 initialize_af2_model 替代硬编码的环境安装与加载。
# # # 行 148-156: 调用 evaluate_static_pae 获取 PAE 并返回渲染好的 HTML 表格。
# # # 行 160-167: 补全之前版本缺失的 global_best_pae_tracker 更新逻辑。
# # # 行 229-236: 将记录下来的全局最优氨基酸 PAE 持久化导出为 csv，以支持下游的演化算法。

In [ ]:
# @title Cell_22_Island_Constrained_Single_Point_Scanning.py
# 需求：提取满足 pLDDT ≥ 0.40 的单一物理标杆构象，解析其独立 PAE 矩阵以提取连贯的 4 个物理基石 (Coherent Core)。以此为基准，在 20Å 范围内执行外延扫描，并实施 pLDDT 联合过滤以防止边缘假阳性。

# ==========================================
# 🟢 User Configuration
# ==========================================
force_restart_edge_islands = False
specific_resume_run_id_edge = ""
REACHABLE_DISTANCE = 20.0
PAE_DISCARD_THRESHOLD = 12.0

import os, sys, json, datetime, re, subprocess, gc
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import KDTree
import torch
from IPython.display import HTML, display

print("\n🔥 Initializing Island-Constrained Scanning (Coherent Core + Validated Single Point Mode)...")

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
ROOT_SETTING = BASE_DIR / 'internal_settings.json'
with open(ROOT_SETTING, 'r') as f: root_cfg = json.load(f)
task_name = root_cfg['task_name']
force_restart_section_7 = bool(root_cfg.get('force_restart_section_7', False))

run_id, final_dir = resolve_workspace(BASE_DIR, task_name, specific_resume_run_id_edge)

# ----------------------------------------------------
# 📡 内部双向流同步探针 (Dual-Channel Logger)
# ----------------------------------------------------
LIVE_LOG_FILE = final_dir / f"Live_Sync_{run_id}.log"

class DualLogger:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        # buffering=1 启用行级强缓冲，确保 minimum invasive 的实时性
        self.log = open(filepath, "a", buffering=1, encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# 全局接管标准输出与标准错误
sys.stdout = DualLogger(LIVE_LOG_FILE)
sys.stderr = sys.stdout
print(f"📡 笔记本模式双向同步日志流已接管: {LIVE_LOG_FILE}")
# ----------------------------------------------------


ISOLATED_SETTING = final_dir / 'internal_settings.json'
GEOMETRIC_MANIFEST = final_dir / "Geometric_Islands_Manifest.json"
with open(ISOLATED_SETTING, 'r') as f: cfg = json.load(f)
with open(GEOMETRIC_MANIFEST, 'r') as f: geo_data = json.load(f)

SESSION_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
SESSION_NAME = f"IslandScan_{task_name}_{SESSION_TIMESTAMP}"

target_chains = str(cfg.get('target_chains', 'A')).split(',')[0].strip()
satellite_islands = geo_data.get("satellite_islands", [])
evaluation_batch_size = int(root_cfg.get('evaluation_batch_size', 1))
base_batch_size = 4

SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"
master_csv_path = final_dir / "EdgeIslands_DeepDive_MasterScores.csv"
res_master_csv = final_dir / "Edge_Residue_PAE_Master.csv"

env = os.environ.copy()
env.update({
    'HYDRA_FULL_ERROR': '1',
    'PYTHONUNBUFFERED': '1',
    'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
    'XLA_PYTHON_CLIENT_ALLOCATOR': 'platform',
    'TF_FORCE_GPU_ALLOW_GROWTH': 'true'
})

should_reset = force_restart_section_7 or force_restart_edge_islands
if should_reset:
    print("⚠️ [FORCE RESTART] Overwriting previous Edge Island data...")
    if master_csv_path.exists(): master_csv_path.unlink()
    if res_master_csv.exists(): res_master_csv.unlink()

completed_tasks = set()
if master_csv_path.exists():
    try:
        existing_df = pd.read_csv(master_csv_path)
        if not existing_df.empty and 'Island' in existing_df.columns and 'Sampled_Patch' in existing_df.columns:
            for _, row in existing_df.iterrows():
                target_res_list = str(row['Sampled_Patch']).split(',')
                if target_res_list:
                    completed_tasks.add(f"{row['Island']}_R{target_res_list[-1]}")
        print(f"✅ 已识别 {len(completed_tasks)} 个已完成遍历位点，准备断点续传。")
    except: pass

# ==========================================
# 📐 Load Coordinates & Sequence Mapping
# ==========================================
ca_coords, global_ordered_resis = {}, []
with open(SOURCE_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
            resi = line[22:26].strip()
            ca_coords[resi] = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            global_ordered_resis.append(resi)

natural_to_af2 = {nat: i for i, nat in enumerate(global_ordered_resis)}
af2_to_natural = {i: nat for i, nat in enumerate(global_ordered_resis)}
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K', 'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N', 'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W', 'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

# ==========================================
# 💎 Extract Coherent Validated Core
# ==========================================
validated_core = []
core_master_scores = final_dir / "Isl_0_DeepDive_MasterScores.csv"

if core_master_scores.exists():
    try:
        df_scores = pd.read_csv(core_master_scores)
        # 1. 过滤出物理骨架合规的构象 (pLDDT >= 0.40)
        valid_df = df_scores[df_scores['pLDDT'] >= 0.40]
        if not valid_df.empty:
            # 2. 找到 ipSAE 最高的那个单一物理标杆构象
            best_design = valid_df.sort_values(by='ipSAE', ascending=False).iloc[0]
            abs_pdb = Path(best_design['Abs_Path'])
            pae_json = abs_pdb.parent / f"{abs_pdb.stem}_pae.json"

            # 3. 从单一连贯的矩阵中提取 PAE 最小的 4 个残基
            if pae_json.exists():
                with open(pae_json, 'r') as f: pae_res = json.load(f)
                cross_pae = np.array(pae_res['mean_pae_array'])
                ranked_indices = np.argsort(cross_pae)

                for idx in ranked_indices:
                    nat_r = af2_to_natural.get(idx)
                    if nat_r:
                        validated_core.append(nat_r)
                    if len(validated_core) == 4:
                        break
                print(f"✅ Successfully extracted Top 4 Coherent Core from the best physical structure: {validated_core}")
    except Exception as e:
        print(f"⚠️ Error extracting coherent core: {e}")

if not validated_core or len(validated_core) < 4:
    print("⚠️ Coherent core extraction failed. Falling back to top 4 raw base_hotspots.")
    validated_core = geo_data.get("base_hotspots", [])[:4]

primary_core_coords = [ca_coords[r] for r in validated_core if r in ca_coords]
core_tree = KDTree(primary_core_coords) if primary_core_coords else None

edge_res_pae_tracker = {}
if res_master_csv.exists() and not should_reset:
    try:
        rdf = pd.read_csv(res_master_csv)
        edge_res_pae_tracker = {str(r['Residue_ID']): float(r['Best_PAE']) for _, r in rdf.iterrows()}
    except: pass

clean_vram()

# ==========================================
# 🐺 Processing Islands with Distance Pruning
# ==========================================
for isl in satellite_islands:
    name, island_residues = isl["name"], isl["residues"]

    reachable_shell = []
    if core_tree:
        for r in island_residues:
            if r in ca_coords and r not in validated_core:
                dist, _ = core_tree.query(ca_coords[r], k=1)
                if dist <= REACHABLE_DISTANCE:
                    reachable_shell.append(r)

    actual_batches = len(reachable_shell)
    display(HTML(f"""<hr><h3 style='color: #2E8B57;'>🏝️ Island: {name} | {actual_batches} Reachable Residues</h3>
    <p><b>Constraint:</b> Distance to Validated Core ≤ {REACHABLE_DISTANCE}Å | <b>Strategy:</b> Single-Point Scan.</p>"""))

    if actual_batches == 0: continue

    for batch_idx, target_res in enumerate(reachable_shell, 1):
        current_task_id = f"{name}_R{target_res}"
        if current_task_id in completed_tasks: continue

        run_name = f"{SESSION_NAME}_{name}_R{target_res}"
        # 严格锁定输入：Top 4 Core + 1 个被试边缘残基
        sampled_patch = validated_core + [target_res]
        sampled_patch = sorted(list(set(sampled_patch)), key=lambda x: int(re.sub(r'\D', '', x)))

        print(f"\n   🎯 [{name} - Point {batch_idx}/{actual_batches}] Testing Target: {target_res} (Lineup: {','.join(sampled_patch)})")

        success, parsed_data, duration, inf_dir = run_complexa_batch(
            BASE_DIR, task_name, run_name, target_chains, sampled_patch,
            base_batch_size, base_batch_size, evaluation_batch_size, random.randint(10000, 99999), env
        )

        if success and parsed_data:
            # 寻找既有高 ipSAE，又满足基础折叠度 (pLDDT >= 0.40) 的构象
            valid_designs = [d for d in parsed_data if float(d['pLDDT']) >= 0.40]

            if not valid_designs:
                print(f"      ⚠️ Ghost conformation detected (pLDDT < 0.40) while testing {target_res}. Discarding to prevent edge false positives.")
                completed_tasks.add(current_task_id)
                continue

            best_design = valid_designs[0] # 取物理合规的最高分构象
            abs_pdb_path = best_design['Abs_Path']
            res_pae_val = 999.0
            pae_table_html = "N/A"

            if Path(abs_pdb_path).exists():
                with open(abs_pdb_path, 'r') as f: lines = f.readlines()
                chains_seq = {}
                for l in lines:
                    if l.startswith("ATOM") and l[12:16].strip() == "CA":
                        r_n, c_id = l[17:20].strip(), l[21]
                        chains_seq.setdefault(c_id, "")
                        chains_seq[c_id] += d3to1.get(r_n, 'X')
                binder_seq = chains_seq.get(min(chains_seq.keys(), key=lambda k: len(chains_seq[k])), "N/A")

                if binder_seq != "N/A":
                    pae_out = Path(abs_pdb_path).parent / f"{Path(abs_pdb_path).stem}_pae.json"
                    try:
                        subprocess.run(["python3", str(BASE_DIR / 'run_pae_hook.py'), str(SOURCE_PDB_PATH), target_chains, binder_seq, str(pae_out), str(BASE_DIR)], check=True, capture_output=True)

                        if pae_out.exists():
                            with open(pae_out, 'r') as f: pae_res = json.load(f)
                            cross_pae = np.array(pae_res['mean_pae_array'])

                            for t_idx, p_val in enumerate(cross_pae):
                                nat_r = af2_to_natural.get(t_idx)
                                if nat_r:
                                    if nat_r not in edge_res_pae_tracker or p_val < edge_res_pae_tracker[nat_r]:
                                        edge_res_pae_tracker[nat_r] = float(p_val)

                            if target_res in natural_to_af2:
                                res_pae_val = cross_pae[natural_to_af2[target_res]]

                            r_idx = np.argsort(cross_pae)
                            table_html = "<table style='width:100%; text-align:center; border-collapse: collapse; font-size: 11px;'>"
                            table_html += "<tr><th style='background:#f8f9fa;'>Residue</th>"
                            for i in range(10): table_html += f"<td>{af2_to_natural.get(r_idx[i], 'N/A')}</td>"
                            table_html += "</tr><tr><th style='background:#f8f9fa;'>PAE</th>"
                            for i in range(10): table_html += f"<td>{cross_pae[r_idx[i]]:.1f}</td>"
                            table_html += "</tr></table>"
                            pae_table_html = table_html
                    except: print("      ⚠️ PAE Subprocess failed.")

            if res_pae_val < PAE_DISCARD_THRESHOLD:
                status_badge = f"<span style='color: green; font-weight: bold;'>[ACCEPTED] PAE: {res_pae_val:.2f}Å</span>"
            else:
                status_badge = f"<span style='color: red; font-weight: bold;'>[DISCARDED] PAE: {res_pae_val:.2f}Å</span>"

            batch_df = pd.DataFrame(parsed_data)
            batch_df['Island'], batch_df['Batch'], batch_df['Sampled_Patch'] = name, batch_idx, ','.join(sampled_patch)
            batch_df.to_csv(master_csv_path, mode='a', index=False, header=not master_csv_path.exists())
            pd.DataFrame([{'Residue_ID': k, 'Best_PAE': v} for k, v in edge_res_pae_tracker.items()]).to_csv(res_master_csv, index=False)

            display_text = "\n".join([f"{r['Design_ID']:<16} | ipSAE: {float(r['ipSAE']):.4f}" for r in parsed_data])
            display(HTML(f"""<div style='background: #f8f9fa; padding: 10px; border-left: 4px solid #0056b3;'>
            <b>Testing: {target_res}</b> {status_badge}<pre>{display_text}</pre><div>{pae_table_html}</div></div>"""))

            print(f"   💾 Test Complete. VRAM reclaimed safely.")
            completed_tasks.add(current_task_id)

print("\n🏆 Island-Constrained Scanning Complete.")

# 目的: 执行外周位点单点 PAE 筛选，引入物理强制连贯性约束与假阳性拦截机制。
# 上游代码: 读取 Cell 20b 生成的 Isl_0_DeepDive_MasterScores.csv。
# 运行环境: Google Colab。
# 生成时间: 2026-05-01 14:08 EDT。
# 更改说明:
# 第 82-120 行: 重写 Validated Core 提取逻辑。从读取全局 PAE 大表变更为读取单一物理标杆构象 (pLDDT >= 0.40 且 ipSAE 最高) 的独立 json 矩阵，以确保提取出的 4 个核心在三维空间中保持拓扑连贯，避免弗兰肯斯坦陷阱。
# 第 175-182 行: 添加 pLDDT >= 0.40 的联合验证。发现配体骨架崩溃的 Ghost conformation 直接废弃当前轮次，阻止因极端张力产生的假性极低 PAE 录入追踪器。


🔥 Initializing Island-Constrained Scanning (Coherent Core + Validated Single Point Mode)...
⚠️ [FORCE RESTART] Overwriting previous Edge Island data...
✅ Successfully extracted Top 4 Coherent Core from the best physical structure: ['740', '741', '729', '712']



   🎯 [Isl_A - Point 1/3] Testing Target: 749 (Lineup: 712,729,740,741,749)


Residue,733,831,881,911,835,897,737,855,712,701
PAE,13.1,13.1,13.2,13.2,13.3,13.3,13.3,13.3,13.4,13.4


   💾 Test Complete. VRAM reclaimed safely.

   🎯 [Isl_A - Point 2/3] Testing Target: 751 (Lineup: 712,729,740,741,751)


Residue,701,881,709,715,733,712,877,721,737,740
PAE,10.2,10.2,10.2,10.3,10.3,10.3,10.3,10.3,10.3,10.3


...

   💾 Test Complete. VRAM reclaimed safely.



   🎯 [Isl_Q - Point 1/1] Testing Target: 748 (Lineup: 712,729,740,741,748)


Residue,783,855,786,740,742,804,802,713,871,806
PAE,14.2,14.2,14.3,14.3,14.4,14.4,14.5,14.5,14.6,14.6


   💾 Test Complete. VRAM reclaimed safely.



🏆 Island-Constrained Scanning Complete.


In [ ]:

#@title Cell_23_Peripheral_Islands_Visualizations.py
# --- Separate Visualizations (Smart God-Tier Watch Mode for Edge Islands) ---
# Requirements:
# 1. Integrate dynamic AlphaFold2 static forward pass for PAE calculation on peripheral islands.
# 2. Save PAE matrices as local JSON files alongside the generated PDBs.
# 3. Output the top 10 best amino acid positions in a 3x10 HTML table.
# 4. Compile a master summary table tracking the best PAE for all target residues across evaluated peripheral decoys.

import os, sys, json, re
from pathlib import Path
import numpy as np
import py3Dmol
import pandas as pd
from IPython.display import HTML, display, clear_output

clear_output(wait=True)
print("🎨 Initiating Smart Watch Mode: Rendering Top Edge Island Binders with Static PAE Evaluation...")

try:
    from colabdesign import mk_afdesign_model, clear_mem
except ImportError:
    print("📦 Installing ColabDesign...")
    os.system("pip install git+https://github.com/sokrypton/ColabDesign.git")
    from colabdesign import mk_afdesign_model, clear_mem

# ==========================================
# 🛠️ 1. Directory & Configuration Setup
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
ROOT_SETTING = BASE_DIR / 'internal_settings.json'

if not ROOT_SETTING.exists():
    print("❌ Error: Root configuration missing.")
    sys.exit(1)

with open(ROOT_SETTING, 'r') as f:
    root_cfg = json.load(f)

task_name = root_cfg['task_name']
last_run_file = BASE_DIR / 'screening_results' / 'last_runs.txt'

run_id, target_dir_name = None, None
if last_run_file.exists():
    with open(last_run_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        pattern = rf"Section2_TargetPreprocess(?:_TimeMachine)? \| Run_ID: ({re.escape(task_name)}_\d+_\d+) \| Dir: (final_\d+_\d+)"
        for line in reversed(lines):
            match = re.search(pattern, line)
            if match:
                run_id = match.group(1)
                target_dir_name = match.group(2)
                break

final_dir = BASE_DIR / 'screening_results' / task_name / target_dir_name if run_id else None

print("\n" + "="*80)
print(f"🔑 【Cell 23 Visualizations 启动】 当前全局 Run ID: {run_id}")
print(f"📁 当前 Workspace 目录: {final_dir.relative_to(BASE_DIR) if final_dir else 'N/A'}")
print("="*80 + "\n")

if not final_dir:
    print(f"❌ Error: Run history not found for {task_name}.")
    sys.exit(1)

ISOLATED_SETTING = final_dir / 'internal_settings.json'
with open(ISOLATED_SETTING, 'r') as f:
    cfg = json.load(f)
target_chains = cfg.get('target_chains', 'A').split(',')[0].strip()

# ==========================================
# 🛠️ 2. Load Edge Island Data & Filtering
# ==========================================
master_csv_path = final_dir / "EdgeIslands_DeepDive_MasterScores.csv"

if not master_csv_path.exists():
    print("⏳ Waiting for Peripheral Islands to generate the Master Scores table. Run generation first.")
    sys.exit(1)

df = pd.read_csv(master_csv_path)
if df.empty:
    print("⚠️ Master table is empty. No valid designs generated yet.")
    sys.exit(1)

score_col = 'ipSAE'
df[score_col] = pd.to_numeric(df[score_col], errors='coerce')
df_sorted = df.sort_values(by=score_col, ascending=False).reset_index(drop=True)

god_tier_df = df_sorted[df_sorted[score_col] >= 0.5]

if not god_tier_df.empty:
    print(f"🌟 发现 {len(god_tier_df)} 个 God-Tier 构象 (ipSAE >= 0.5)! 正在为您渲染顶级序列 (Max 10)...")
    render_df = god_tier_df.head(10)
else:
    print("📊 当前批次未发现 >0.5 的构象。已启用兜底机制，展示全局 Top 10...")
    render_df = df_sorted.head(10)

# ==========================================
# 🛠️ 3. Mapping Engine & Static AF2 Setup
# ==========================================
SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"
global_ordered_resis = []

if SOURCE_PDB_PATH.exists():
    with open(SOURCE_PDB_PATH, 'r') as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
                global_ordered_resis.append(line[22:26].strip())

natural_to_af2 = {nat: i+1 for i, nat in enumerate(global_ordered_resis)}
af2_to_natural = {i: nat for i, nat in enumerate(global_ordered_resis)}
target_chain_length = len(global_ordered_resis)

def translate_resi(resi_input):
    if not resi_input or pd.isna(resi_input): return []
    if isinstance(resi_input, str):
        resi_list = [r.strip() for r in str(resi_input).replace(',', ' ').split() if r.strip()]
    else:
        resi_list = resi_input
    res = []
    for x in resi_list:
        if str(x).strip():
            num_str = re.sub(r'\D', '', str(x))
            if num_str:
                if natural_to_af2 and num_str in natural_to_af2:
                    res.append(int(natural_to_af2[num_str]))
                else:
                    res.append(int(num_str))
    return res

# --- AlphaFold2 Parameters Smart Resolver ---
print("⚙️ Verifying AlphaFold2 Multimer Parameters...")
af2_params_found = False
colab_params_dir = Path("/content/params")

common_weights_dirs = [
    BASE_DIR / 'weights',
    BASE_DIR / 'params',
    BASE_DIR / 'assets' / 'weights'
]

for d in common_weights_dirs:
    if (d / "params_model_1_multimer_v3.npz").exists():
        os.system(f"ln -sf {d} /content/params")
        af2_params_found = True
        print(f"🔗 Established symlink to existing parameters: {d}")
        break
    elif (d / "params" / "params_model_1_multimer_v3.npz").exists():
        os.system(f"ln -sf {d}/params /content/params")
        af2_params_found = True
        print(f"🔗 Established symlink to existing parameters: {d}/params")
        break

if not af2_params_found and not (colab_params_dir / "params_model_1_multimer_v3.npz").exists():
    print("📦 Parameters not found. Downloading to local ephemeral storage (Fast, ~1-2 mins)...")
    os.system("mkdir -p /content/params")
    os.system("curl -fsSL https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar | tar x -C /content/params")
    print("✅ Download completed.")

print("⚙️ Initializing AlphaFold2 Static Forward Pass Model Environment...")
clear_mem()
af_model = mk_afdesign_model(protocol="binder", use_multimer=True, data_dir="/content")

# Global tracker for best PAE per residue across all evaluated edge designs
global_best_pae_tracker = {}
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K', 'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N', 'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W', 'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

# ==========================================
# 🎨 4. Render Loop with PAE Extraction
# ==========================================
for idx, row in render_df.iterrows():
    island_name = row['Island']
    batch_idx = row['Batch']
    design_id = row['Design_ID']
    ipsae_val = row['ipSAE']
    abs_pdb_path = row['Abs_Path']

    core_residues_raw = str(row['Core_Residues'])
    sampled_patch_raw = str(row['Sampled_Patch'])

    if abs_pdb_path == "N/A" or not Path(abs_pdb_path).exists():
        print(f"⚠️ PDB not found for {design_id} at {abs_pdb_path}, skipping...")
        continue

    with open(abs_pdb_path, 'r') as f: pdb_data = f.read()

    chains_seq = {}
    first_resi_binder = None
    for line in pdb_data.split('\n'):
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            res_name = line[17:20].strip()
            chain_id = line[21]
            if chain_id not in chains_seq: chains_seq[chain_id] = ""
            chains_seq[chain_id] += d3to1.get(res_name, 'X')

    chain_lengths = {c: len(seq) for c, seq in chains_seq.items()}
    if not chain_lengths: continue
    sorted_chains = sorted(chain_lengths.keys(), key=lambda c: chain_lengths[c])
    binder_chains = [sorted_chains[0]]
    target_chain_ids = [c for c in sorted_chains[1:]]

    for line in pdb_data.split('\n'):
        if line.startswith("ATOM") and line[21] == binder_chains[0]:
            first_resi_binder = line[22:26].strip()
            break

    binder_seq = chains_seq.get(binder_chains[0], "N/A")
    if binder_seq == "N/A": continue

    island_core_int = translate_resi(core_residues_raw)
    sampled_patch_int = translate_resi(sampled_patch_raw)

    first_resi_binder_int = None
    if first_resi_binder and str(first_resi_binder).strip():
        num_str = re.sub(r'\D', '', str(first_resi_binder))
        if num_str: first_resi_binder_int = int(num_str)

    # --- Start Dynamic PAE Evaluation & Export ---
    binder_length = len(binder_seq)
    print(f"🧬 Extracting PAE for {design_id} (Binder Length: {binder_length})...")

    af_model.prep_inputs(pdb_filename=str(SOURCE_PDB_PATH), chain=target_chains, binder_len=binder_length)
    len_target = af_model._lengths[0]

    af_model.restart(seq=binder_seq)
    af_model.predict(num_recycles=1)

    pae_matrix = af_model.aux["pae"]

    pdb_path_obj = Path(abs_pdb_path)
    pae_out_path = pdb_path_obj.parent / f"{pdb_path_obj.stem}_pae.json"
    with open(pae_out_path, 'w') as f:
        json.dump({"predicted_aligned_error": pae_matrix.tolist()}, f)

    cross_pae_target_to_binder = pae_matrix[:len_target, len_target:]
    mean_pae_per_residue = np.mean(cross_pae_target_to_binder, axis=1)

    # Update global tracker for all residues
    for t_idx, p_val in enumerate(mean_pae_per_residue):
        nat_r = af2_to_natural.get(t_idx)
        if nat_r:
            if nat_r not in global_best_pae_tracker or p_val < global_best_pae_tracker[nat_r]['Best_PAE']:
                global_best_pae_tracker[nat_r] = {
                    'Residue_ID': nat_r,
                    'Best_PAE': float(p_val),
                    'Source_Design': design_id
                }

    ranked_indices = np.argsort(mean_pae_per_residue)

    # Build 3-row, 10-column HTML table
    table_html = "<table style='width:100%; text-align:center; border-collapse: collapse; font-size: 12px; margin-top: 5px;'>"
    table_html += "<tr><th style='border: 1px solid #ddd; padding: 4px; background-color: #e9ecef;'>Rank</th>"
    for i in range(10):
        table_html += f"<td style='border: 1px solid #ddd; padding: 4px; font-weight: bold; background-color: #f8f9fa;'>#{i+1}</td>"
    table_html += "</tr><tr><th style='border: 1px solid #ddd; padding: 4px; background-color: #e9ecef;'>Residue</th>"
    for i in range(10):
        target_idx = ranked_indices[i]
        nat_resi = af2_to_natural.get(target_idx, f"Idx_{target_idx}")
        table_html += f"<td style='border: 1px solid #ddd; padding: 4px;'>{nat_resi}</td>"
    table_html += "</tr><tr><th style='border: 1px solid #ddd; padding: 4px; background-color: #e9ecef;'>PAE</th>"
    for i in range(10):
        target_idx = ranked_indices[i]
        pae_val = mean_pae_per_residue[target_idx]
        color = "green" if pae_val < 10 else ("orange" if pae_val < 20 else "red")
        table_html += f"<td style='border: 1px solid #ddd; padding: 4px; color:{color}; font-weight: bold;'>{pae_val:.2f}</td>"
    table_html += "</tr></table>"

    top_10_display = table_html
    # --- End Dynamic PAE Evaluation ---

    html_content = f"""
    <hr>
    <h3 style='color: #0066cc;'>{island_name} | Batch {batch_idx} | Design: {design_id}</h3>
    <div style='background: #f8f9fa; padding: 10px; border-radius: 6px; margin-bottom: 10px; border-left: 4px solid #0066cc;'>
        <p style='margin: 5px 0;'><b>🏆 ipSAE:</b> <span style='color:red; font-size:16px; font-weight:bold;'>{ipsae_val:.4f}</span> | <b style='color:red;'>Sampled Extended Patch:</b> {sampled_patch_raw}</p>
        <p style='margin: 5px 0;'><b>🧬 Peptide Sequence:</b> <code style='background:#fff; padding:2px 6px; border: 1px solid #ddd; border-radius:4px; font-size:14px; color:#d63384;'>{binder_seq}</code></p>
        <p style='margin: 5px 0; font-size: 13px;'><b>💾 PAE Saved To:</b> {pae_out_path.name}</p>
        <div style='margin-top: 8px; padding-top: 8px; border-top: 1px dashed #ccc;'>
            <p style='margin: 0 0 5px 0; font-size: 13px; font-weight: bold; color: #555;'>🔬 Best PAE Amino Acids:</p>
            {top_10_display}
        </div>
    </div>
    <div style='font-size: 13px; color: #555; margin-bottom: 5px;'>
        <b>Color Map:</b>
        <span style='color:orange; font-weight:bold;'>Island Core Anchor (Orange)</span>
        <span style='color:red; font-weight:bold;'>Active Sampled Patch (Red)</span>
        <span style='color:lime; font-weight:bold;'>Peptide (Green)</span>
        <span style='color:blue; font-weight:bold;'>Peptide N-Term (Blue)</span>
    </div>
    """
    display(HTML(html_content))

    viewer = py3Dmol.view(width=800, height=500)
    viewer.addModel(pdb_data, 'pdb')

    target_sel = {'chain': target_chain_ids}
    viewer.setStyle(target_sel, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.7}})
    viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'wireframe': True}, target_sel)

    if island_core_int:
        c_sel = {'chain': target_chain_ids, 'resi': island_core_int}
        viewer.setStyle(c_sel, {'cartoon': {'color': 'orange', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.VDW, {'color': 'orange', 'opacity': 0.8}, c_sel)

    if sampled_patch_int:
        n_sel = {'chain': target_chain_ids, 'resi': sampled_patch_int}
        viewer.setStyle(n_sel, {'cartoon': {'color': 'red', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.VDW, {'color': 'red', 'opacity': 1.0}, n_sel)

    if binder_chains:
        b_sel = {'chain': binder_chains}
        viewer.setStyle(b_sel, {'cartoon': {'color': 'lime', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.SES, {'color': 'lime', 'opacity': 1.0}, b_sel)

    if first_resi_binder_int and binder_chains:
        nterm_sel = {'chain': binder_chains[0], 'resi': [first_resi_binder_int]}
        viewer.setStyle(nterm_sel, {'cartoon': {'color': 'blue', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.VDW, {'color': 'blue', 'opacity': 1.0}, nterm_sel)

    viewer.zoomTo()
    viewer.show()

# ==========================================
# 📊 5. Export Global Best PAE Summary for Peripheral Islands
# ==========================================
if global_best_pae_tracker:
    pae_summary_path = final_dir / "Peripheral_Global_Best_Cross_PAE_Summary.csv"
    pae_summary_list = list(global_best_pae_tracker.values())
    df_pae_summary = pd.DataFrame(pae_summary_list)
    df_pae_summary['Residue_Num'] = df_pae_summary['Residue_ID'].apply(lambda x: int(re.sub(r'\D', '', str(x))))
    df_pae_summary = df_pae_summary.sort_values('Residue_Num').drop(columns=['Residue_Num'])
    df_pae_summary.to_csv(pae_summary_path, index=False)
    print(f"\n✅ 边缘 Island 全局最优 PAE 总表已保存至: {pae_summary_path.absolute()}")

# # # Purpose: Render top candidates for peripheral islands, perform a dynamic static PAE matrix extraction, format output into a 3x10 HTML table, and compile a master tracking table recording the lowest PAE scores per residue across all decoys.
# # # Upstream Code: Peripheral Islands Screening Pipeline (EdgeIslands_DeepDive_MasterScores.csv).
# # # Runtime Environment: Google Colab.
# # # Generation Time: 2026-04-28 18:14 EDT.
# # # Changed Lines:
# # # 行 111-137: 引入了 colabdesign 环境检测与参数软链接逻辑。
# # # 行 206-224: 移除了原有的静态 JSON 解析函数，改为读取 binder 长度后重构 JAX 计算图，实时输出同级 PAE JSON 文件。
# # # 行 225-233: 添加了 global_best_pae_tracker 记录逻辑，与 Cell 21 保持算法一致。
# # # 行 236-250: 生成包含 Rank、Residue、PAE 结构的紧凑型 HTML 3行10列数据表。
# # # 行 307-316: 将聚合字典持久化保存为 Peripheral_Global_Best_Cross_PAE_Summary.csv。

In [ ]:

# Cell_24_Head_Wolf_Evolution.py
# 最终修复版：强制 Pre-flight Summary Panel + DualLogger 同步问题修复

# ==========================================
# 🟢 User Configuration
# ==========================================
force_restart_evo = True
specific_resume_run_id_evo = ""
max_parallel_universes = 4

spatial_distance_cutoff = 20.0
dynamic_batch_early_stop_plddt = 0.52
PATIENCE_LIMIT = 5
BEAM_WIDTH = 4
IP_WEIGHT = 1.0
PLDDT_WEIGHT = 2.8

import os, sys, json, datetime, re, random, subprocess, gc
from pathlib import Path
from typing import List, Dict
import numpy as np
import pandas as pd
from IPython.display import HTML, display
import torch
from scipy.spatial import KDTree
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import AgglomerativeClustering

# ====================== DualLogger 加强版 ======================
class DualLogger:
    def __init__(self, filepath):
        self.terminal = sys.__stdout__
        self.log = open(filepath, "a", buffering=1, encoding="utf-8")

    def write(self, message):
        if self.terminal:
            self.terminal.write(message)
        if self.log:
            self.log.write(message)

    def flush(self):
        if self.terminal:
            self.terminal.flush()
        if self.log:
            self.log.flush()

# 强制输出函数（解决 Jupyter stdout 不同步）
def force_print(msg: str):
    sys.__stdout__.write(msg + "\n")
    sys.__stdout__.flush()

# ====================== 初始化 ======================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
ROOT_SETTING = BASE_DIR / 'internal_settings.json'
with open(ROOT_SETTING, 'r') as f:
    root_cfg = json.load(f)

task_name = root_cfg['task_name']
force_restart_section_7 = bool(root_cfg.get('force_restart_section_7', False))

run_id, final_dir = resolve_workspace(BASE_DIR, task_name, specific_resume_run_id_evo)

LIVE_LOG_FILE = final_dir / f"Live_Sync_{run_id}.log"
logger = DualLogger(LIVE_LOG_FILE)
sys.stdout = logger
sys.stderr = logger

print("🐺 Initializing Head-Wolf Algorithm: Target-Biased Empirical Footprint + Tension Diagnosis Evolution...")
print(f"🏆 Balanced Parent Selection 已启用 | IP_WEIGHT={IP_WEIGHT} | PLDDT_WEIGHT={PLDDT_WEIGHT} | Cutoff={spatial_distance_cutoff}Å")
print(f"📡 Dual Logger 已接管: {LIVE_LOG_FILE}")

# ==========================================
# Balanced Score
# ==========================================
def balanced_score(c: Dict, ip_weight: float = 1.0, plddt_weight: float = 2.8) -> float:
    ip = float(c.get('ipSAE', 0.0))
    pld = float(c.get('pLDDT', 0.0))
    if pld < 0.40:
        pld_factor = pld * 0.5
    elif pld < 0.55:
        pld_factor = pld * 1.8
    else:
        pld_factor = pld ** 1.8
    return (ip ** ip_weight) * (pld_factor ** plddt_weight)

# ====================== 辅助函数 ======================
def extract_footprint_and_scores(pdb_path: str, target_chain: str, af2_to_nat: dict) -> dict:
    consensus = {}
    if not Path(pdb_path).exists():
        return consensus
    af2_tgt, af2_bnd = {}, {}
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() in ["CA", "CB"]:
                c, r, n = line[21], line[22:26].strip(), line[12:16].strip()
                co = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                if c == target_chain:
                    if r not in af2_tgt: af2_tgt[r] = {}
                    af2_tgt[r][n] = co
                else:
                    if r not in af2_bnd: af2_bnd[r] = {}
                    af2_bnd[r][n] = co
    t_rep = {r: v.get('CB', v.get('CA')) for r, v in af2_tgt.items()}
    b_rep = [v.get('CB', v.get('CA')) for v in af2_bnd.values() if v.get('CB', v.get('CA')) is not None]
    if not b_rep:
        return consensus
    tree = KDTree(b_rep)
    for r, co in t_rep.items():
        if co is None: continue
        dist, _ = tree.query(co, k=1)
        if dist < 4.5:
            try:
                idx = int(r) - 1
                nat_r = af2_to_nat.get(idx)
                if nat_r:
                    consensus[nat_r] = 1.0 / (dist + 0.1)
            except:
                pass
    return consensus

def cluster_by_footprint(designs: List[Dict], af2_to_nat: dict, distance_threshold=0.6) -> Dict[int, List[Dict]]:
    if not designs:
        return {}
    all_res_set = set()
    for d in designs:
        fp_dict = extract_footprint_and_scores(d['pdb_path'], target_chains, af2_to_nat)
        d['footprint'] = set(fp_dict.keys())
        all_res_set.update(d['footprint'])
    all_residues = list(all_res_set)
    if not all_residues:
        return {0: designs}
    feature_matrix = np.zeros((len(designs), len(all_residues)))
    for i, d in enumerate(designs):
        for j, res in enumerate(all_residues):
            if res in d['footprint']:
                feature_matrix[i, j] = 1
    dist_matrix = pdist(feature_matrix, metric='jaccard')
    if len(dist_matrix) == 0 or np.all(dist_matrix == 0):
        return {0: designs}
    clustering = AgglomerativeClustering(n_clusters=None, metric='precomputed', linkage='average', distance_threshold=distance_threshold)
    labels = clustering.fit_predict(squareform(dist_matrix))
    clusters = {}
    for label, d in zip(labels, designs):
        clusters.setdefault(label, []).append(d)
    return clusters

def select_cluster_champions(clusters: Dict[int, List[Dict]]) -> List[Dict]:
    champions = []
    for cid, members in clusters.items():
        if not members:
            continue
        sorted_m = sorted(members, key=lambda x: x.get('balanced_score', 0), reverse=True)
        champions.append(sorted_m[0])
    return champions

# ====================== 配置加载 ======================
ISOLATED_SETTING = final_dir / 'internal_settings.json'
GEOMETRIC_MANIFEST = final_dir / "Geometric_Islands_Manifest.json"

with open(ISOLATED_SETTING, 'r') as f:
    cfg = json.load(f)
with open(GEOMETRIC_MANIFEST, 'r') as f:
    geo_data = json.load(f)

target_chains = str(cfg.get('target_chains', 'A')).split(',')[0].strip()
gpu_batch_size = 4
generation_batches_per_ev = 3
evaluation_batch_size = int(root_cfg.get('evaluation_batch_size', 1))
SESSION_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
TRAJECTORY_CSV = final_dir / "Evolution_Trajectory.csv"

should_reset = force_restart_section_7 or force_restart_evo
if should_reset and TRAJECTORY_CSV.exists():
    print("🔄 [FORCE RESTART] Clearing previous evolution trajectory...")
    TRAJECTORY_CSV.unlink()

if not TRAJECTORY_CSV.exists():
    with open(TRAJECTORY_CSV, 'w') as f:
        f.write("Iteration,Cluster_ID,Branch_ID,ipSAE,pLDDT,Action,Current_Hotspots,PDB_Path\n")

env = os.environ.copy()
env.update({
    'HYDRA_FULL_ERROR': '1',
    'PYTHONUNBUFFERED': '1',
    'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
    'XLA_PYTHON_CLIENT_ALLOCATOR': 'platform',
    'TF_FORCE_GPU_ALLOW_GROWTH': 'true'
})

# ====================== 数据准备 ======================
SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"
rep_coords, master_pool = {}, []
with open(SOURCE_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
            r = line[22:26].strip()
            rep_coords[r] = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            master_pool.append(r)

natural_to_af2 = {nat: i for i, nat in enumerate(master_pool)}
af2_to_natural = {i: nat for i, nat in enumerate(master_pool)}
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K', 'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N', 'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W', 'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

edge_pae_tracker = {}
EDGE_RES_CSV = final_dir / "Edge_Residue_PAE_Master.csv"
if EDGE_RES_CSV.exists():
    e_df = pd.read_csv(EDGE_RES_CSV)
    for _, row in e_df.iterrows():
        p_val = float(row['Best_PAE'])
        if p_val <= 12.0:
            clean_id = str(int(float(row['Residue_ID'])))
            edge_pae_tracker[clean_id] = p_val

# ====================== 强制 Pre-flight Summary Panel ======================
force_print("\n" + "═"*110)
force_print("🚀 PRE-FLIGHT SUMMARY PANEL - HEAD WOLF EVOLUTION")
force_print("═"*110)

alpha_seeds = {}

user_target_res = geo_data.get("base_hotspots", [])
target_coords = [rep_coords[r] for r in user_target_res if r in rep_coords]
target_centroid = np.mean(target_coords, axis=0) if target_coords else np.mean(list(rep_coords.values()), axis=0)

UPSTREAM_CSV = final_dir / "EdgeIslands_DeepDive_MasterScores.csv"
if not UPSTREAM_CSV.exists():
    UPSTREAM_CSV = final_dir / "Isl_0_DeepDive_MasterScores.csv"

if UPSTREAM_CSV.exists():
    df_upstream = pd.read_csv(UPSTREAM_CSV)
    df_upstream['pLDDT'] = pd.to_numeric(df_upstream['pLDDT'], errors='coerce')
    valid_designs = df_upstream[df_upstream['pLDDT'] >= 0.40].copy()

    force_print(f"1. 上游筛选结果：共读取 {len(valid_designs)} 个 pLDDT ≥ 0.40 的合规底座。")

    if not valid_designs.empty:
        valid_designs['balanced_score'] = valid_designs.apply(
            lambda row: balanced_score(row.to_dict(), IP_WEIGHT, PLDDT_WEIGHT), axis=1
        )
        top_pool = valid_designs.nlargest(30, 'balanced_score')

        force_print(f"\n2. 入围的 Top 结构名单（Top {len(top_pool)}）：")
        force_print("-" * 130)
        force_print(f"{'Rank':<4} {'Filename/ID':<55} {'ipSAE':<8} {'pLDDT':<8} {'Balanced Score':<14} Hotspots")
        force_print("-" * 130)

        cluster_input = []
        for rank, (_, row) in enumerate(top_pool.iterrows(), 1):
            filename = Path(row['Abs_Path']).name
            bscore = row['balanced_score']
            hotspots = str(row.get('Hotspots', 'N/A'))[:50]
            force_print(f"{rank:<4} {filename:<55} {row['ipSAE']:<8.4f} {row['pLDDT']:<8.3f} {bscore:<14.5f} {hotspots}")

            cluster_input.append({
                'pdb_path': row['Abs_Path'],
                'ipSAE': row['ipSAE'],
                'pLDDT': row['pLDDT'],
                'balanced_score': bscore,
                'row_data': row.to_dict()
            })

        force_print(f"\n 🧬 Clustering Top {len(top_pool)} structures to ensure spatial diversity...")
        struct_clusters = cluster_by_footprint(cluster_input, af2_to_natural, distance_threshold=0.6)
        force_print(f" 🔍 分析完毕：共解析出 {len(struct_clusters)} 个独立结构簇。")

        champions_pool = []
        for cid, members in struct_clusters.items():
            best_member = max(members, key=lambda x: x.get('balanced_score', 0))
            fp_coords = [rep_coords[r] for r in best_member.get('footprint', []) if r in rep_coords]
            fp_centroid = np.mean(fp_coords, axis=0) if fp_coords else np.zeros(3)
            dist_to_target = np.linalg.norm(fp_centroid - target_centroid)
            best_member['dist_to_target'] = dist_to_target
            champions_pool.append(best_member)

        champions_pool.sort(key=lambda x: (x['dist_to_target'], -x.get('balanced_score', 0)))

        force_print(f"\n3. 最终建立的 Alpha Seeds（共 {min(len(champions_pool), max_parallel_universes)} 个）：")
        force_print("-" * 100)
        rank = 1
        for champ in champions_pool:
            if rank > max_parallel_universes:
                break
            seed_lineup = sorted(list(champ.get('footprint', set())), key=lambda x: int(re.sub(r'\D', '', x)))
            if not seed_lineup:
                continue
            lineup_sig = ",".join(seed_lineup)
            cluster_id = f"Clst_Top{rank}_{seed_lineup[0]}"
            alpha_seeds[cluster_id] = seed_lineup
            force_print(f" 🎯 [Seed {cluster_id}]")
            force_print(f"    Distance to Target : {champ['dist_to_target']:.1f}Å")
            force_print(f"    ipSAE              : {champ['ipSAE']:.4f}")
            force_print(f"    pLDDT              : {champ['pLDDT']:.3f}")
            force_print(f"    Balanced Score     : {champ.get('balanced_score', 0):.5f}")
            force_print(f"    Initial Anchors    : {lineup_sig} ({len(seed_lineup)} aa)")
            force_print("-" * 90)
            rank += 1
else:
    force_print("⚠️ 上游 CSV 文件不存在！")

force_print(f"\n🌟 最终注入 {len(alpha_seeds)} 个高质量 Alpha Seeds 进入进化主循环。")
force_print("═"*110 + "\n")

if not alpha_seeds:
    raise ValueError("⚠️ No valid physical structures found to extract seeds.")

best_global_ipsae = -999.0
completed_clusters = set()

if TRAJECTORY_CSV.exists() and os.path.getsize(TRAJECTORY_CSV) > 0:
    df_traj_init = pd.read_csv(TRAJECTORY_CSV)
    if not df_traj_init.empty:
        completed_clusters.update(df_traj_init[df_traj_init['Action'].str.contains("CONVERGED", na=False)]['Cluster_ID'].astype(str).tolist())

clean_vram()

# ==========================================
# 🐺 PAE-Guided Greedy Evolution Loop
# ==========================================
for cluster_id, initial_lineup in alpha_seeds.items():
    if cluster_id in completed_clusters:
        print(f"⏩ Skipping {cluster_id} (Already Converged).")
        continue

    print(f"\n" + "="*70)
    print(f"📍 [{cluster_id}] Initiating Footprint-Clustered Greedy Evolution")
    print("="*70)

    local_pool = [r for r in master_pool if r not in initial_lineup and r in rep_coords]
    local_pool.sort(key=lambda x: edge_pae_tracker.get(x, 999.0))

    iteration = 1
    active_branches = {0: {'lineup': initial_lineup[:], 'champion_score': -999.0}}
    cluster_best_score = -999.0
    cluster_valid_best = -999.0
    current_patience = PATIENCE_LIMIT

    while True:
        display(HTML(f"<hr><h4 style='color: #8B008B;'>🔄 {cluster_id} | Iteration {iteration}</h4>"))
        current_generation_designs = []
        iteration_all_data = []

        for branch_id, branch_data in active_branches.items():
            lineup = branch_data['lineup']
            print(f"🚀 Branch {branch_id} Lineup ({len(lineup)} aa): {','.join(lineup)}", flush=True)

            for batch_idx in range(1, generation_batches_per_ev + 1):
                run_name = f"Beam_{cluster_id}_I{iteration}_Br{branch_id}_B{batch_idx}_{SESSION_TIMESTAMP}"
                dynamic_seed = random.randint(10000, 99999)
                print(f" ⏳ [Batch {batch_idx}/{generation_batches_per_ev}] Generating (BSize=4)... ", end="", flush=True)

                success, parsed_data, duration, inf_dir = run_complexa_batch(
                    BASE_DIR, task_name, run_name, target_chains, lineup,
                    gpu_batch_size, gpu_batch_size, evaluation_batch_size, dynamic_seed, env
                )

                if success and parsed_data:
                    print("✅", flush=True)
                    display_text = "\n".join([f" {r['Design_ID']:<16} | ipSAE: {float(r['ipSAE']):.4f} | pLDDT: {float(r['pLDDT']):.2f}" for r in parsed_data])
                    display(HTML(f"<div style='background: #f8f9fa; padding: 5px; margin-left: 20px; border-left: 3px solid #0056b3;'><pre>{display_text}</pre></div>"))

                    df = pd.DataFrame(parsed_data)
                    for _, row in df.nlargest(2, 'ipSAE').iterrows():
                        current_generation_designs.append({
                            'branch_id': branch_id,
                            'ipSAE': row['ipSAE'],
                            'pLDDT': row['pLDDT'],
                            'pdb_path': row['Abs_Path'],
                            'lineup': lineup
                        })
                    for _, row in df.iterrows():
                        row_dict = row.to_dict()
                        row_dict['Branch'] = branch_id
                        row_dict['Batch'] = batch_idx
                        iteration_all_data.append(row_dict)

                    if float(df['pLDDT'].max()) >= dynamic_batch_early_stop_plddt:
                        print(f" ⚡ 能量漏斗极度收敛 (pLDDT {df['pLDDT'].max():.2f} >= {dynamic_batch_early_stop_plddt})。提前终止冗余采样.", flush=True)
                        break
                else:
                    print("❌", flush=True)

        if not current_generation_designs:
            break

        struct_clusters = cluster_by_footprint(current_generation_designs, af2_to_natural)
        champions_pool = select_cluster_champions(struct_clusters)
        sorted_champions = sorted(champions_pool, key=lambda c: balanced_score(c, IP_WEIGHT, PLDDT_WEIGHT), reverse=True)
        champions = sorted_champions[:BEAM_WIDTH]

        print(f" 🏆 Selected {len(champions)} Alpha Wolves from distinct structural factions.", flush=True)

        current_max = float(champions[0]['ipSAE']) if champions else -999.0
        if current_max > best_global_ipsae:
            best_global_ipsae = current_max

        valid_champs = [c for c in champions if float(c.get('pLDDT', 0)) >= 0.40]
        made_progress = False
        if valid_champs:
            current_valid_max = float(valid_champs[0]['ipSAE'])
            if current_valid_max > cluster_valid_best:
                cluster_valid_best = current_valid_max
                made_progress = True
        if current_max > cluster_best_score:
            cluster_best_score = current_max
            made_progress = True

        if made_progress:
            current_patience = PATIENCE_LIMIT
        else:
            current_patience -= 1
            print(f"📉 No improvement. Patience: {current_patience}/{PATIENCE_LIMIT}", flush=True)

        if not local_pool or current_patience <= 0:
            with open(TRAJECTORY_CSV, 'a') as f:
                f.write(f"{iteration},{cluster_id},Converged,{cluster_best_score:.4f},-999.0,CONVERGED,\"\",\"\"\n")
            completed_clusters.add(cluster_id)
            break

        new_active_branches = {}
        for idx, champ in enumerate(champions):
            parent_b_id = champ.get('branch_id', 0)
            parent_lineup = champ.get('lineup', [])
            abs_pdb_path = champ.get('pdb_path')
            champ_plddt = float(champ.get('pLDDT', 0.0))

            kicked_by_tension = None
            res_pae_dict = {}

            sorted_by_pae = sorted(parent_lineup, key=lambda x: res_pae_dict.get(x, 999.0))
            drop_count = 2 if len(sorted_by_pae) >= 6 else 1
            survivors = sorted_by_pae[:-drop_count] if drop_count < len(sorted_by_pae) else sorted_by_pae

            target_anchors = len(parent_lineup)
            if champ_plddt < 0.40 and len(parent_lineup) > 3:
                target_anchors = max(3, target_anchors - 1)
                print(f" 📉 张力惩罚: pLDDT={champ_plddt:.2f} → 动态降级至 {target_anchors} 个锚点", flush=True)

            add_count = max(0, target_anchors - len(survivors))

            current_coords = [rep_coords[r] for r in parent_lineup if r in rep_coords]
            centroid = np.mean(current_coords, axis=0) if current_coords else None

            new_adds = []
            elements_to_remove = []
            for candidate in local_pool[:]:
                if len(new_adds) >= add_count:
                    break
                if centroid is not None and candidate in rep_coords:
                    dist = np.linalg.norm(rep_coords[candidate] - centroid)
                    if dist <= spatial_distance_cutoff:
                        new_adds.append(candidate)
                        elements_to_remove.append(candidate)
                else:
                    new_adds.append(candidate)
                    elements_to_remove.append(candidate)

            for item in elements_to_remove:
                if item in local_pool:
                    local_pool.remove(item)

            new_lineup = sorted(list(set(survivors + new_adds)), key=lambda x: int(re.sub(r'\D', '', x)))

            new_active_branches[idx] = {'lineup': new_lineup, 'champion_score': float(champ.get('ipSAE', 0))}

            kicked = [r for r in parent_lineup if r not in new_lineup]
            print(f" 🔪 [Faction {idx}] Kicked: {kicked} | Added: {new_adds}", flush=True)

            with open(TRAJECTORY_CSV, 'a') as f:
                f.write(f"{iteration},{cluster_id},{idx},{champ.get('ipSAE',0):.4f},{champ_plddt:.4f},\"Greedy Rep Br{parent_b_id}\",\"{','.join(new_lineup)}\",\"{abs_pdb_path}\"\n")

        active_branches = new_active_branches
        iteration += 1

print("\n🏆 Evolution Sweeps Completed!")

In [ ]:
# @title Cell_25_analyze_evolution_trajectory.py
# 需求：自动解析并定位最新的 Evolution_Trajectory.csv 演化日志路径，提取 ipSAE 和 pLDDT 的合规统计信息。在终端打印文字版进化过程二维坐标流向，并为每个 Seed 宇宙独立生成一张二维散点流向图，横坐标为 ipSAE，纵坐标为 pLDDT，使用箭头指示演化方向。

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import json
import re

# 忽略作图时的一些常规警告
warnings.filterwarnings("ignore")

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
ROOT_SETTING = BASE_DIR / 'internal_settings.json'
LAST_RUN_FILE = BASE_DIR / 'screening_results' / 'last_runs.txt'

def get_latest_trajectory_path() -> str:
    """自动解析最新的 Run ID 并定位 Evolution_Trajectory.csv 路径"""
    default_path = "Evolution_Trajectory.csv"
    if not ROOT_SETTING.exists() or not LAST_RUN_FILE.exists():
        return default_path

    try:
        with open(ROOT_SETTING, 'r') as f:
            task_name = json.load(f).get('task_name', '')

        with open(LAST_RUN_FILE, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            pattern = rf"Section2_TargetPreprocess(?:_TimeMachine)? \| Run_ID: ({re.escape(task_name)}_\d+_\d+)"

            for line in reversed(lines):
                match = re.search(pattern, line)
                if match:
                    run_id = match.group(1)
                    # 尝试定位 inference 目录或 final 目录下的 CSV
                    target_csv = BASE_DIR / 'inference' / run_id / "Evolution_Trajectory.csv"
                    if target_csv.exists(): return str(target_csv)

                    parts = run_id.split('_')
                    if len(parts) >= 2:
                        final_dir_name = f"final_{parts[-2]}_{parts[-1]}"
                        target_csv_alt = BASE_DIR / 'screening_results' / task_name / final_dir_name / "Evolution_Trajectory.csv"
                        if target_csv_alt.exists(): return str(target_csv_alt)
    except Exception as e:
        print(f"⚠️ 路径自动解析异常: {e}")

    return default_path

def analyze_and_plot_trajectory(csv_path: str, plddt_threshold: float = 0.40):
    """
    解析演化轨迹，统计优秀结构，打印文字版流向并绘制独立流向图。
    """
    file_path = Path(csv_path)
    if not file_path.exists():
        print(f"❌ 找不到文件: {file_path}")
        return

    print(f"📡 已锁定演化日志数据源: {file_path}")

    # 1. 数据清洗与加载
    df = pd.read_csv(file_path)
    df_clean = df[~df['Action'].astype(str).str.contains("Converged|Action", case=False, na=False)].copy()

    for col in ['ipSAE', 'pLDDT', 'Iteration']:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

    df_clean = df_clean.dropna(subset=['ipSAE', 'pLDDT', 'Iteration'])
    df_clean['Iteration'] = df_clean['Iteration'].astype(int)

    # 2. 核心统计输出
    print(f"\n=== 🌟 演化轨迹全景统计 (物理合规基准: pLDDT >= {plddt_threshold}) ===")
    df_valid = df_clean[df_clean['pLDDT'] >= plddt_threshold]
    clusters = df_clean['Cluster_ID'].unique()

    for cluster in clusters:
        c_data = df_clean[df_clean['Cluster_ID'] == cluster]
        c_valid = df_valid[df_valid['Cluster_ID'] == cluster]

        best_raw = c_data.loc[c_data['ipSAE'].idxmax()] if not c_data.empty else None
        best_valid = c_valid.loc[c_valid['ipSAE'].idxmax()] if not c_valid.empty else None

        print(f"\n📍 种子宇宙: {cluster} (共 {len(c_data)} 个采样记录)")
        if best_raw is not None:
            print(f"  🏆 极限探索 (Raw Max):  ipSAE = {best_raw['ipSAE']:.4f} | pLDDT = {best_raw['pLDDT']:.4f} (Iter {int(best_raw['Iteration'])})")
            print(f"      ➥ 对应锚点: {best_raw['Current_Hotspots']}")

        if best_valid is not None:
            print(f"  🛡️ 物理合规 (Valid Max): ipSAE = {best_valid['ipSAE']:.4f} | pLDDT = {best_valid['pLDDT']:.4f} (Iter {int(best_valid['Iteration'])})")
            print(f"      ➥ 对应路径: {best_valid['PDB_Path']}")
        else:
            print(f"  ⚠️ 本支线未能产出 pLDDT >= {plddt_threshold} 的合规底座。")

        # 3. 提取每一代头狼，打印文字版二维坐标流向
        iter_max = c_data.loc[c_data.groupby('Iteration')['ipSAE'].idxmax()].sort_values('Iteration')

        print(f"\n  🧬 [文字版轨迹] {cluster} 进化坐标 (ipSAE, pLDDT):")
        path_str = " ➔\n    ".join([
            f"Iter {int(row['Iteration'])} ({row['ipSAE']:.4f}, {row['pLDDT']:.4f})"
            for _, row in iter_max.iterrows()
        ])
        print(f"    {path_str}\n")

        # 4. 为每个 Seed 生成独立的二象限轨迹图
        plt.figure(figsize=(7, 6))

        sns.scatterplot(
            data=c_data, x='ipSAE', y='pLDDT', hue='Iteration',
            palette='viridis', alpha=0.5, edgecolor=None, s=40
        )

        points = iter_max[['ipSAE', 'pLDDT', 'Iteration']].values

        for j in range(len(points) - 1):
            x1, y1, _ = points[j]
            x2, y2, _ = points[j+1]
            plt.annotate(
                "", xy=(x2, y2), xycoords='data',
                xytext=(x1, y1), textcoords='data',
                arrowprops=dict(arrowstyle="->,head_width=0.4,head_length=0.6", color="red", lw=2.0, alpha=0.8)
            )

        plt.axhline(y=plddt_threshold, color='crimson', linestyle='--', alpha=0.6, label='Physical Threshold (0.4)')
        plt.axvline(x=0.40, color='grey', linestyle=':', alpha=0.5, label='High Affinity Proxy (0.4)')

        if len(points) > 0:
            plt.scatter(points[0][0], points[0][1], color='blue', s=100, marker='*', label='Start (Iter 1)', zorder=5)
            plt.scatter(points[-1][0], points[-1][1], color='red', s=100, marker='X', label='End (Terminal)', zorder=5)

        plt.axhspan(plddt_threshold, max(1.0, c_data['pLDDT'].max()), xmin=0.4, xmax=1.0, color='green', alpha=0.05)

        plt.title(f'Evolution Flow: {cluster}', fontweight='bold')
        plt.xlabel('ipSAE (Affinity Proxy)', fontweight='bold')
        plt.ylabel('pLDDT (Folding Confidence)', fontweight='bold')
        plt.grid(True, linestyle='--', alpha=0.3)
        plt.legend(loc='lower left')

        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    auto_csv_target = get_latest_trajectory_path()
    analyze_and_plot_trajectory(auto_csv_target)


# 目的: 提供演化日志的自动化分析与可视化，打印演化坐标路径，生成防重叠的独立相空间流向图。
# 上游代码: Cell_24_Head_Wolf_Evolution.py
# 运行环境: Google Colab / 本地 Jupyter Notebook
# 生成时间: 2026-05-08 09:41 EDT
# 更改说明:
# 第 15-43 行: 引入 json 与 re 模块，新增 get_latest_trajectory_path() 函数，通过读取 setting 自动拼接最新 CSV 的绝对路径。
# 第 79-84 行: 在核心统计输出后方，新增遍历输出 iter_max 中每一代的 (ipSAE, pLDDT) 坐标，使用 ➔ 符号拼接。
# 第 87 行及以下: 移除 fig, axes = plt.subplots，将 plt.figure(figsize=(7, 6)) 移入 for cluster in clusters 循环内部，确保每个 Seed 独占一张输出图表，避免多维图表挤压。
# 第 116 行: 主程序入口替换为调用自动获取路径的函数。

# ⚠️ CRITICAL: Irreversible Environment Eradication & Total Data Purge. Back up Data Before Execution.

In [ ]:
# # # Cell_17_Forced_cleanup.py
# '''

# # # ==========================================
# # # Forced Environment Cleanup and State Synchronization (Python Native + Terminal Force)
# # # ==========================================
# # import os
# # import shutil

# # ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
# # BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"

# # print(">>> Initiating full-path forced cleanup sequence...")

# # # 1. Use Terminal commands for physical erasure (handling FUSE mount synchronization)
# # print(">>> [1/2] Forcing directory deletion via terminal commands...")
# # !rm -rf "{ROOT_DIR}"
# # !rm -rf "{BACKUP_DIR}"

# # # 2. Use native Python libraries for kernel-state verification
# # print(">>> [2/2] Verifying kernel-state via Python...")
# # for target_path in [ROOT_DIR, BACKUP_DIR]:
# #     if os.path.exists(target_path):
# #         print(f"⚠️ Warning: Remnants detected after terminal command. Initiating secondary destruction: {target_path}")
# #         try:
# #             shutil.rmtree(target_path)
# #             print(f"✅ Completely wiped: {target_path}")
# #         except Exception as e:
# #             print(f"❌ Force deletion failed (please check Drive permissions): {e}")
# #     else:
# #         print(f"✅ Confirmed path is physically cleared: {target_path}")

# # print("\n>>> State refresh complete! The current Google Drive environment is confirmed to be completely clean.")
# # print(">>> Please re-run [Cell 3] (Unified Environment Setup) immediately.")

# # # Purpose: Completely clear project directories and cache backups using a dual approach of terminal rm -rf and Python's shutil.rmtree to resolve the "ghost folder" issue caused by cloud drive synchronization delays.
# # # Upstream Code: User-provided native Python cleanup script.
# # # Runtime Environment: Google Colab.
# # # Generation Time: 2026-04-01 11:01 EDT.
# # # Changed Lines:
# # # * Added the !rm -rf terminal command as the first line of defense for cleanup.
# # # * Optimized output logs to clearly distinguish between the terminal erasure and kernel verification phases.

# Section 4: Threshold-Driven Autonomous Screening

In [ ]:
# @title Cell_7b_Autonomous_Generation_Loop.py
# Requirement: Execute an automated loop for high-throughput generation and evaluation. Read task_name from root.
# ENHANCEMENT: Fixed fatal Regex truncation bug and ID collision bug.
# UPDATE: Implemented dynamic 'Cycle{X}_' prefix injection and Section-specific Smart Restart.
# 🔥 GLOBAL TRACKER UPDATE: Dynamically injects master log link, Screening Cycles, and Top Max ipSAE into history_run.csv.
# 🛡️ VRAM OPTIMIZATION: Added PYTORCH_ALLOC_CONF=expandable_segments to prevent memory fragmentation.

import os, json, sys, time, subprocess, csv, datetime, shutil, math, random, gc, re
from pathlib import Path
import torch

print("🔍 Initializing Auto-Pilot Engine & Global Tracker...")

# ---------------------------------------------------------
# 1. Decoupled Configuration Loading via last_runs.txt
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
ROOT_SETTING = BASE_DIR / 'internal_settings.json'

if not ROOT_SETTING.exists():
    print("❌ Error: Root configuration missing. Run Section 0 first.")
    sys.exit(1)

with open(ROOT_SETTING, 'r') as f: root_cfg = json.load(f)
task_name = root_cfg['task_name']
last_run_file = BASE_DIR / 'screening_results' / 'last_runs.txt'

run_id, target_dir_name = None, None

if last_run_file.exists():
    with open(last_run_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        pattern = rf"Section2_TargetPreprocess \| Run_ID: ({re.escape(task_name)}_\d+_\d+) \| Dir: (final_\d+_\d+)"
        for line in reversed(lines):
            match = re.search(pattern, line)
            if match:
                run_id, target_dir_name = match.groups()
                break

if not run_id or not target_dir_name:
    print(f"❌ Error: No valid run history found for task [{task_name}] in last_runs.txt.")
    sys.exit(1)

final_dir = BASE_DIR / 'screening_results' / task_name / target_dir_name
ISOLATED_SETTING = final_dir / 'internal_settings.json'

with open(ISOLATED_SETTING, 'r') as f: cfg = json.load(f)

target_threshold = float(cfg['target_threshold'])
target_success_count = int(cfg['target_success_count'])
designs_per_loop = int(cfg['designs_per_loop'])
max_iterations = int(cfg['max_iterations'])
generation_batch_size = int(cfg.get('generation_batch_size', 36))
evaluation_batch_size = int(cfg.get('evaluation_batch_size', 1))
force_restart = bool(root_cfg.get('force_restart_section_5', False))

print(f"✅ State Secured: Loaded settings from isolated workspace [{target_dir_name}]")
print(f"🤖 Auto-Pilot Engine Activated for Target: [{task_name}]")

# ---------------------------------------------------------
# 2. Tracking Variables, Logging & Master Hierarchy Setup
# ---------------------------------------------------------
ALL_DESIGNS_POOL = []
GLOBAL_HITS = []
existing_master_ids = set()
max_loop_found = 0

master_log_path = final_dir / f"{run_id}_AutoPilot_master_log.csv"
LIVE_PDB_DIR = final_dir / 'AutoPilot_Live_Top10_PDBs'

# --- 🌟 Global History Tracker Helper Function ---
def update_global_history(current_cycle=None, top_ipsae=None):
    HISTORY_CSV = BASE_DIR / 'screening_results' / 'history_run.csv'
    if not HISTORY_CSV.exists(): return
    try:
        updated_rows = []
        with open(HISTORY_CSV, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            header = next(reader, None)
            if not header: return

            if "Screening_Cycles" not in header: header.append("Screening_Cycles")
            if "Top_Max_ipSAE" not in header: header.append("Top_Max_ipSAE")
            if "Overall_Data_Link" not in header: header.append("Overall_Data_Link")

            r_idx = header.index("Run_ID") if "Run_ID" in header else -1
            c_idx = header.index("Screening_Cycles")
            i_idx = header.index("Top_Max_ipSAE")
            l_idx = header.index("Overall_Data_Link")

            updated_rows.append(header)
            for row in reader:
                while len(row) < len(header): row.append("")
                if r_idx != -1 and row[r_idx] == run_id:
                    if current_cycle is not None: row[c_idx] = str(current_cycle)
                    if top_ipsae is not None: row[i_idx] = f"{top_ipsae:.4f}"

                    link = f"[View Master Log]({master_log_path.relative_to(BASE_DIR)})"
                    if link not in row[l_idx]:
                        row[l_idx] = f"{row[l_idx]} | {link}" if row[l_idx].strip() else link
                updated_rows.append(row)

        with open(HISTORY_CSV, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerows(updated_rows)
    except Exception as e:
        print(f"⚠️ Warning: Global Tracker update failed: {e}")

update_global_history()

if force_restart:
    print("\n🗑️ [COMMAND] Force Restart Section 5 is ENABLED.")
    if master_log_path.exists(): master_log_path.unlink()
    if LIVE_PDB_DIR.exists(): shutil.rmtree(LIVE_PDB_DIR)

LIVE_PDB_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

SESSION_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
SESSION_NAME = f"AutoPilot_{task_name}_{SESSION_TIMESTAMP}"
SESSION_MASTER_DIR = BASE_DIR / 'assets' / 'target_data' / SESSION_NAME
SESSION_MASTER_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env.update({'HYDRA_FULL_ERROR': '1', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'PYTHONUNBUFFERED': '1'})

# ---------------------------------------------------------
# 3. State Hydration (Breakpoint Resume Protocol)
# ---------------------------------------------------------
if master_log_path.exists():
    try:
        with open(master_log_path, 'r', encoding='utf-8') as f_log:
            reader = csv.DictReader(f_log)
            for row in reader:
                d_id = row.get('design_id', '').strip()
                if not d_id: continue
                existing_master_ids.add(d_id)
                session_loop = int(row.get('session_loop', 0)) if row.get('session_loop', '').isdigit() else 0
                max_loop_found = max(max_loop_found, session_loop)

                ipsae_val = float(row.get('af2folding_max_ipsae', row.get('max_ipSAE', -999.0)))
                design_entry = {
                    'id': d_id, 'score': ipsae_val,
                    'ptm': float(row.get('af2folding_ptm_log', -999.0)),
                    'iptm': float(row.get('af2folding_i_ptm_log', -999.0)),
                    'plddt': float(row.get('af2folding_plddt', -999.0)),
                    'rmsd': float(row.get('af2folding_rmsd', -999.0)),
                    'total_reward': float(row.get('total_reward', -999.0)),
                    'batch_num': session_loop, 'idx_in_batch': 0,
                    'dir': row.get('source_path', ''), 'pdb_path': row.get('pdb_path', '')
                }
                ALL_DESIGNS_POOL.append(design_entry)
                if ipsae_val >= target_threshold and not any(h['id'] == d_id for h in GLOBAL_HITS):
                    GLOBAL_HITS.append(design_entry)

        print(f"📥 RESUME STATE DETECTED: Synchronized {len(ALL_DESIGNS_POOL)} historical designs. Highest loop completed: {max_loop_found}")

        if ALL_DESIGNS_POOL:
            top_historical_score = max(ALL_DESIGNS_POOL, key=lambda x: x['score'])['score']
            update_global_history(current_cycle=max_loop_found, top_ipsae=top_historical_score)

    except Exception as e:
        print(f"⚠️ Warning: Master log parsing error: {e}")

# ---------------------------------------------------------
# 4. High-Throughput Engine Execution Loop
# ---------------------------------------------------------
current_iteration = max_loop_found

while current_iteration < max_iterations and len(GLOBAL_HITS) < target_success_count:
    current_iteration += 1
    loop_start_time = time.time()
    batch_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    current_seed = random.randint(1, 1000000)
    current_run_name = f"{SESSION_NAME}_Cycle{current_iteration}"
    RUN_ISOLATED_DIR = SESSION_MASTER_DIR / current_run_name
    RUN_ISOLATED_DIR.mkdir(parents=True, exist_ok=True)
    ISOLATED_PDB_PATH = RUN_ISOLATED_DIR / f"{task_name}_fixed.pdb"
    shutil.copy(SOURCE_PDB_PATH, ISOLATED_PDB_PATH)

    print(f"\n=======================================================")
    print(f"🔄 Auto-Pilot Cycle Iteration [{current_iteration}/{max_iterations}] Commencing...")
    print(f"=======================================================")

    num_batches = math.ceil(designs_per_loop / generation_batch_size)
    total_designs = num_batches * generation_batch_size

    # 🛡️ 在这里加入了 export PYTORCH_ALLOC_CONF=expandable_segments:True
    cmd_str = (
        f"export PYTORCH_ALLOC_CONF=expandable_segments:True && "
        f"complexa design configs/search_binder_local_pipeline.yaml ++run_name={current_run_name} "
        f"++generation.task_name={task_name} ++generation.num_designs={total_designs} "
        f"++generation.dataloader.batch_size={generation_batch_size} ++generation.search.max_batch_size={generation_batch_size} ++evaluation.dataloader.batch_size={evaluation_batch_size} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={ISOLATED_PDB_PATH} "
        f"++run_filter=True ++run_evaluate=True ++evaluate.num_recycles=1 ++evaluate.pad_to_max_length=True ++evaluate.use_msa=False ++evaluate.msa_mode=single_sequence ++run_analyze=True ++seed={current_seed}"
    )

    process = subprocess.Popen(
        f"source env.sh && {cmd_str}",
        env=env, shell=True, executable='/bin/bash', cwd=str(BASE_DIR),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout: print(line, end='', flush=True)
    process.wait()

    if process.returncode != 0:
        print(f"❌ Warning: Generation engine crashed (Code {process.returncode}). Releasing memory and retrying...")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        time.sleep(3)
        continue

    # ---------------------------------------------------------
    # 5. Report Parsing & Dynamic Leaderboard
    # ---------------------------------------------------------
    inference_dir = BASE_DIR / 'inference' / f'search_binder_local_pipeline_{task_name}_{current_run_name}'
    valid_csvs = [f for f in inference_dir.rglob('*.csv') if 'timing' not in f.name.lower()]

    if valid_csvs:
        target_csv = max(valid_csvs, key=lambda x: x.stat().st_size)
        with open(target_csv, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            master_fieldnames = ['session_loop', 'timestamp', 'source_path'] + list(reader.fieldnames)
            if 'design_id' not in master_fieldnames: master_fieldnames.insert(3, 'design_id')

            rows_to_write = []
            for i, row in enumerate(reader):
                base_d_id = row.get('design_id', '').strip() or f"Design_{row.get('pdb_index', i)}"
                d_id = f"Cycle{current_iteration}_{base_d_id}"

                if d_id in existing_master_ids: continue
                existing_master_ids.add(d_id)

                ipsae_val = float(row.get('af2folding_max_ipsae', row.get('self_complex_max_ipAE', -999.0)))
                row['design_id'] = d_id
                row['session_loop'] = current_iteration
                row['timestamp'] = batch_timestamp
                row['source_path'] = str(inference_dir)
                rows_to_write.append(row)

                design_entry = {
                    'id': d_id, 'score': ipsae_val,
                    'ptm': float(row.get('af2folding_ptm_log', -999.0)),
                    'iptm': float(row.get('af2folding_i_ptm_log', -999.0)),
                    'plddt': float(row.get('af2folding_plddt', -999.0)),
                    'rmsd': float(row.get('af2folding_rmsd', -999.0)),
                    'total_reward': float(row.get('total_reward', -999.0)),
                    'batch_num': current_iteration, 'idx_in_batch': i + 1,
                    'dir': str(inference_dir), 'pdb_path': row.get('pdb_path', '')
                }
                ALL_DESIGNS_POOL.append(design_entry)
                if ipsae_val >= target_threshold and not any(h['id'] == d_id for h in GLOBAL_HITS):
                    GLOBAL_HITS.append(design_entry)

        if rows_to_write:
            file_exists = master_log_path.exists()
            with open(master_log_path, 'a', encoding='utf-8', newline='') as f_log:
                writer = csv.DictWriter(f_log, fieldnames=master_fieldnames, extrasaction='ignore')
                if not file_exists: writer.writeheader()
                writer.writerows(rows_to_write)

    # ---------------------------------------------------------
    # 6. Cycle Summary, Live Manifest & Metrics Display
    # ---------------------------------------------------------
    ALL_DESIGNS_POOL.sort(key=lambda x: x['score'], reverse=True)

    current_top_score = ALL_DESIGNS_POOL[0]['score'] if ALL_DESIGNS_POOL else -999.0
    update_global_history(current_cycle=current_iteration, top_ipsae=current_top_score)
    print(f"📊 Global History Tracker Updated: Cycle {current_iteration} | Top ipSAE: {current_top_score:.4f}")

    current_top_ids = set([d['id'] for d in ALL_DESIGNS_POOL[:10]])
    for existing_file in LIVE_PDB_DIR.glob('*.pdb'):
        if existing_file.stem not in current_top_ids: existing_file.unlink()

    for d in ALL_DESIGNS_POOL[:10]:
        target_d_id = d['id']
        exact_pdb_path = d.get('pdb_path', '')
        resolved_path = Path(exact_pdb_path) if exact_pdb_path and Path(exact_pdb_path).exists() else None

        if not resolved_path and Path(d['dir']).exists():
            for pdb_file in Path(d['dir']).rglob('*.pdb'):
                base_d_id = re.sub(r'^Cycle\d+_', '', target_d_id)
                if base_d_id.lower() in pdb_file.name.lower() or pdb_file.name == f"{base_d_id}.pdb":
                    resolved_path = pdb_file; break

        if resolved_path:
            dest_path = LIVE_PDB_DIR / f"{target_d_id}.pdb"
            if not dest_path.exists(): shutil.copy2(resolved_path, dest_path)

    print("\n🏆 Live Top 10 Design Roster:")
    header_format = "{:<5} | {:<16} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8}"
    print(header_format.format("Rank", "Design_ID", "ipSAE", "pTM", "iPTM", "pLDDT", "RMSD", "Reward"))
    print("-" * 85)
    for rank, d in enumerate(ALL_DESIGNS_POOL[:10], 1):
        print(header_format.format(rank, d['id'][:16], f"{d['score']:.4f}", f"{d['ptm']:.4f}", f"{d['iptm']:.4f}", f"{d['plddt']:.2f}", f"{d['rmsd']:.2f}Å", f"{d['total_reward']:.4f}"))

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.ipc_collect()

print("\n🛑 Auto-Pilot execution state terminated.")

In [ ]:

# @title Cell_8_Top10_Complex_Visualization_Viewer_with_PDF.py
# User Requirement:
# 1. Color the first amino acid of the peptide (binder) blue.
# 2. Make the PyMOL PDF output images match the py3Dmol output (Target: mesh; Peptide: solid green; Hotspots: solid orange).
# 3. Output the entire code. Do not suppress intermediate output.
# 4. Strictly use English in code and comments. Integer casting implemented for py3Dmol resi selectors.

import os, json, sys, shutil, time, math, re, csv
from pathlib import Path
import py3Dmol
from IPython.display import display, HTML

# ---------------------------------------------------------
# 0. Dependency Resolution
# ---------------------------------------------------------
dependencies_needed = []
try:
    from fpdf import FPDF
except ImportError:
    dependencies_needed.append("fpdf")

try:
    import pymol
except ImportError:
    dependencies_needed.append("pymol-open-source")

if dependencies_needed:
    print(f"📦 Installing missing dependencies: {', '.join(dependencies_needed)}...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + dependencies_needed)
    from fpdf import FPDF
    import pymol

try:
    from google.colab import files
except ImportError:
    pass

# Amino Acid Mapping
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
         'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
         'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
         'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

# ---------------------------------------------------------
# 1. Configuration and Robust Path Setup (via last_runs.txt)
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'
LOCAL_TEMP_DIR = Path('/content/temp_pymol_renders')
LOCAL_TEMP_DIR.mkdir(parents=True, exist_ok=True)

if not SETTING_FILE.exists():
    print("❌ Error: internal_settings.json not found.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
target_chain_config = str(cfg.get('target_chains', 'A')).split(',')[0].strip()

# ----- Robust Directory Resolution -----
last_run_file = BASE_DIR / 'screening_results' / 'last_runs.txt'
run_id = None
target_dir_name = None

if last_run_file.exists():
    with open(last_run_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        pattern = rf"Section2_TargetPreprocess \| Run_ID: ({re.escape(task_name)}_\d+_\d+) \| Dir: (final_\d+_\d+)"
        for line in reversed(lines):
            match = re.search(pattern, line)
            if match:
                run_id = match.group(1)
                target_dir_name = match.group(2)
                break

if not run_id or not target_dir_name:
    print(f"❌ Error: No valid run history found for task [{task_name}] in last_runs.txt")
    sys.exit(1)

screening_dir = BASE_DIR / 'screening_results' / task_name
latest_final = screening_dir / target_dir_name
LIVE_PDB_DIR = latest_final / 'AutoPilot_Live_Top10_PDBs'
timestamp_str = run_id.split('_')[-1] # For bundle naming compatibility downstream

if not LIVE_PDB_DIR.exists():
    print(f"❌ Error: Live PDB directory not found at {LIVE_PDB_DIR}")
    sys.exit(1)

# ---------------------------------------------------------
# 2. Structural Subtraction Mapping & AF2 Numbering Translator
# ---------------------------------------------------------
FIXED_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"
chain_to_af2_map = {}

with open(FIXED_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            chain = line[21]; resi = line[22:26].strip()
            if chain not in chain_to_af2_map:
                chain_to_af2_map[chain] = []
            chain_to_af2_map[chain].append(resi)

af2_mapping = {}
if target_chain_config in chain_to_af2_map:
    orig_resis = chain_to_af2_map[target_chain_config]
    af2_mapping = {orig: str(i + 1) for i, orig in enumerate(orig_resis)}

display(HTML(f"<h2>--- Auto-Pilot Top 10 Visualization: [{run_id}] ---</h2>"))

# ---------------------------------------------------------
# 3. Initialize Staging Directory & PDF
# ---------------------------------------------------------
bundle_name = f"{task_name}_{timestamp_str}_autopilot"
PACK_STAGING = Path(f"/content/{bundle_name}")
if PACK_STAGING.exists(): shutil.rmtree(PACK_STAGING)
PACK_STAGING.mkdir(parents=True, exist_ok=True)

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()
pdf.set_font("Arial", style='B', size=16)
pdf.cell(0, 20, txt=f"Proteina-Complexa Auto-Pilot Report", ln=True, align='C')
pdf.set_font("Arial", size=12)
pdf.cell(0, 10, txt=f"Task: {task_name} | ID: {run_id}", ln=True, align='C')
pdf.line(10, 40, 200, 40)
pdf.ln(15)

# --- Preload CSV Metrics for HTML Display ---
design_metrics = {}
master_log_csv = latest_final / f"{run_id}_AutoPilot_master_log.csv"
if master_log_csv.exists():
    with open(master_log_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            design_metrics[row.get('design_id', '').strip()] = row

# ---------------------------------------------------------
# 4. Processing Loop: PDB Extraction, Rendering & PDF Construction
# ---------------------------------------------------------
pymol.pymol_argv = ['pymol', '-c']
try: pymol.finish_launching()
except: pass

live_pdbs = sorted(list(LIVE_PDB_DIR.glob('*.pdb')))

for idx, target_pdb in enumerate(live_pdbs, 1):
    d_id = target_pdb.stem
    shutil.copy2(target_pdb, PACK_STAGING / f"Top_{idx}_{target_pdb.name}")

    with open(target_pdb, 'r') as f:
        pdb_data = f.read()

    chains_seq = {}
    first_resi_binder = None
    for line in pdb_data.split('\n'):
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            res_name = line[17:20].strip(); chain_id = line[21]
            if chain_id not in chains_seq: chains_seq[chain_id] = ""
            chains_seq[chain_id] += d3to1.get(res_name, 'X')

    chain_lengths = {c: len(seq) for c, seq in chains_seq.items()}
    sorted_chains = sorted(chain_lengths.keys(), key=lambda c: chain_lengths[c])
    binder_chains = [sorted_chains[0]]
    target_chain_ids = [c for c in sorted_chains[1:]]

    # Extract the exact starting residue number for the binder chain for precise mapping
    for line in pdb_data.split('\n'):
        if line.startswith("ATOM") and line[21] == binder_chains[0]:
            first_resi_binder = line[22:26].strip()
            break

    # -------------------------------------------------------
    # 4a. HTML Advanced Evaluation Data UI Output
    # -------------------------------------------------------
    peptide_len = len(chains_seq[binder_chains[0]]) if binder_chains else 0
    met = design_metrics.get(d_id, {})

    def safe_f(v):
        try: return float(v)
        except: return -999.0

    def fs(val, is_rmsd=False, is_plddt=False):
        if val == -999.0: return "N/A"
        if is_rmsd: return f"{val:<8.2f}Å"
        if is_plddt: return f"{val:<8.2f}"
        return f"{val:<8.4f}"

    ipsae_val = safe_f(met.get('af2folding_max_ipsae', met.get('self_complex_max_ipAE', met.get('complex_i_pAE', -999.0))))
    ptm_val   = safe_f(met.get('af2folding_ptm_log', met.get('self_complex_pTM', met.get('complex_pTM', -999.0))))
    iptm_val  = safe_f(met.get('af2folding_i_ptm_log', met.get('self_complex_i_pTM', met.get('complex_ipTM', -999.0))))
    plddt_val = safe_f(met.get('af2folding_plddt', met.get('self_complex_pLDDT', met.get('complex_pLDDT', -999.0))))
    rmsd_val  = safe_f(met.get('af2folding_rmsd', met.get('self_binder_scRMSD_ca', met.get('binder_scRMSD_ca', -999.0))))
    reward_val= safe_f(met.get('total_reward', -999.0))
    batch_val = met.get('session_loop', met.get('batch_num', 'N/A'))
    idx_val   = met.get('pdb_index', met.get('idx_in_batch', 'N/A'))

    table_html = f"""
    <div style='background: #f8f9fa; padding: 15px; border-radius: 8px; margin-bottom: 20px;'>
        <b style='font-size: 16px; color: #0056b3;'>[{idx}] Design: {d_id}</b><br>
        <b style='color: #28a745; font-size: 14px;'>Target Pipeline | Peptide Length: {peptide_len} AA</b>
        <pre style='margin-top: 10px; background: #fff; padding: 12px; border: 1px solid #ddd; border-radius: 4px; overflow-x: auto; color: #333;'>
Design_ID        | ipSAE    | pTM      | iPTM     | pLDDT    | RMSD       | Reward   | Batch  | Index
------------------------------------------------------------------------------------------------------------------
{d_id:<16} | {fs(ipsae_val):<8} | {fs(ptm_val):<8} | {fs(iptm_val):<8} | {fs(plddt_val, is_plddt=True):<8} | {fs(rmsd_val, is_rmsd=True):<10} | {fs(reward_val):<8} | {batch_val:<6} | {idx_val:<6}
</pre>
    </div>
    """
    display(HTML(table_html))

    if task_name == "GFP":
        mapped_hotspots = ['39', '40', '41', '73', '74', '200', '201', '202', '203', '204']
    else:
        cfg_hotspots = cfg.get('hotspots_input', '')
        raw_hotspots = [x.strip() for x in str(cfg_hotspots).split(',') if x.strip() and x != 'nan']
        mapped_hotspots = [af2_mapping.get(orig, orig) for orig in raw_hotspots]

    # Convert to integers to prevent py3Dmol silent selector failure
    mapped_hotspots_int = [int(re.sub(r'\D', '', str(x))) for x in mapped_hotspots if str(x).strip()]

    # 4b. PDF Page Content
    pdf.add_page()
    pdf.set_font("Arial", style='B', size=12)
    pdf.cell(0, 8, txt=f"Hit Index [{idx}]: {d_id} (Length: {peptide_len} AA)", ln=True)
    pdf.set_font("Courier", style='B', size=9)
    for bc in binder_chains:
        pdf.multi_cell(0, 5, txt=f"Binder Sequence (Chain {bc}): {chains_seq[bc]}")
    pdf.ln(4)

    # 4c. PyMOL Headless Render with Orientation
    img_path = LOCAL_TEMP_DIR / f"render_{idx}.png"
    pymol.cmd.reinitialize()
    pymol.cmd.load(str(target_pdb), 'complex')
    pymol.cmd.hide('everything', 'all')

    # Target rendering: Cartoon + Mesh
    target_sel = 'chain ' + '+'.join(target_chain_ids)
    pymol.cmd.show('cartoon', target_sel)
    pymol.cmd.color('gray80', target_sel)
    pymol.cmd.show('mesh', target_sel)

    # Hotspots rendering: Solid Surface
    match_c = target_chain_ids[0] if target_chain_ids else 'A'
    if mapped_hotspots:
        hotspot_sel = f"resi {'+'.join(mapped_hotspots)} and chain {match_c}"
        pymol.cmd.color('orange', hotspot_sel)
        pymol.cmd.show('surface', hotspot_sel)

    # Binder rendering: Solid Surface
    for bc in binder_chains:
        binder_sel = f"chain {bc}"
        pymol.cmd.show('cartoon', binder_sel)
        pymol.cmd.color('green', binder_sel)
        pymol.cmd.show('surface', binder_sel)

        # Color N-term blue using exact residue ID
        if first_resi_binder:
            pymol.cmd.color('blue', f"chain {bc} and resi {first_resi_binder}")

    if binder_chains and target_chain_ids:
        pymol.cmd.pseudoatom("com_t", selection='chain ' + '+'.join(target_chain_ids))
        pymol.cmd.pseudoatom("com_b", selection='chain ' + '+'.join(binder_chains))

        try:
            t_c = pymol.cmd.get_model("com_t").atom[0].coord
            b_c = pymol.cmd.get_model("com_b").atom[0].coord
            dx, dy, dz = b_c[0]-t_c[0], b_c[1]-t_c[1], b_c[2]-t_c[2]
            ty = math.degrees(math.atan2(-dx, dz))
            new_dz = -dx * math.sin(math.radians(ty)) + dz * math.cos(math.radians(ty))
            tx = math.degrees(math.atan2(dy, new_dz))
            pymol.cmd.center("all")
            pymol.cmd.turn("y", ty); pymol.cmd.turn("x", tx)
            pymol.cmd.delete("com_t"); pymol.cmd.delete("com_b")
        except Exception as e:
            pass

    pymol.cmd.bg_color('white')
    pymol.cmd.png(str(img_path), width=800, height=600, ray=0)
    time.sleep(0.5)

    if img_path.exists():
        pdf.image(str(img_path), x=15, y=None, w=180)
        os.remove(img_path)

    # 4d. py3Dmol Frontend - Safe Logic rendering
    viewer = py3Dmol.view(width=800, height=500)
    viewer.addModel(pdb_data, 'pdb')

    viewer.setStyle({'chain': target_chain_ids}, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.7}})
    viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'wireframe': True}, {'chain': target_chain_ids})

    if mapped_hotspots_int:
        h_sel = {'chain': target_chain_ids, 'resi': mapped_hotspots_int}
        viewer.setStyle(h_sel, {'cartoon': {'color': 'orange', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.SES, {'color': 'orange', 'opacity': 1.0}, h_sel)

    viewer.setStyle({'chain': binder_chains}, {'cartoon': {'color': 'lime', 'opacity': 1.0}})
    viewer.addSurface(py3Dmol.SES, {'color': 'lime', 'opacity': 1.0}, {'chain': binder_chains})

    if first_resi_binder:
        try:
            first_resi_binder_int = int(re.sub(r'\D', '', str(first_resi_binder)))
            b_first_sel = {'chain': binder_chains[0], 'resi': [first_resi_binder_int]}
            viewer.setStyle(b_first_sel, {'cartoon': {'color': 'blue', 'opacity': 1.0}})
            viewer.addSurface(py3Dmol.SES, {'color': 'blue', 'opacity': 1.0}, b_first_sel)
        except Exception:
            pass

    viewer.zoomTo(); viewer.show()

# ---------------------------------------------------------
# 5. Final Assembly: PDF, CSVs, Parameters, and ZIP
# ---------------------------------------------------------
# 5a. Save PDF to Staging
report_path = PACK_STAGING / f"{bundle_name}_Report.pdf"
pdf.output(str(report_path))

pred_csv = latest_final / f"{task_name}_AutoPilot_prediction_results.csv"
if not pred_csv.exists(): pred_csv = latest_final / "prediction_results.csv"

if pred_csv.exists(): shutil.copy2(pred_csv, PACK_STAGING / f"{task_name}_final_results.csv")
if master_log_csv.exists(): shutil.copy2(master_log_csv, PACK_STAGING / f"{run_id}_full_history_log.csv")

param_json = latest_final / 'internal_settings.json'
param_yaml = latest_final / f'targets_dict_{run_id}.yaml'
if param_json.exists(): shutil.copy2(param_json, PACK_STAGING / f"{run_id}_internal_settings.json")
if param_yaml.exists(): shutil.copy2(param_yaml, PACK_STAGING / f"{run_id}_targets_dict.yaml")

# 5c. Create ZIP Archive
zip_out_base = screening_dir / bundle_name
shutil.make_archive(str(zip_out_base), 'zip', str(PACK_STAGING))
final_zip_path = Path(str(zip_out_base) + ".zip")

# 5d. Cleanup
shutil.rmtree(PACK_STAGING)
if LOCAL_TEMP_DIR.exists(): shutil.rmtree(LOCAL_TEMP_DIR)

print(f"\n✅ All artifacts bundled into single ZIP: {final_zip_path.name}")
display(HTML(f"<div style='color: #155724; background-color: #d4edda; padding: 10px; border-radius: 5px; margin-top: 20px;'><b>📥 Download Ready:</b><br>{final_zip_path.relative_to(BASE_DIR)}</div>"))

try:
    files.download(str(final_zip_path))
except: pass

'''
==============================================================================
Objective: Generate top 10 complex visualizations, align PyMOL PDF output style with HTML py3Dmol (Mesh target, Solid green peptide, Solid orange hotspots, Blue N-terminus).
           Integer casting and regex applied to py3Dmol selections to prevent UI silent failure.
           Exact residue mapping applied to PyMOL selectors to ensure 100% accurate N-term coloring.
Upstream Dependencies: Design generation pipeline and PDB complex folding output.
Runtime Environment: Google Colab / Jupyter Notebook with PyMOL and py3Dmol.
Generation Timestamp: Auto-generated via Script execution.
==============================================================================
'''

In [ ]:
#@title Cell_9_Top10_Solubility_Analyzer.py
# Requirement: Load top designs directly from the live staging directory created by upstream modules, dynamically extract the binder sequence via structural subtraction, and evaluate physicochemical properties including solubility.
# Update Requirement: Refactor architecture to consume physical structures directly from the `AutoPilot_Live_Top10_PDBs` staging directory. Eliminate all legacy CSV parsing and manifest JSON dependencies.
# UPDATE: Synchronized directory resolution with the global run_id state. Replaced error-prone os.stat().st_mtime globbing with exact path construction based on internal_settings.json to guarantee 100% accurate session loading during resume operations.

import os, json, sys
from pathlib import Path
try:
    from Bio.SeqUtils.ProtParam import ProteinAnalysis
except ImportError:
    import subprocess
    print("Installing Biopython...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "biopython"])
    from Bio.SeqUtils.ProtParam import ProteinAnalysis

# 1. Configuration and Path Setup
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please run Section 0.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

run_id = cfg.get('run_id')
if not run_id:
    print("❌ Error: run_id not found in settings. Pipeline cannot proceed.")
    sys.exit(1)

task_name = cfg['task_name']
FIXED_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not FIXED_PDB_PATH.exists():
    print(f"❌ Error: Template fixed.pdb not found at {FIXED_PDB_PATH}.")
    sys.exit(1)

# Locate the exact final output directory using the global run_id
timestamp_suffix = run_id.split('_')[-1]
latest_final = BASE_DIR / 'screening_results' / task_name / f"final_{timestamp_suffix}"

if not latest_final.exists():
    print(f"❌ Error: Final output directory not found at {latest_final.relative_to(BASE_DIR)}. Upstream processing must be run first.")
    sys.exit(1)

# STRICT LIVE DIRECTORY LOADING
LIVE_PDB_DIR = latest_final / 'AutoPilot_Live_Top10_PDBs'
if not LIVE_PDB_DIR.exists() or not list(LIVE_PDB_DIR.glob('*.pdb')):
    print(f"❌ Error: No live PDB models found in {LIVE_PDB_DIR.relative_to(BASE_DIR)}.")
    sys.exit(1)

print(f"✅ Interfacing directly with live structural models from: {LIVE_PDB_DIR.name}")

# 2. Build Target Exclusion Dictionary
template_chains_resis = {}
with open(FIXED_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            chain = line[21]
            resi = line[22:26].strip()
            if chain not in template_chains_resis:
                template_chains_resis[chain] = set()
            template_chains_resis[chain].add(resi)

d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
         'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
         'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
         'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

print("\n🚀 --- [Top Candidate Designs: Solubility and Physicochemical Properties Evaluation] ---")

# 3. Process Live PDBs Directly
live_pdbs = sorted(list(LIVE_PDB_DIR.glob('*.pdb')))

for idx, target_pdb in enumerate(live_pdbs, 1):
    d_id = target_pdb.stem

    # Extract sequence data
    with open(target_pdb, 'r') as f:
        pdb_data = f.read()

    chains_seq = {}
    for line in pdb_data.split('\n'):
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            res_name = line[17:20].strip()
            chain_id = line[21]
            if chain_id not in chains_seq:
                chains_seq[chain_id] = ""
            chains_seq[chain_id] += d3to1.get(res_name, 'X')

    # Identify binder chains by excluding target chains
    binder_chains = [c for c in chains_seq.keys() if c not in template_chains_resis.keys()]
    binder_seq = "".join([chains_seq[c] for c in binder_chains]).replace('X', '')

    if not binder_seq:
        print(f"\n[{idx}] {d_id} | Failed to retrieve binder sequence from the PDB file.")
        continue

    # Execute physicochemical analysis
    analysis = ProteinAnalysis(binder_seq)
    gravy_score = analysis.gravy()
    pi_value = analysis.isoelectric_point()
    charge_at_7_4 = analysis.charge_at_pH(7.4)

    print(f"\n🏅 Candidate [{idx}] | Design: {d_id}")
    print(f"   🧬 Extracting sequence : {binder_seq}")

    # GRAVY Evaluation
    if gravy_score < 0:
        print(f"   💧 GRAVY score : {gravy_score:.3f} (Hydrophilic; typically exhibits good water solubility. ✅)")
    else:
        print(f"   🪨 GRAVY score : {gravy_score:.3f} (Hydrophobic; potential of aggregation and precipitation. ⚠️)")

    # pI Evaluation
    if 6.5 <= pi_value <= 8.5:
        print(f"   ⚡ pI : {pi_value:.2f} (⚠️ Possible protein precipitation)")
    else:
        print(f"   ⚡ pI : {pi_value:.2f} (Good solubility ✅)")

    print(f"   🔋 Net charge under physiological conditions (pH 7.4) : {charge_at_7_4:.2f} (Higher absolute value: Stronger electrostatic repulsion, lower risk of aggregation.)")

print("\n" + "="*60)
print("📌 Interpretation:")
print("- GRAVY < 0 and pI away from 7.4 has higher solubility.")
print("- High binding score with high GRAVY: Recommend subsequent hydrophilic mutations on non-binding interfaces to mitigate aggregation chance.")

# ==============================================================================
# Purpose: Eradicated the error-prone `.glob("final_*")` time-sorting mechanism. Upgraded to deterministic directory matching by exclusively deriving the target path from the `run_id` securely stored within `internal_settings.json`. This guarantees precise data alignment during breakpoint resumption.
# Upstream Code: Tightly coupled with the `RESUME_LAST_SESSION` variable propagated by Section 0.
# Runtime Environment: Google Colab
# Generation Time: 2026-04-11 13:11 EDT
# Changed Lines:
# - Lines 38-44: Deleted array sorting parameters and decoupled `st_mtime` dependency. Forced literal directory assembly utilizing `run_id.split('_')[-1]`.
# ==============================================================================

In [ ]:

#@title Cell_15_Geometric_Surface_Patch_Scanning_Engine.py
# Requirement: Determine exposed surface residues based on anchored geometrical boundaries and sequentially process designs for derived hotspot patches.
# Read task_name from root, scan last_runs.txt for the correct isolated directory, and load the specific internal_settings.json.
# ENHANCEMENT: Integrated "Live Top 10 Design Roster" reporting function to display the global best candidates across all processed patches after every cycle.
# ENHANCEMENT: Fixed fatal Regex truncation bug where the final directory name was prematurely cut off.
# UPDATE: Persist ALL columns from the raw inference CSV into the master_log directly, appending patch_site and timestamp metadata without dropping any metrics.
# UPDATE: Implement dynamic PDB file synchronization to maintain a live directory of the Top 10 models.
# UPDATE: Section 6 Smart Restart - Read force_restart_section_6 to wipe old logs, manifests, and Live PDBs if triggered.

import os, json, sys, subprocess, yaml, textwrap, shutil, time, random, csv, re
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.spatial import KDTree, Delaunay

print("🔍 Initializing Geometric Surface Scanning Engine...")

# ---------------------------------------------------------
# 1. Decoupled Configuration Loading via last_runs.txt
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
ROOT_SETTING = BASE_DIR / 'internal_settings.json'

if not ROOT_SETTING.exists():
    print("❌ Error: Root configuration missing. Please navigate to Section 0 and configure parameters.")
    sys.exit(1)

with open(ROOT_SETTING, 'r') as f:
    root_cfg = json.load(f)

task_name = root_cfg['task_name']
last_run_file = BASE_DIR / 'screening_results' / 'last_runs.txt'

run_id = None
target_dir_name = None

if last_run_file.exists():
    with open(last_run_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        pattern = rf"Section2_TargetPreprocess \| Run_ID: ({re.escape(task_name)}_\d+_\d+) \| Dir: (final_\d+_\d+)"
        for line in reversed(lines):
            match = re.search(pattern, line)
            if match:
                run_id = match.group(1)
                target_dir_name = match.group(2)
                break

if not run_id or not target_dir_name:
    print(f"❌ Error: No valid run history found for task [{task_name}] in last_runs.txt. Run Section 2 first.")
    sys.exit(1)

SCREENING_RESULTS_DIR = BASE_DIR / 'screening_results' / task_name
final_dir = SCREENING_RESULTS_DIR / target_dir_name
ISOLATED_SETTING = final_dir / 'internal_settings.json'

if not ISOLATED_SETTING.exists():
    print(f"❌ Error: Isolated configuration file missing in {final_dir.name}.")
    sys.exit(1)

with open(ISOLATED_SETTING, 'r') as f:
    cfg = json.load(f)

target_chains = cfg['target_chains']
anchor_str = str(cfg['anchor_residues'])
scan_margin = float(cfg['scan_margin'])
local_patch_radius = float(cfg['local_patch_radius'])
designs_per_patch = int(cfg['designs_per_patch'])
batch_size = int(cfg['scan_batch_size'])
evaluation_batch_size = int(cfg.get('evaluation_batch_size', 1))

# --- SMART RESTART LOGIC FOR SECTION 6 ---
force_restart = bool(root_cfg.get('force_restart_section_6', False))

print(f"✅ State Secured: Loaded settings from isolated workspace [{target_dir_name}]")
print(f"✅ Loaded global run_id: {run_id}")

anchor_res_list = [int(x.strip()) for x in anchor_str.split(',') if x.strip()]

SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not SOURCE_PDB_PATH.exists():
    print(f"❌ Error: Required processed PDB structure not found at {SOURCE_PDB_PATH}. Prior execution of Section 2 is mandatory.")
    sys.exit(1)

SESSION_NAME = f"GeoScan_{run_id}"
SESSION_MASTER_DIR = BASE_DIR / 'assets' / 'target_data' / SESSION_NAME
SESSION_MASTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Master Scan Directory established at: {SESSION_MASTER_DIR.relative_to(BASE_DIR)}")

def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}h {m}m {s}s"

# ---------------------------------------------------------
# 2. Parse PDB for Spatial Coordinates & Amino Acid Nomenclature
# ---------------------------------------------------------
coords, res_ids, res_names = [], [], []

with open(SOURCE_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
            res_names.append(line[17:20].strip())
            res_ids.append(int(line[22:26].strip()))
            coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])

coords, res_ids, res_names = np.array(coords), np.array(res_ids), np.array(res_names)
protein_kdtree = KDTree(coords)

SURFACE_NEIGHBOR_THRESHOLD = 24

# ---------------------------------------------------------
# 3. Anchor Validation & Geometric Convex Hull Construction
# ---------------------------------------------------------
anchor_coords = []
for a_res in anchor_res_list:
    if a_res in res_ids:
        idx = np.where(res_ids == a_res)[0][0]
        neighbors_10A = protein_kdtree.query_ball_point(coords[idx], r=10.0)
        if len(neighbors_10A) > SURFACE_NEIGHBOR_THRESHOLD:
            sys.exit(f"\n❌ FATAL ERROR: Specified Anchor {a_res} ({res_names[idx]}) is mathematically buried within the protein core!")
        anchor_coords.append(coords[idx])
    else:
        sys.exit(f"\n❌ FATAL ERROR: Requested Anchor Residue {a_res} cannot be mapped to target chain {target_chains}.")

if not anchor_coords:
    sys.exit("\n❌ Error: Geometry construction aborted. No validated anchor residues acquired.")

anchor_coords = np.array(anchor_coords)
shape_points = []
spacing = 1.0

n_anchors = len(anchor_coords)
if n_anchors == 1:
    shape_points = anchor_coords
elif n_anchors == 2:
    d = np.linalg.norm(anchor_coords[1] - anchor_coords[0])
    shape_points = np.linspace(anchor_coords[0], anchor_coords[1], max(2, int(d / spacing)))
elif n_anchors == 3:
    d1, d2 = np.linalg.norm(anchor_coords[1] - anchor_coords[0]), np.linalg.norm(anchor_coords[2] - anchor_coords[0])
    for u in np.linspace(0, 1, max(2, int(d1/spacing))):
        for v in np.linspace(0, 1-u, max(2, int(d2/spacing))):
            shape_points.append(anchor_coords[0] + u*(anchor_coords[1]-anchor_coords[0]) + v*(anchor_coords[2]-anchor_coords[0]))
    shape_points = np.array(shape_points)
else:
    hull = Delaunay(anchor_coords, qhull_options='QJ')
    min_b, max_b = np.min(anchor_coords, axis=0), np.max(anchor_coords, axis=0)
    grid_x, grid_y, grid_z = np.mgrid[min_b[0]:max_b[0]:spacing, min_b[1]:max_b[1]:spacing, min_b[2]:max_b[2]:spacing]
    grid_pts = np.vstack((grid_x.ravel(), grid_y.ravel(), grid_z.ravel())).T
    inside = hull.find_simplex(grid_pts) >= 0
    shape_points = grid_pts[inside]

shape_kdtree = KDTree(shape_points if shape_points.ndim == 2 else [shape_points])
candidate_indices = shape_kdtree.query_ball_tree(protein_kdtree, r=scan_margin)
candidate_idx_set = set([idx for sublist in candidate_indices for idx in sublist])

# ---------------------------------------------------------
# 4. Rigorous Topographic Filtration & Resume State Interception
# ---------------------------------------------------------
surface_residues_raw = []
for idx in candidate_idx_set:
    neighbors_10A = protein_kdtree.query_ball_point(coords[idx], r=10.0)
    if len(neighbors_10A) <= SURFACE_NEIGHBOR_THRESHOLD:
        surface_residues_raw.append(idx)

surface_residues = sorted(surface_residues_raw, key=lambda idx: res_ids[idx])

timestamp_suffix = run_id.split('_')[-1]

# Strict GeoScan Namespace applied - MIGRATED CHECKPOINT TO ISOLATED DIRECTORY
CHECKPOINT_CSV = final_dir / f'{run_id}_GeoScan_checkpoint.csv'
MASTER_LOG_CSV = final_dir / f'{run_id}_GeoScan_master_log.csv'
LIVE_PDB_DIR = final_dir / 'GeoScan_Live_Top10_PDBs'

# --- ENFORCING SECTION 6 FORCE RESTART ---
if force_restart:
    print("\n🗑️ [COMMAND] Force Restart Section 6 is ENABLED.")
    print("🧹 Wiping previous GeoScan checkpoints, master logs, and live PDBs to start completely fresh...")
    if CHECKPOINT_CSV.exists(): CHECKPOINT_CSV.unlink()
    if MASTER_LOG_CSV.exists(): MASTER_LOG_CSV.unlink()
    if LIVE_PDB_DIR.exists(): shutil.rmtree(LIVE_PDB_DIR)

LIVE_PDB_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Live Final Artifact Directory established at: {final_dir.relative_to(BASE_DIR)}")

all_scan_results = []
global_design_pool = []
completed_res_ids = set()
existing_master_ids = set()

# Load Checkpoint (Resume State)
if CHECKPOINT_CSV.exists():
    try:
        df_checkpoint = pd.read_csv(CHECKPOINT_CSV)
        if not df_checkpoint.empty:
            all_scan_results = df_checkpoint.to_dict('records')
            for val in df_checkpoint['Center_Residue']:
                res_id_str = str(val).split('(')[0].strip()
                if res_id_str.isdigit():
                    completed_res_ids.add(int(res_id_str))
            print(f"📥 RESUME STATE DETECTED: Successfully loaded {len(completed_res_ids)} processed patches.")
    except Exception as e:
        print(f"⚠️ Warning: Checkpoint parsing error: {e}")

# Load existing Master Log to populate pool and IDs for exact resume consistency
if MASTER_LOG_CSV.exists():
    try:
        with open(MASTER_LOG_CSV, 'r', encoding='utf-8') as f_log:
            reader = csv.DictReader(f_log)
            for row in reader:
                d_id = row.get('design_id', '').strip()
                if not d_id: continue
                existing_master_ids.add(d_id)
                global_design_pool.append({
                    'id': d_id,
                    'score': float(row.get('af2folding_max_ipsae', row.get('max_ipSAE', -999.0))),
                    'ptm': float(row.get('af2folding_ptm_log', row.get('pTM', -999.0))),
                    'iptm': float(row.get('af2folding_i_ptm_log', row.get('iPTM', -999.0))),
                    'plddt': float(row.get('af2folding_plddt', row.get('pLDDT', -999.0))),
                    'rmsd': float(row.get('af2folding_rmsd', row.get('RMSD', -999.0))),
                    'reward': float(row.get('total_reward', -999.0)),
                    'dir': row.get('source_path', ''),
                    'pdb_path': row.get('pdb_path', '')
                })
        print(f"📥 MASTER LOG DETECTED: Re-synchronized {len(global_design_pool)} existing designs to leaderboard.")
    except Exception as e:
        print(f"⚠️ Warning: Master log sync error: {e}")

pending_surface_residues = [idx for idx in surface_residues if res_ids[idx] not in completed_res_ids]

# ---------------------------------------------------------
# 5. Detection Log Formatting
# ---------------------------------------------------------
print("\n" + "="*70)
print(f"🎯 ALGORITHMIC GEOMETRIC SCANNING SECTOR ESTABLISHED")
print(f"Geometry Parameters: {n_anchors} Fixed Anchors | Volumetric Expansion Margin: {scan_margin}Å")
print(f"Total Isolated Candidates: {len(surface_residues)} residues.")
print(f"Pending Execution Queue  : {len(pending_surface_residues)} residues remaining.")
print("-" * 70)

res_strings = [f"{res_ids[idx]}({res_names[idx]})" for idx in pending_surface_residues]
formatted_list = textwrap.fill(", ".join(res_strings), width=80, initial_indent="➤ ", subsequent_indent="  ")
print(formatted_list if res_strings else "➤ Queue Empty. All patches have been scanned.")
print("="*70)

if not pending_surface_residues:
    print("✅ Geometric scanning fully completed. Proceeding to final analytics.")
else:
    print("🚀 Queuing generative algorithmic pipelines targeting topographical patches...\n")

# ---------------------------------------------------------
# 6. Core Execution Loop: Patch Isolation & Candidate Synthesis
# ---------------------------------------------------------
env = os.environ.copy()
env.update({'HYDRA_FULL_ERROR': '1', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'PYTHONUNBUFFERED': '1'})

for i, idx in enumerate(pending_surface_residues, 1):
    loop_start_time = time.time()
    center_res = res_ids[idx]
    center_name = res_names[idx]

    local_neighbors = protein_kdtree.query_ball_point(coords[idx], r=local_patch_radius)
    patch_residues = sorted([res_ids[n] for n in local_neighbors])
    patch_str = ",".join(map(str, patch_residues))

    patch_run_name = f"{SESSION_NAME}_Res{center_res}"
    RUN_ISOLATED_DIR = SESSION_MASTER_DIR / patch_run_name
    RUN_ISOLATED_DIR.mkdir(parents=True, exist_ok=True)

    ISOLATED_PDB_PATH = RUN_ISOLATED_DIR / f"{task_name}_fixed.pdb"
    shutil.copy(SOURCE_PDB_PATH, ISOLATED_PDB_PATH)

    print(f"\n=======================================================")
    print(f"🔄 [Active Sweep {i}/{len(pending_surface_residues)}] Center Assignment: {center_res} ({center_name})")
    print(f"📂 Local Sandbox: {RUN_ISOLATED_DIR.relative_to(SESSION_MASTER_DIR)}")
    print(f"=======================================================")

    yaml_path = BASE_DIR / 'configs/targets/targets_dict.yaml'
    with open(yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)
    yaml_data['target_dict_cfg'][task_name]['hotspot_residues'] = [f"{target_chains}{r}" for r in patch_residues]
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

    dynamic_seed = random.randint(1, 999999)
    cmd_str = (
        f"complexa design configs/search_binder_local_pipeline.yaml ++run_name={patch_run_name} "
        f"++generation.task_name={task_name} ++generation.num_designs={designs_per_patch} "
        f"++generation.dataloader.batch_size={batch_size} ++generation.search.max_batch_size={batch_size} ++evaluation.dataloader.batch_size={evaluation_batch_size} "
        f"++generation.seed={dynamic_seed} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={ISOLATED_PDB_PATH} "
        f"++run_filter=True ++run_evaluate=False ++run_analyze=False"
    )

    process = subprocess.run(f"source env.sh && {cmd_str}", env=env, shell=True, executable='/bin/bash', cwd=str(BASE_DIR), check=False)

    inference_dir = BASE_DIR / 'inference' / f'search_binder_local_pipeline_{task_name}_{patch_run_name}'

    if process.returncode == 0:
        valid_csvs = [f for f in inference_dir.rglob('*.csv') if 'timing' not in f.name.lower()]

        if valid_csvs:
            target_csv = max(valid_csvs, key=lambda x: x.stat().st_size)
            try:
                df = pd.read_csv(target_csv)
                score_col = 'af2folding_max_ipsae' if 'af2folding_max_ipsae' in df.columns else 'total_reward'

                if score_col in df.columns:
                    df_sorted = df.sort_values(by=score_col, ascending=False)
                    top_score = df_sorted[score_col].max()

                    master_log_exists = MASTER_LOG_CSV.exists()
                    appended_count = 0
                    batch_timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

                    raw_fieldnames = list(df_sorted.columns)
                    master_fieldnames = ['patch_site', 'timestamp'] + raw_fieldnames
                    if 'design_id' not in master_fieldnames:
                        master_fieldnames.insert(2, 'design_id')

                    rows_to_write = []

                    for _, row in df_sorted.iterrows():
                        row_dict = row.to_dict()

                        base_d_id = str(row_dict.get('design_id', '')).strip() or f"Design_{row_dict.get('pdb_index', 0)}"
                        d_id = f"Res{center_res}_{base_d_id}"

                        if d_id in existing_master_ids: continue
                        existing_master_ids.add(d_id)

                        row_dict['design_id'] = d_id
                        row_dict['patch_site'] = f"{center_res}_{center_name}"
                        row_dict['timestamp'] = batch_timestamp
                        row_dict['source_path'] = str(inference_dir.relative_to(BASE_DIR))

                        rows_to_write.append(row_dict)

                        ipsae = float(row_dict.get(score_col, -999.0))
                        ptm = float(row_dict.get('af2folding_ptm_log', row_dict.get('complex_pTM', -999.0)))
                        iptm = float(row_dict.get('af2folding_i_ptm_log', row_dict.get('complex_ipTM', -999.0)))
                        plddt = float(row_dict.get('af2folding_plddt', row_dict.get('complex_pLDDT', -999.0)))
                        rmsd = float(row_dict.get('af2folding_rmsd', row_dict.get('binder_scRMSD_ca', -999.0)))
                        reward = float(row_dict.get('total_reward', -999.0))

                        global_design_pool.append({
                            'id': d_id, 'score': ipsae, 'ptm': ptm, 'iptm': iptm,
                            'plddt': plddt, 'rmsd': rmsd, 'reward': reward,
                            'dir': inference_dir, 'pdb_path': row_dict.get('pdb_path', '')
                        })
                        appended_count += 1

                    if rows_to_write:
                        with open(MASTER_LOG_CSV, 'a', encoding='utf-8', newline='') as f_log:
                            writer = csv.DictWriter(f_log, fieldnames=master_fieldnames, extrasaction='ignore')
                            if not master_log_exists:
                                writer.writeheader()
                            writer.writerows(rows_to_write)

                    result_entry = {'Center_Residue': f"{center_res} ({center_name})", f'Max_{score_col}': round(top_score, 4), 'Designs_Generated': len(df_sorted), 'Hotspot_Patch': patch_str}
                    all_scan_results.append(result_entry)
                    pd.DataFrame(all_scan_results).to_csv(CHECKPOINT_CSV, index=False)

                    print(f"💾 Log Aggregated: {appended_count} full-data designs recorded. Checkpoint updated.")

                    # --- DYNAMIC MANIFEST GENERATION & DELTA LIVE PDB SYNC ---
                    global_design_pool.sort(key=lambda x: x['score'], reverse=True)

                    current_top_ids = set([d['id'] for d in global_design_pool[:10]])

                    for existing_file in LIVE_PDB_DIR.glob('*.pdb'):
                        if existing_file.stem not in current_top_ids:
                            existing_file.unlink()

                    top_pdb_paths = []
                    top_scores_list = []

                    for d in global_design_pool[:10]:
                        target_d_id = d['id']
                        exact_pdb_path = d.get('pdb_path', '')
                        resolved_path = None

                        if exact_pdb_path and Path(exact_pdb_path).exists():
                            resolved_path = Path(exact_pdb_path)
                        else:
                            source_dir = Path(d['dir']) if isinstance(d.get('dir'), Path) else BASE_DIR / str(d.get('dir', ''))
                            if source_dir.exists():
                                for pdb_file in source_dir.rglob('*.pdb'):
                                    base_d_id = re.sub(r'^Res\d+_', '', target_d_id)
                                    if base_d_id.lower() in pdb_file.name.lower() or pdb_file.name == f"{base_d_id}.pdb":
                                        resolved_path = pdb_file
                                        break

                        if resolved_path:
                            top_pdb_paths.append(str(resolved_path.relative_to(BASE_DIR)))
                            dest_path = LIVE_PDB_DIR / f"{target_d_id}.pdb"
                            if not dest_path.exists():
                                shutil.copy2(resolved_path, dest_path)
                        else:
                            top_pdb_paths.append(f"NOT_FOUND_{target_d_id}")

                        top_scores_list.append(d['score'])

                    manifest = {
                        "run_id": run_id,
                        "timestamp": timestamp_suffix,
                        "top_pdb_paths": top_pdb_paths,
                        "scores": top_scores_list
                    }

                    with open(final_dir / f'GeoScan_{task_name}_top_hits_manifest.json', 'w') as f:
                        json.dump(manifest, f, indent=4)

                    print(f"🔄 Live GeoScan_{task_name}_top_hits_manifest.json and Live PDB directory synchronized in {final_dir.name}")

                    print(f"\n🏆 Live Top 10 Design Roster (Global Index prioritized by max_ipSAE):")
                    header_format = "{:<5} | {:<26} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8}"
                    print(header_format.format("Rank", "Design_ID", "ipSAE", "pTM", "iPTM", "pLDDT", "RMSD", "Reward"))
                    print("-" * 105)
                    for rank, d in enumerate(global_design_pool[:10], 1):
                        print(header_format.format(
                            rank, d['id'], f"{d['score']:.4f}", f"{d['ptm']:.4f}",
                            f"{d['iptm']:.4f}", f"{d['plddt']:.2f}", f"{d['rmsd']:.2f}", f"{d['reward']:.4f}"
                        ))
                    print("-" * 105)

            except Exception as e:
                print(f"⚠️ Data Sync Failure for {center_res}: {e}")
        else:
            print(f"⚠️ Warning: No CSV output found in {inference_dir.name}.")
    else:
        print(f"⚠️ Alert: Pipeline crashed for {center_res}. Skipping.")

    cycle_duration = format_time(time.time() - loop_start_time)
    print(f"\n⏱️ Patch Sweep Completed in: {cycle_duration}")

    if i < len(pending_surface_residues):
        import gc; gc.collect()
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        time.sleep(2)

# ---------------------------------------------------------
# 7. Final Post-Processing Report Output
# ---------------------------------------------------------
if all_scan_results:
    results_df = pd.DataFrame(all_scan_results)
    sort_col = [col for col in results_df.columns if col.startswith('Max_')][0]
    results_df = results_df.sort_values(by=sort_col, ascending=False)
    FINAL_CSV = final_dir / f'geometric_scan_results_FINAL_{run_id}.csv'
    results_df.to_csv(FINAL_CSV, index=False)
    if CHECKPOINT_CSV.exists(): os.remove(CHECKPOINT_CSV)

    if MASTER_LOG_CSV.exists():
        shutil.copy2(MASTER_LOG_CSV, final_dir / f'{task_name}_GeoScan_prediction_results.csv')

    print("\n🏆 ======================================================= 🏆")
    print("           GEOMETRIC SCANNING PROTOCOL CONCLUDED              ")
    print("🏆 ======================================================= 🏆")
    print(f"Total Patches Scanned: {len(results_df)}")
    print(f"\n📂 Comprehensive analytic report finalized: {FINAL_CSV.relative_to(BASE_DIR)}")
    print(f"📂 Prediction results synced as {task_name}_GeoScan_prediction_results.csv in: {final_dir.relative_to(BASE_DIR)}")
    print(f"📂 Global master log DETAIL registry: {MASTER_LOG_CSV.relative_to(BASE_DIR)}")
    print(f"📂 Physical PDB models retained in: {LIVE_PDB_DIR.relative_to(BASE_DIR)}")
else:
    print("\n🛑 Geometric Scanning Protocol Terminated. No viable candidates found.")

'''
==============================================================================
Objective: Execute a geometric patch scanning engine isolated from generic global configurations.
           It extracts the active task_name from the root, utilizes regex to pinpoint the corresponding final output directory, and applies internal_settings.json.
           INCLUDES Section 6 Smart Restart Logic to wipe cache and force-restart execution if triggered.
Upstream Code: Architected to align with the namespace separation protocols established for Cell_7b (AutoPilot) and initialized by Section 2.
Runtime Environment: Google Colab.
Generation Timestamp: Auto-generated via Script execution.
==============================================================================
'''